In [ ]:
# ============================================================
# FIXED-CANDIDATE FULL CATALOG REBUILD
# ------------------------------------------------------------
# Frozen branch:
#   source: rho_eff ~ (1/r^2) d/dr [ r * V_bar^2 ]
#   screened field equation
#   fixed Rs = 1.5
#   shape map: V/V_flat = U/U_inf
#
# Frozen amplitude law:
#   U(r_t) = 0.6 U_inf
#   V_flat^2 = C * [r_t u(r_t)]^alpha * [Vbar^2(r_t)]^beta
#
# with fixed constants:
#   C     = 164.0
#   alpha = 0.175
#   beta  = 0.55
#
# No fitting in this script.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_fixed_candidate_catalog_rebuild")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# RUN FULL CATALOG
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()
rotmod_map = {display_name_from_path(p): p for p in rotmod_files}

rows = []
fails = []

print("=" * 60)
print("RUNNING FIXED-CANDIDATE FULL CATALOG")
print("=" * 60)
print(f"Candidate files : {len(rotmod_files)}")
print(f"Fixed Rs        : {Rs_fixed}")
print(f"Fixed f         : {F_FRAC}")
print(f"Fixed alpha     : {ALPHA}")
print(f"Fixed beta      : {BETA}")
print(f"Fixed C         : {C_AMP}")
print()

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        U_inf = float(np.max(U))
        rt = radius_at_fraction(r, U, F_FRAC)

        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            raise RuntimeError("Invalid transition quantities")

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)
        chi2_mean = float(np.mean(((rot["vobs"] - V_obs_pred) / rot["ev"]) ** 2))

        obs_shape = rot["vobs"] / max(float(np.max(rot["vobs"])), 1.0e-12)
        pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
        shape_rmse = safe_rmse(obs_shape, pred_shape)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "Rs_fixed": Rs_fixed,
            "rt": rt,
            "u_t": u_t,
            "vbar2_t": vbar2_t,
            "carrier": carrier,
            "V_flat_pred": V_flat_pred,
            "rmse": rmse,
            "mae": mae,
            "chi2_mean": chi2_mean,
            "shape_rmse": shape_rmse,
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

res_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(res_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

# Simple bins for tail view
res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

print()
print("=" * 60)
print("FIXED-CANDIDATE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

RES_CSV = os.path.join(OUTDIR, "fixed_candidate_results.csv")
FAIL_CSV = os.path.join(OUTDIR, "fixed_candidate_failures.csv")
BEST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_best20.csv")
WORST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_worst20.csv")
BIN_CSV = os.path.join(OUTDIR, "fixed_candidate_bin_summary.csv")
TXT_OUT = os.path.join(OUTDIR, "fixed_candidate_summary.txt")

res_df.to_csv(RES_CSV, index=False)
fail_df.to_csv(FAIL_CSV, index=False)
best20.to_csv(BEST_CSV_OUT, index=False)
worst20.to_csv(WORST_CSV_OUT, index=False)
bin_summary.to_csv(BIN_CSV, index=False)

with open(TXT_OUT, "w", encoding="utf-8") as f:
    f.write("FIXED-CANDIDATE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

Auto-extracting LTG rotmods from: /content/Rotmod_LTG (4).zip
RUNNING FIXED-CANDIDATE FULL CATALOG
Candidate files : 175
Fixed Rs        : 1.5
Fixed f         : 0.6
Fixed alpha     : 0.175
Fixed beta      : 0.55
Fixed C         : 164.0

Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

FIXED-CANDIDATE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
median_rmse            = 20.38786367728187
mean_rmse              = 32.72222113537459
p90_rmse               = 77.35749478059131
p95_rmse               = 88.8575199776623
median_mae             = 17.608721572718746
median_chi2_mean       = 26.12511787253237
median_shape_rmse      = 0.15760201986858705

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 11.457480 10.528257  14.716546    0.128219
     mid 18.074875 15.106115  13.274674    0.115517
    hig

In [ ]:
# ============================================================
# HIGH-END AMPLITUDE TAIL CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Tail correction:
#   V_flat,corr = V_flat,base * [1 + eta * (Vbar2_t / V0^2)^delta]
#
# Only amplitude is modified. Source/field/shape stay fixed.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_tail_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
V0_LIST    = [120.0, 150.0, 180.0, 210.0, 240.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TAIL-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "vbar2_t": vbar2_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for V0 in V0_LIST:
            rmses = []
            vmax_bins = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["vbar2_t"] / (V0**2)) ** delta
                V_flat_pred = row["V_flat_base"] * correction
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmax_bins.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmax_bins = np.asarray(vmax_bins, float)

            high_mask = vmax_bins >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "V0": V0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("HIGH-END TAIL CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_tail_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_tail_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END TAIL CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TAIL-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END TAIL CORRECTION SUMMARY
 eta  delta    V0  median_rmse  p90_rmse  high_bin_median_rmse
0.00   0.25 120.0    20.387864 77.357495             63.789441
0.00   0.25 150.0    20.387864 77.357495             63.789441
0.00   0.25 180.0    20.387864 77.357495             63.789441
0.00   0.25 210.0    20.387864 77.357495             63.789441
0.00   0.25 240.0    20.387864 77.357495             63.789441
0.00   0.50 120.0    20.387864 77.357495             63.789441
0.00   0.50 150.0    20.387864 77.357495             63.789441
0.00   0.50 180.0    20.387864 77.357495             63.789441
0.00   0.50 210.0    20.387864 77.357495             63.789441
0.00   0.50 240.0    20.387864 77.357495             63.789441
0.00   0.75 120.0    20.387864 77.357495             6

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# BULGE-SENSITIVE TRANSITION CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Bulge correction:
#   V_flat,corr^2 = V_flat,base^2 * [1 + eta * f_bulge(rt)^delta]
#
# where
#   f_bulge(rt) = Vbulge^2(rt) / Vbar^2(rt)
#
# Goal:
#   see if high-vmax tail improves without harming the catalog.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_bulge_transition_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
DELTA_LIST = [0.50, 0.75, 1.00, 1.25, 1.50]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING BULGE-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
        vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
        vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
        vbar2  = vgas2 + vdisk2 + vbul2

        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_interp = interp1d(rot["r"], vbul2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        vbul2_t = max(float(vbul2_interp(rt)), 0.0)
        f_bulge_t = vbul2_t / vbar2_t if vbar2_t > 0 else 0.0

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "f_bulge_t": f_bulge_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        rmses = []
        vmaxs = []

        for _, row in df.iterrows():
            correction = 1.0 + eta * (row["f_bulge_t"] ** delta)
            V_flat_pred = row["V_flat_base"] * np.sqrt(correction)
            V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

            V_obs_pred = interp1d(
                row["r_grid"], V_theory,
                kind="linear", bounds_error=False, fill_value="extrapolate"
            )(row["r_obs"])

            rmse = safe_rmse(row["v_obs"], V_obs_pred)
            rmses.append(rmse)
            vmaxs.append(row["vmax_obs"])

        rmses = np.asarray(rmses, float)
        vmaxs = np.asarray(vmaxs, float)

        high_mask = vmaxs >= 150.0
        high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

        summary_rows.append({
            "eta": eta,
            "delta": delta,
            "median_rmse": float(np.median(rmses)),
            "p90_rmse": float(np.percentile(rmses, 90)),
            "high_bin_median_rmse": high_med,
        })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("BULGE-CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "bulge_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "bulge_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("BULGE-CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING BULGE-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

BULGE-CORRECTION SUMMARY
 eta  delta  median_rmse  p90_rmse  high_bin_median_rmse
0.10   1.00    20.387864 76.658395             62.747288
0.10   1.25    20.387864 76.767365             62.760592
0.10   0.75    20.387864 76.558643             62.809717
0.10   1.50    20.387864 76.870950             62.844898
0.10   0.50    20.469380 77.125908             62.872927
0.05   0.50    20.387864 76.927244             63.666174
0.05   0.75    20.387864 76.974112             63.716538
0.05   1.00    20.387864 77.033582             63.764955
0.05   1.25    20.387864 77.093313             63.789441
0.05   1.50    20.387864 77.148207             63.789441
0.00   0.50    20.387864 77.358929             63.789441
0.00   0.75    20.387864 77.358929             63.789441
0.00   1.00  

In [ ]:
# ============================================================
# TRANSITION COMPACTNESS CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Compactness correction:
#   c_t = Vbar^2(rt) / rt
#   V_flat,corr^2 = V_flat,base^2 * [1 + eta * (c_t / c0)^delta]
#
# Goal:
#   improve the high-vmax tail without reopening the model.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_transition_compactness_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
C0_LIST    = [5_000.0, 10_000.0, 20_000.0, 40_000.0, 80_000.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING COMPACTNESS-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vbar2 = np.maximum(
            rot["vgas"] * np.abs(rot["vgas"])
            + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
            + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
            0.0
        )
        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        c_t = vbar2_t / max(rt, 1.0e-12)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "c_t": c_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for c0 in C0_LIST:
            rmses = []
            vmaxs = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["c_t"] / c0) ** delta
                V_flat_pred = row["V_flat_base"] * np.sqrt(correction)
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmaxs.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmaxs = np.asarray(vmaxs, float)

            high_mask = vmaxs >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "c0": c0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("COMPACTNESS-CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "compactness_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "compactness_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("COMPACTNESS-CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING COMPACTNESS-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

COMPACTNESS-CORRECTION SUMMARY
 eta  delta      c0  median_rmse  p90_rmse  high_bin_median_rmse
0.20   0.75 80000.0    20.492094 77.523220             63.289288
0.10   0.75 40000.0    20.475352 77.384626             63.452850
0.05   0.25 10000.0    20.720630 77.126245             63.592638
0.15   0.75 80000.0    20.465813 77.402844             63.598978
0.05   1.00 20000.0    20.421821 77.538450             63.623908
0.10   1.00 40000.0    20.421821 77.538450             63.623908
0.20   1.00 80000.0    20.421821 77.538450             63.623908
0.15   1.00 40000.0    20.438896 77.450237             63.664564
0.30   1.00 80000.0    20.438896 77.450237             63.664564
0.05   0.75 20000.0    20.461319 77.411665             63.669028
0.05   0.50 10000.0    20.5

In [ ]:
# ============================================================
# TWO-POPULATION NORMALIZATION TEST
# ------------------------------------------------------------
# Fixed branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = C_pop * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Split by bulge fraction at rt:
#   f_bulge(rt) = Vbulge^2(rt) / Vbar^2(rt)
#
# We test several thresholds and fit ONLY the normalization
# separately for low-bulge and high-bulge populations.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_two_population_normalization")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
ALPHA  = 0.175
BETA   = 0.55

# Bulge-fraction thresholds to try
THRESH_LIST = [0.05, 0.10, 0.20, 0.30, 0.40]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

def fit_C(subdf):
    x = subdf["carrier"].values.astype(float)
    y = subdf["V_flat_fit"].values.astype(float) ** 2
    m = np.isfinite(x) & np.isfinite(y) & (x > 0)
    return float(np.sum(x[m] * y[m]) / np.sum(x[m] ** 2))

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TWO-POP TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2 = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
        vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
        vbul2 = np.maximum(UPS_BUL * rot["vbul"] * np.abs(rot["vbul"]), 0.0)
        vbar2 = vgas2 + vdisk2 + vbul2

        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_interp = interp1d(rot["r"], vbul2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        vbul2_t = max(float(vbul2_interp(rt)), 0.0)

        f_bulge_t = vbul2_t / vbar2_t if vbar2_t > 0 else 0.0
        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "f_bulge_t": f_bulge_t,
            "carrier": carrier,
            "V_flat_fit": float(np.max(rot["vobs"])),  # same target convention used in these tests
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for thresh in THRESH_LIST:
    low = df[df["f_bulge_t"] < thresh].copy()
    high = df[df["f_bulge_t"] >= thresh].copy()

    if len(low) < 10 or len(high) < 10:
        continue

    C_low = fit_C(low)
    C_high = fit_C(high)

    eval_rows = []
    for _, row in df.iterrows():
        C_use = C_low if row["f_bulge_t"] < thresh else C_high
        V_flat_pred = np.sqrt(max(C_use * row["carrier"], 0.0))
        V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

        V_obs_pred = interp1d(
            row["r_grid"], V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(row["r_obs"])

        rmse = safe_rmse(row["v_obs"], V_obs_pred)

        eval_rows.append({
            "vmax_obs": row["vmax_obs"],
            "rmse": rmse,
        })

    eval_df = pd.DataFrame(eval_rows)
    high_mask = eval_df["vmax_obs"] >= 150.0

    summary_rows.append({
        "thresh": thresh,
        "n_lowbulge": int(len(low)),
        "n_highbulge": int(len(high)),
        "C_lowbulge": C_low,
        "C_highbulge": C_high,
        "C_ratio_high_over_low": C_high / C_low if C_low != 0 else np.nan,
        "median_rmse": float(eval_df["rmse"].median()),
        "p90_rmse": float(np.percentile(eval_df["rmse"], 90)),
        "high_bin_median_rmse": float(eval_df.loc[high_mask, "rmse"].median()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("TWO-POPULATION NORMALIZATION SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "two_population_normalization_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "two_population_normalization_summary.txt"), "w", encoding="utf-8") as f:
    f.write("TWO-POPULATION NORMALIZATION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TWO-POP TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

TWO-POPULATION NORMALIZATION SUMMARY
 thresh  n_lowbulge  n_highbulge  C_lowbulge  C_highbulge  C_ratio_high_over_low  median_rmse  p90_rmse  high_bin_median_rmse
   0.40         156           19  146.447719   203.300229               1.388210    20.603011 76.133411             61.506610
   0.30         155           20  147.671731   195.243356               1.322144    20.699052 76.818147             61.526319
   0.05         144           31  142.754414   174.181652               1.220149    20.969344 78.742874             63.592373
   0.20         151           24  141.596637   186.750763               1.318893    21.522831 79.221134             63.731267
   0.10         146           29  138.148873   182.260437               1.319305    21.745393 80.295051             64.23

In [ ]:
# ============================================================
# DERIVED SQRT-BRIDGE TEST
# ------------------------------------------------------------
# Field:
#   (1/r^2) d/dr (r^2 dm/dr) - (1/Rs^2)(m-m_inf) = -A rho_eff
#
# Define:
#   u(r) = m(r) - m_inf
#   U(r) = ∫_0^r u(s) ds
#
# Derived observable law:
#   g_obs(r) = K * U(r) / r
# so
#   V(r)^2 = K * U(r)
#   V(r)   = sqrt(K * U(r))
#
# One global K across the full catalog.
# Fixed source branch and fixed Rs = 1.5.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_derived_sqrt_bridge_test")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# BUILD FIELD TABLE
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()

rows = []
fails = []

print("=" * 60)
print("BUILDING SQRT-BRIDGE FIELD TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        if not np.isfinite(np.max(U)) or np.max(U) <= 0:
            raise RuntimeError("Invalid U profile")

        rows.append({
            "galaxy": galaxy,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": float(np.max(U)),
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

field_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(field_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

# ------------------------------------------------------------
# FIT GLOBAL K FROM V^2 = K U
# ------------------------------------------------------------

num = 0.0
den = 0.0

for _, row in field_df.iterrows():
    U_obs = interp1d(
        row["r_grid"], row["U_grid"],
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    m = np.isfinite(U_obs) & np.isfinite(row["v_obs"]) & (U_obs > 0)
    if np.sum(m) == 0:
        continue

    x = U_obs[m]
    y = row["v_obs"][m] ** 2
    num += np.sum(x * y)
    den += np.sum(x * x)

if den <= 0:
    raise RuntimeError("Could not fit global K")

K_global = num / den

# ------------------------------------------------------------
# EVALUATE
# ------------------------------------------------------------

res_rows = []

for _, row in field_df.iterrows():
    V_theory = np.sqrt(np.maximum(K_global * row["U_grid"], 0.0))
    V_obs_pred = interp1d(
        row["r_grid"], V_theory,
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    rmse = safe_rmse(row["v_obs"], V_obs_pred)
    mae = safe_mae(row["v_obs"], V_obs_pred)
    chi2_mean = float(np.mean(((row["v_obs"] - V_obs_pred) / row["e_obs"]) ** 2))

    obs_shape = row["v_obs"] / max(float(np.max(row["v_obs"])), 1.0e-12)
    pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
    shape_rmse = safe_rmse(obs_shape, pred_shape)

    V_flat_pred = float(np.sqrt(max(K_global * row["U_inf"], 0.0)))

    res_rows.append({
        "galaxy": row["galaxy"],
        "n_points": len(row["r_obs"]),
        "vmax_obs": float(np.max(row["v_obs"])),
        "U_inf": row["U_inf"],
        "V_flat_pred": V_flat_pred,
        "rmse": rmse,
        "mae": mae,
        "chi2_mean": chi2_mean,
        "shape_rmse": shape_rmse,
    })

res_df = pd.DataFrame(res_rows)

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "K_global": float(K_global),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

print()
print("=" * 60)
print("DERIVED SQRT-BRIDGE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

res_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_results.csv"), index=False)
fail_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_failures.csv"), index=False)
bin_summary.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_bin_summary.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_best20.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_worst20.csv"), index=False)

with open(os.path.join(OUTDIR, "derived_sqrt_bridge_summary.txt"), "w", encoding="utf-8") as f:
    f.write("DERIVED SQRT-BRIDGE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING SQRT-BRIDGE FIELD TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

DERIVED SQRT-BRIDGE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
K_global               = 36582.793628590174
median_rmse            = 43.36683077126389
mean_rmse              = 57.61969794883952
p90_rmse               = 121.34847005642484
p95_rmse               = 148.87374190105436
median_mae             = 41.33523616131542
median_chi2_mean       = 108.24295608329402
median_shape_rmse      = 0.1035927421678175

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 41.847865 39.337542 149.385139    0.093540
     mid 28.832875 28.342812  42.857838    0.073662
    high 77.419750 65.099313 212.123979    0.147856

BEST 20 BY RMSE
     galaxy  vmax_obs  V_flat_pred      rmse       mae  shape_rmse
   UGC04483      24.3    

In [ ]:
# ============================================================
# GLOBAL BALANCE / ENERGY-CLOSURE TEST
# ------------------------------------------------------------
# Purpose:
#   Test whether the missing amplitude V_flat is controlled by
#   global balance integrals of the surviving screened field.
#
# Starting field:
#   (1/r^2) d/dr (r^2 dm/dr) - (1/Rs^2)(m-m_inf) = -A rho_eff
#
# Define:
#   u(r) = m(r) - m_inf
#   U(r) = ∫_0^r u(s) ds
#
# New diagnostics:
#   E1 = ∫ (du/dr)^2 r^2 dr                 # gradient / field energy
#   E2 = ∫ (u^2 / Rs^2) r^2 dr              # screening energy
#   E3 = ∫ rho_eff(r) u(r) r^2 dr           # source-field coupling
#
# We then test simple no-fit amplitude laws:
#   V_flat^2 ~ C * E3
#   V_flat^2 ~ C * (E1 + E2)
#   V_flat^2 ~ C * E3 / (E1 + E2)
#   V_flat^4 ~ C * E3
#   V_flat^4 ~ C * (E1 + E2)
#   V_flat^4 ~ C * E3 / (E1 + E2)
#
# plus optional log-regression diagnostics across:
#   [U_inf, int_U, E1, E2, E3, E3/(E1+E2), r50]
#
# One-cell Colab script.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_global_balance_test")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED SURVIVING MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)

    return {
        "r_grid": r,
        "m_grid": m,
        "rho_grid": rho,
    }

def compute_r50_from_u(r, u):
    U = cumulative_trapezoid(np.maximum(u, 0.0), r, initial=0.0)
    U_inf = float(np.max(U))
    if not np.isfinite(U_inf) or U_inf <= 0:
        return np.nan
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    if idx <= 0:
        return float(r[0])
    if idx >= len(r):
        return float(r[-1])
    x0, x1 = U[idx - 1], U[idx]
    y0, y1 = r[idx - 1], r[idx]
    if x1 == x0:
        return float(y0)
    return float(y0 + (target - x0) * (y1 - y0) / (x1 - x0))

def fit_single_constant(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sum(x[m] * y[m]) / np.sum(x[m] * x[m]))

def evaluate_amp_law(df, x_col, target="vflat2"):
    if target == "vflat2":
        y = df["vmax_obs"].values**2
    elif target == "vflat4":
        y = df["vmax_obs"].values**4
    else:
        raise ValueError("target must be vflat2 or vflat4")

    x = df[x_col].values
    C = fit_single_constant(x, y)

    if not np.isfinite(C):
        return None

    y_pred = C * x
    if target == "vflat2":
        vflat_pred = np.sqrt(np.maximum(y_pred, 0.0))
    else:
        vflat_pred = np.power(np.maximum(y_pred, 0.0), 0.25)

    amp_rmse = safe_rmse(df["vmax_obs"].values, vflat_pred)
    amp_mae = safe_mae(df["vmax_obs"].values, vflat_pred)

    return {
        "driver": x_col,
        "target": target,
        "C_global": C,
        "amp_rmse": amp_rmse,
        "amp_mae": amp_mae,
    }

def log_regression(X_cols, y_col, df):
    X = df[X_cols].copy()
    y = df[y_col].copy()

    mask = np.isfinite(y)
    for c in X_cols:
        mask &= np.isfinite(X[c]) & (X[c] > 0)
    mask &= (y > 0)

    X = X.loc[mask]
    y = y.loc[mask]

    if len(X) < len(X_cols) + 5:
        return None

    Xmat = np.column_stack([np.ones(len(X))] + [np.log(X[c].values) for c in X_cols])
    yvec = np.log(y.values)

    beta, *_ = np.linalg.lstsq(Xmat, yvec, rcond=None)

    yhat = np.exp(Xmat @ beta)
    rmse = safe_rmse(y.values, yhat)
    mae = safe_mae(y.values, yhat)

    out = {
        "formula_target": y_col,
        "n_used": int(len(X)),
        "intercept_exp": float(np.exp(beta[0])),
        "rmse": rmse,
        "mae": mae,
    }
    for i, c in enumerate(X_cols, start=1):
        out[f"coef_{c}"] = float(beta[i])
    return out

# ------------------------------------------------------------
# BUILD FIELD TABLE
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()

rows = []
fails = []

print("=" * 70)
print("BUILDING GLOBAL BALANCE FIELD TABLE")
print("=" * 70)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        solved = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = solved["r_grid"]
        m = solved["m_grid"]
        rho = solved["rho_grid"]

        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        if not np.isfinite(np.max(U)) or np.max(U) <= 0:
            raise RuntimeError("Invalid U profile")

        du_dr = np.gradient(u, r)

        # Global balance integrals
        E1 = float(trapezoid((du_dr**2) * (r**2), r))
        E2 = float(trapezoid((u**2 / (Rs_fixed**2)) * (r**2), r))
        E3 = float(trapezoid(rho * u * (r**2), r))

        int_U = float(trapezoid(U, r))
        U_inf = float(np.max(U))
        r50 = compute_r50_from_u(r, u)

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "n_points": int(len(rot["r"])),
            "U_inf": U_inf,
            "int_U": int_U,
            "r50": r50,
            "E1_grad": E1,
            "E2_screen": E2,
            "E3_source_coupling": E3,
            "E_balance_sum": E1 + E2,
            "E_balance_ratio": E3 / max(E1 + E2, 1.0e-30),
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

feat_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(feat_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

# ------------------------------------------------------------
# GLOBAL NO-FIT AMPLITUDE TESTS
# ------------------------------------------------------------

tests = []
for driver in [
    "U_inf",
    "int_U",
    "E1_grad",
    "E2_screen",
    "E3_source_coupling",
    "E_balance_sum",
    "E_balance_ratio",
]:
    out2 = evaluate_amp_law(feat_df, driver, target="vflat2")
    out4 = evaluate_amp_law(feat_df, driver, target="vflat4")
    if out2 is not None:
        tests.append(out2)
    if out4 is not None:
        tests.append(out4)

tests_df = pd.DataFrame(tests).sort_values(["amp_rmse", "amp_mae"]).reset_index(drop=True)

# ------------------------------------------------------------
# OPTIONAL LOG-REGRESSION DIAGNOSTICS
# ------------------------------------------------------------

log_models = []

candidate_sets = [
    ["U_inf"],
    ["int_U"],
    ["E3_source_coupling"],
    ["E_balance_ratio"],
    ["int_U", "r50", "U_inf"],
    ["E3_source_coupling", "r50"],
    ["E_balance_sum", "E3_source_coupling"],
    ["E_balance_ratio", "r50"],
    ["E1_grad", "E2_screen", "E3_source_coupling"],
    ["int_U", "r50", "E3_source_coupling"],
]

for cols in candidate_sets:
    out = log_regression(cols, "vmax_obs", feat_df)
    if out is not None:
        out["model_cols"] = ", ".join(cols)
        log_models.append(out)

log_df = pd.DataFrame(log_models).sort_values(["rmse", "mae"]).reset_index(drop=True)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print()
print("=" * 70)
print("GLOBAL BALANCE FEATURE SUMMARY")
print("=" * 70)
print(f"successful_galaxies = {len(feat_df)}")
print(f"failed_galaxies     = {len(fail_df)}")

print()
print("=" * 70)
print("BEST GLOBAL NO-FIT AMPLITUDE LAWS")
print("=" * 70)
if len(tests_df):
    print(tests_df.to_string(index=False))
else:
    print("No valid amplitude tests.")

print()
print("=" * 70)
print("BEST LOG-REGRESSION DIAGNOSTICS")
print("=" * 70)
if len(log_df):
    show_cols = ["model_cols", "intercept_exp", "rmse", "mae"] + [c for c in log_df.columns if c.startswith("coef_")]
    print(log_df[show_cols].to_string(index=False))
else:
    print("No valid log-regression models.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

feat_df.to_csv(os.path.join(OUTDIR, "global_balance_features.csv"), index=False)
fail_df.to_csv(os.path.join(OUTDIR, "global_balance_failures.csv"), index=False)
tests_df.to_csv(os.path.join(OUTDIR, "global_balance_nofit_tests.csv"), index=False)
log_df.to_csv(os.path.join(OUTDIR, "global_balance_log_diagnostics.csv"), index=False)

with open(os.path.join(OUTDIR, "global_balance_summary.txt"), "w", encoding="utf-8") as f:
    f.write("GLOBAL BALANCE FEATURE SUMMARY\n")
    f.write("=" * 70 + "\n")
    f.write(f"successful_galaxies = {len(feat_df)}\n")
    f.write(f"failed_galaxies     = {len(fail_df)}\n\n")

    f.write("BEST GLOBAL NO-FIT AMPLITUDE LAWS\n")
    f.write("=" * 70 + "\n")
    if len(tests_df):
        f.write(tests_df.to_string(index=False))
    else:
        f.write("No valid amplitude tests.")
    f.write("\n\n")

    f.write("BEST LOG-REGRESSION DIAGNOSTICS\n")
    f.write("=" * 70 + "\n")
    if len(log_df):
        show_cols = ["model_cols", "intercept_exp", "rmse", "mae"] + [c for c in log_df.columns if c.startswith("coef_")]
        f.write(log_df[show_cols].to_string(index=False))
    else:
        f.write("No valid log-regression models.")
    f.write("\n")

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING GLOBAL BALANCE FIELD TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

GLOBAL BALANCE FEATURE SUMMARY
successful_galaxies = 175
failed_galaxies     = 0

BEST GLOBAL NO-FIT AMPLITUDE LAWS
            driver target     C_global   amp_rmse   amp_mae
             int_U vflat2 3.905848e+02  79.235962 54.697965
             U_inf vflat2 2.299660e+04  80.765368 57.009928
           E1_grad vflat4 8.384261e+09  83.589439 62.504562
   E_balance_ratio vflat2 2.300375e+03  84.063098 74.686174
             int_U vflat4 3.075622e+07  86.156884 74.770500
             U_inf vflat4 1.629426e+09  87.546771 76.096023
E3_source_coupling vflat4 2.695407e+07  88.187893 59.720184
     E_balance_sum vflat4 2.695417e+08  88.188070 59.720336
         E2_screen vflat4 2.750834e+08  90.018506 61.378212
   E_balance_ratio vflat4 1.273481e+08 100.930071 90.568111

In [ ]:
# ------------------------------------------------------------
# ENERGY / LENGTH-SCALE AMPLITUDE LAWS
# ------------------------------------------------------------

def test_amp_relation(df, driver, power=2):
    x = df[driver].values
    v = df["vmax_obs"].values

    m = np.isfinite(x) & np.isfinite(v) & (x > 0) & (v > 0)
    if np.sum(m) == 0:
        return None

    if power == 2:
        y = v[m]**2
    elif power == 4:
        y = v[m]**4
    else:
        raise ValueError

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(np.maximum(y_pred, 0))
    else:
        v_pred = np.power(np.maximum(y_pred, 0), 0.25)

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))

    return {
        "driver": driver,
        "power": power,
        "C_global": C,
        "amp_rmse": rmse,
        "amp_mae": mae,
    }

# Build new drivers
feat_df["E_over_r50"] = feat_df["E_balance_sum"] / np.maximum(feat_df["r50"], 1e-6)
feat_df["E3_over_r50"] = feat_df["E3_source_coupling"] / np.maximum(feat_df["r50"], 1e-6)
feat_df["intU_over_r50"] = feat_df["int_U"] / np.maximum(feat_df["r50"], 1e-6)

tests2 = []

for drv in ["E3_over_r50", "E_over_r50", "intU_over_r50"]:
    for p in [2, 4]:
        out = test_amp_relation(feat_df, drv, power=p)
        if out:
            tests2.append(out)

tests2_df = pd.DataFrame(tests2).sort_values(["amp_rmse", "amp_mae"]).reset_index(drop=True)

print()
print("=" * 70)
print("ENERGY / LENGTH-SCALE AMPLITUDE TESTS")
print("=" * 70)
print(tests2_df.to_string(index=False))

tests2_df.to_csv(os.path.join(OUTDIR, "energy_length_amplitude_tests.csv"), index=False)


ENERGY / LENGTH-SCALE AMPLITUDE TESTS
       driver  power     C_global   amp_rmse   amp_mae
intU_over_r50      2 1.821368e+03  79.813911 66.349374
   E_over_r50      4 2.627637e+09  81.307425 58.356850
  E3_over_r50      4 2.627617e+08  81.307513 58.357092
intU_over_r50      4 1.148377e+08  98.377114 88.444586
  E3_over_r50      2 3.012415e+03 106.112388 79.475328
   E_over_r50      2 3.012433e+04 106.112613 79.475728


In [ ]:
# ============================================================
# FLUX AT SCREENING RADIUS TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_flux_rs_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r_grid, m_grid = solve_field(rho_r, rho_vals)

    u = np.maximum(m_grid - m_inf, 0)
    du_dr = np.gradient(u, r_grid)

    # Flux at screening radius Rs
    flux_rs = (Rs_fixed**2) * np.interp(Rs_fixed, r_grid, du_dr)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "flux_rs": flux_rs,
    })

df = pd.DataFrame(rows)

print("\nFLUX AT Rs TEST")
print("="*40)

for p in [2, 4]:
    rmse, mae = test_relation(df["flux_rs"].values, df["vmax"].values, p)
    print(f"flux_rs power={p} RMSE={rmse:.2f} MAE={mae:.2f}")

df.to_csv(os.path.join(OUTDIR, "flux_rs_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

FLUX AT Rs TEST
flux_rs power=2 RMSE=108.70 MAE=81.82
flux_rs power=4 RMSE=102.05 MAE=79.49

Saved to: /content/mts_realdata_workspace_v2/mts_flux_rs_test


In [ ]:
# ============================================================
# MTS FULL AMPLITUDE CLOSURE TEST
# Computes all integrals + tests scalars, flux, ratios
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_full_amplitude_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def compute_r50(r, u):
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    return r[idx] if idx < len(r) else r[-1]

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)
    r50 = compute_r50(r, u)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    flux_rs = (Rs_fixed**2) * np.interp(Rs_fixed, r, du_dr)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "r50": r50,
        "E1": E1,
        "E2": E2,
        "E3": E3,
        "flux_rs": flux_rs,
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# BUILD RATIO DRIVERS
# ------------------------------------------------------------

df["intU_over_E3"] = df["int_U"] / np.maximum(df["E3"], 1e-8)
df["intU_r50_over_E3"] = (df["int_U"] * df["r50"]) / np.maximum(df["E3"], 1e-8)
df["Uinf_r50_over_E3"] = (df["U_inf"] * df["r50"]) / np.maximum(df["E3"], 1e-8)

# ------------------------------------------------------------
# TEST EVERYTHING
# ------------------------------------------------------------

print("\nAMPLITUDE LAW TESTS")
print("="*50)

drivers = [
    "U_inf",
    "int_U",
    "E1",
    "E2",
    "E3",
    "flux_rs",
    "intU_over_E3",
    "intU_r50_over_E3",
    "Uinf_r50_over_E3"
]

for drv in drivers:
    for p in [2, 4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "full_amplitude_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

AMPLITUDE LAW TESTS
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E1                   power=2 RMSE=  104.06 MAE=   76.05
E1                   power=4 RMSE=   83.11 MAE=   61.26
E2                   power=2 RMSE=  121.75 MAE=   98.94
E2                   power=4 RMSE=   89.67 MAE=   61.02
E3                   power=2 RMSE=  120.34 MAE=   97.18
E3                   power=4 RMSE=   87.95 MAE=   59.62
flux_rs              power=2 RMSE=  108.70 MAE=   81.82
flux_rs              power=4 RMSE=  102.05 MAE=   79.49
intU_over_E3         power=2 RMSE=  123.47 MAE=   95.25
intU_over_E3         power=4 RMSE=  100.63 MAE=   76.54
intU_r50_over_E3     power=2 RMSE=  104.59 MAE=   76.89
intU_r50_over_E3     power=4 RMSE=   94.18 MAE=   78.08
Uinf_r50_over_E3     power=2 RMSE=  103.42 MAE=   76.06
Uinf

In [ ]:
# ============================================================
# MTS ENERGY BALANCE AMPLITUDE TEST
# Full one-cell script
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_energy_balance_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "E1": E1,
        "E2": E2,
        "E3": E3
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# ENERGY RATIO DRIVERS
# ------------------------------------------------------------

df["E1_over_E3"] = df["E1"] / np.maximum(df["E3"], 1e-8)
df["E2_over_E3"] = df["E2"] / np.maximum(df["E3"], 1e-8)
df["E12_over_E3"] = (df["E1"] + df["E2"]) / np.maximum(df["E3"], 1e-8)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

print("\nENERGY BALANCE AMPLITUDE TESTS")
print("="*50)

drivers = ["E1_over_E3", "E2_over_E3", "E12_over_E3"]

for drv in drivers:
    for p in [2, 4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "energy_balance_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

ENERGY BALANCE AMPLITUDE TESTS
E1_over_E3           power=2 RMSE=  103.21 MAE=   80.26
E1_over_E3           power=4 RMSE=   98.25 MAE=   83.79
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
E12_over_E3          power=2 RMSE=   84.06 MAE=   74.68
E12_over_E3          power=4 RMSE=  100.93 MAE=   90.57

Saved to: /content/mts_realdata_workspace_v2/mts_energy_balance_test


In [ ]:
# ============================================================
# MTS FINAL COMBINED MODEL (END-TO-END TEST)
# Field + Energy Amplitude + Shape Law
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_final_combined_model")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def safe_rmse(a, b):
    return np.sqrt(np.mean((a - b)**2))

def safe_mae(a, b):
    return np.mean(np.abs(a - b))

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Running FINAL combined model...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)

    # --- Energy amplitude law ---
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    amp_driver = E2 / max(E3, 1e-8)

    rows.append({
        "galaxy": os.path.basename(f),
        "r_grid": r,
        "U_grid": U,
        "U_inf": U_inf,
        "amp_driver": amp_driver,
        "r_obs": rot["r"],
        "v_obs": rot["vobs"],
        "e_obs": rot["ev"],
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# FIT GLOBAL K FROM AMPLITUDE LAW
# Vflat^2 = K * amp_driver
# ------------------------------------------------------------

num = 0.0
den = 0.0

for _, row in df.iterrows():
    Vflat_obs = np.max(row["v_obs"])
    x = row["amp_driver"]
    y = Vflat_obs**2
    num += x * y
    den += x * x

K_global = num / den
print("Global amplitude constant K =", K_global)

# ------------------------------------------------------------
# EVALUATE FULL MODEL
# ------------------------------------------------------------

res_rows = []

for _, row in df.iterrows():
    r = row["r_grid"]
    U = row["U_grid"]
    U_inf = row["U_inf"]

    V_flat = np.sqrt(K_global * row["amp_driver"])
    V_theory = V_flat * np.sqrt(np.maximum(U / U_inf, 0))

    V_pred = interp1d(r, V_theory, bounds_error=False, fill_value="extrapolate")(row["r_obs"])

    rmse = safe_rmse(row["v_obs"], V_pred)
    mae = safe_mae(row["v_obs"], V_pred)

    res_rows.append({
        "galaxy": row["galaxy"],
        "vmax_obs": np.max(row["v_obs"]),
        "vflat_pred": V_flat,
        "rmse": rmse,
        "mae": mae
    })

res_df = pd.DataFrame(res_rows)

print("\nFINAL MODEL SUMMARY")
print("="*50)
print("Galaxies:", len(res_df))
print("Median RMSE:", res_df["rmse"].median())
print("Mean RMSE:", res_df["rmse"].mean())
print("P90 RMSE:", np.percentile(res_df["rmse"], 90))

res_df.to_csv(os.path.join(OUTDIR, "final_model_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Running FINAL combined model...
Global amplitude constant K = 324911.5567336142

FINAL MODEL SUMMARY
Galaxies: 175
Median RMSE: 61.30444135585258
Mean RMSE: 63.140371080233045
P90 RMSE: 93.94683968458814

Saved to: /content/mts_realdata_workspace_v2/mts_final_combined_model


In [ ]:
# ============================================================
# MTS MASTER TEST — FULL PIPELINE + ALL AMPLITUDE LAWS
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_master_test")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def compute_r50(r, u):
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    return r[idx] if idx < len(r) else r[-1]

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)
    r50 = compute_r50(r, u)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "r50": r50,
        "E1": E1,
        "E2": E2,
        "E3": E3
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# BUILD DRIVERS
# ------------------------------------------------------------

df["E2_over_E3"] = df["E2"] / np.maximum(df["E3"],1e-8)
df["E2_over_E3_r50"] = df["E2"] / (np.maximum(df["E3"],1e-8) * np.maximum(df["r50"],1e-8))
df["Uinf_over_r50"] = df["U_inf"] / np.maximum(df["r50"],1e-8)
df["intU_over_r50"] = df["int_U"] / np.maximum(df["r50"],1e-8)

print("\nAMPLITUDE LAW TESTS")
print("="*60)

drivers = [
    "U_inf",
    "int_U",
    "E2_over_E3",
    "E2_over_E3_r50",
    "Uinf_over_r50",
    "intU_over_r50"
]

for drv in drivers:
    for p in [2,4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

Processing galaxies...

AMPLITUDE LAW TESTS
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
E2_over_E3_r50       power=2 RMSE=   91.42 MAE=   77.13
E2_over_E3_r50       power=4 RMSE=   99.40 MAE=   88.70
Uinf_over_r50        power=2 RMSE=   85.31 MAE=   70.92
Uinf_over_r50        power=4 RMSE=   95.86 MAE=   85.58
intU_over_r50        power=2 RMSE=   86.38 MAE=   71.66
intU_over_r50        power=4 RMSE=   95.70 MAE=   85.36


In [ ]:
# ------------------------------------------------------------
# COMBINED AMPLITUDE DRIVER
# ------------------------------------------------------------

df["E2_times_Uinf_over_E3"] = (df["E2"] * df["U_inf"]) / np.maximum(df["E3"],1e-8)

print("\nCOMBINED AMPLITUDE TEST")
print("="*60)

for p in [2,4]:
    rmse, mae = test_relation(df["E2_times_Uinf_over_E3"].values, df["vmax"].values, p)
    print(f"{'E2*Uinf/E3':20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")


COMBINED AMPLITUDE TEST
E2*Uinf/E3           power=2 RMSE=   82.53 MAE=   57.97
E2*Uinf/E3           power=4 RMSE=   85.99 MAE=   73.03


In [ ]:
# ============================================================
# MTS SCREENING-LENGTH AMPLITUDE TEST (CORRECTED)
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_Rs_test")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def estimate_Rs(r, u):
    mask = (r > 0.6*np.max(r)) & (u > 0)
    if np.sum(mask) < 5:
        return np.nan
    slope = np.polyfit(r[mask], np.log(u[mask]), 1)[0]
    if slope >= 0:
        return np.nan
    return -1.0 / slope

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 1e-12)
    U = cumulative_trapezoid(u, r, initial=0)

    U_inf = np.max(U)
    int_U = trapezoid(U, r)

    du_dr = np.gradient(u, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    r50 = r[np.searchsorted(U, 0.5*U_inf)]
    Rs_est = estimate_Rs(r, u)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "E2_over_E3": E2 / max(E3,1e-8),
        "Rs_est": Rs_est
    })

df = pd.DataFrame(rows)
df["inv_Rs"] = 1.0 / df["Rs_est"]

print("\nAMPLITUDE LAW COMPARISON")
print("="*60)

drivers = ["U_inf", "int_U", "E2_over_E3", "inv_Rs"]

for drv in drivers:
    for p in [2,4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "Rs_amplitude_test.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

AMPLITUDE LAW COMPARISON
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
inv_Rs               power=2 RMSE=   74.68 MAE=   54.49
inv_Rs               power=4 RMSE=   78.57 MAE=   66.53

Saved to: /content/mts_realdata_workspace_v2/mts_Rs_test


In [ ]:
# ============================================================
# NONLINEAR / DENSITY-DEPENDENT SCREENING TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.5   # screening strength — try 0.1, 0.3, 0.5, 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field_nonlinear(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# MAIN TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []

print("Running nonlinear screening model...")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field_nonlinear(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)

        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

    except:
        pass

print("\nNONLINEAR MODEL SUMMARY")
print("Galaxies:", len(rmse_list))
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

Running nonlinear screening model...

NONLINEAR MODEL SUMMARY
Galaxies: 175
Median RMSE: 43.0261836555489
Mean RMSE: 56.73852552405732
P90 RMSE: 109.05932633574832


In [ ]:
# ============================================================
# NONLINEAR SCREENING AUTO-SWEEP
# Tests multiple screening strengths automatically
# No editing required
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

BETA_LIST = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5]

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field_nonlinear(rho_r, rho_vals, BETA):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("NONLINEAR SCREENING SWEEP")
print("====================================")

for BETA in BETA_LIST:
    rmse_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field_nonlinear(rho_r, rho_vals, BETA)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            V_flat = np.max(rot["vobs"])
            V_model = V_flat * (U / np.max(U))

            V_interp = np.interp(rot["r"], r, V_model)
            rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
            rmse_list.append(rmse)

        except:
            pass

    print(f"BETA = {BETA:4.2f} | Median RMSE = {np.median(rmse_list):6.2f} | Mean RMSE = {np.mean(rmse_list):6.2f}")

NONLINEAR SCREENING SWEEP
BETA = 0.05 | Median RMSE =  40.93 | Mean RMSE =  52.83
BETA = 0.10 | Median RMSE =  41.34 | Mean RMSE =  53.58
BETA = 0.20 | Median RMSE =  41.82 | Mean RMSE =  54.69
BETA = 0.30 | Median RMSE =  42.24 | Mean RMSE =  55.51
BETA = 0.50 | Median RMSE =  43.03 | Mean RMSE =  56.74
BETA = 0.70 | Median RMSE =  43.66 | Mean RMSE =  57.64
BETA = 1.00 | Median RMSE =  44.25 | Mean RMSE =  58.65
BETA = 1.50 | Median RMSE =  44.86 | Mean RMSE =  59.85


In [ ]:
# ============================================================
# FIELD ENERGY AMPLITUDE TEST (NONLINEAR FIELD)
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.05   # best value from sweep

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

# ------------------------------------------------------------
# ENERGY TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

E_list = []
V_list = []

print("FIELD ENERGY AMPLITUDE TEST")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m, rho = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        du = np.gradient(u, r)

        # Field energy terms
        E_grad = trapezoid((du**2) * r**2, r)
        E_screen = trapezoid((rho * u**2) * r**2, r)
        E_source = trapezoid((rho * u) * r**2, r)

        E_total = E_grad + BETA * E_screen + A_src * E_source

        V_flat = np.max(rot["vobs"])

        E_list.append(E_total)
        V_list.append(V_flat**2)

    except:
        pass

E = np.array(E_list)
V2 = np.array(V_list)

C = np.sum(E * V2) / np.sum(E * E)
V_pred = np.sqrt(C * E)

rmse = np.sqrt(np.mean((np.sqrt(V2) - V_pred)**2))
mae = np.mean(np.abs(np.sqrt(V2) - V_pred))

print("\nENERGY AMPLITUDE RESULT")
print("RMSE:", rmse)
print("MAE :", mae)

FIELD ENERGY AMPLITUDE TEST

ENERGY AMPLITUDE RESULT
RMSE: 122.21977338847826
MAE : 99.63356046920654


In [ ]:
# ============================================================
# MEASURE EFFECTIVE Rs FROM NONLINEAR FIELD
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.05

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# MEASURE Rs
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

Rs_list = []
M_list = []
V_list = []

print("MEASURING EFFECTIVE Rs")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        # Define Rs as radius where U reaches 63% of saturation
        target = 0.63 * U_inf
        idx = np.searchsorted(U, target)
        Rs_eff = r[idx]

        # Baryonic mass proxy
        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        Rs_list.append(Rs_eff)
        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

Rs = np.array(Rs_list)
M = np.array(M_list)
V = np.array(V_list)

# Power law slopes
slope_Rs_M = np.polyfit(np.log(M), np.log(Rs), 1)[0]
slope_V4_M = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("\nSCALING RESULTS")
print("Rs vs M slope:", slope_Rs_M)
print("V^4 vs M slope:", slope_V4_M)

MEASURING EFFECTIVE Rs

SCALING RESULTS
Rs vs M slope: 0.07355623288897552
V^4 vs M slope: 0.745432302582731


In [ ]:
# ============================================================
# CUMULATIVE MASS SCREENING TEST
# Screening depends on enclosed mass, not density
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
M0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    # Enclosed mass
    M_enc = cumulative_trapezoid(rho * r**2, r, initial=0)

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (M_enc[i] + M0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []

print("CUMULATIVE-MASS SCREENING MODEL")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)

        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

    except:
        pass

print("\nMODEL RESULT")
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

CUMULATIVE-MASS SCREENING MODEL

MODEL RESULT
Median RMSE: 25.81479436536363
Mean RMSE: 36.65431547646416
P90 RMSE: 85.57752042115796


In [ ]:
# ============================================================
# SCREENING LAW SWEEP
# Tests multiple cumulative screening laws
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
M0    = 1.0
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

# ------------------------------------------------------------
# FIELD SOLVER WITH DIFFERENT SCREENING LAWS
# ------------------------------------------------------------

def solve_field(rho_r, rho_vals, mode):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    M_enc = cumulative_trapezoid(rho * r**2, r, initial=0)

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    u_tmp = np.zeros_like(r) + 1.0  # for field-dependent screening

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        if mode == "mass":
            screen = GAMMA / (M_enc[i] + M0)

        elif mode == "sqrt_mass":
            screen = GAMMA / (np.sqrt(M_enc[i] + M0))

        elif mode == "mass_pow_075":
            screen = GAMMA / ((M_enc[i] + M0)**0.75)

        elif mode == "field":
            screen = GAMMA / (u_tmp[i] + U0)

        elif mode == "mixed":
            screen = GAMMA / (np.sqrt((M_enc[i] + M0) * (u_tmp[i] + U0)))

        else:
            screen = 0.0

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN TESTS
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
modes = ["mass", "sqrt_mass", "mass_pow_075", "field", "mixed"]

print("SCREENING LAW SWEEP")
print("========================================")

for mode in modes:
    rmse_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field(rho_r, rho_vals, mode)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            V_flat = np.max(rot["vobs"])
            V_model = V_flat * (U / np.max(U))

            V_interp = np.interp(rot["r"], r, V_model)
            rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
            rmse_list.append(rmse)

        except:
            pass

    print(f"{mode:15s} | Median RMSE = {np.median(rmse_list):6.2f} | Mean RMSE = {np.mean(rmse_list):6.2f}")

SCREENING LAW SWEEP
mass            | Median RMSE =  25.81 | Mean RMSE =  36.65
sqrt_mass       | Median RMSE =  16.33 | Mean RMSE =  27.49
mass_pow_075    | Median RMSE =  21.47 | Mean RMSE =  32.16
field           | Median RMSE =  12.28 | Mean RMSE =  21.52
mixed           | Median RMSE =  16.92 | Mean RMSE =  28.11


In [ ]:
# ============================================================
# BTFR SLOPE TEST FOR FIELD-SCREENING MODEL
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
V_list = []

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

M = np.array(M_list)
V = np.array(V_list)

slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("BTFR slope (V^4 vs M):", slope)

BTFR slope (V^4 vs M): 0.745432302582731


In [ ]:
# ============================================================
# NONLINEAR DIFFUSION (p-LAPLACIAN) FIELD TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

A_src = 0.5

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

# ------------------------------------------------------------
# NONLINEAR DIFFUSION SOLVER
# ------------------------------------------------------------

def solve_nonlinear_diffusion(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    # Initialize u
    u = np.ones_like(r) * 0.1

    # Iterative solve
    for _ in range(200):
        du = np.gradient(u, r)
        flux = u * du
        dflux = np.gradient(r**2 * flux, r) / (r**2 + 1e-8)

        u_new = u + 0.1 * (-A_src * rho - dflux)

        u_new = np.maximum(u_new, 0)
        u = 0.5 * u + 0.5 * u_new

    return r, u

# ------------------------------------------------------------
# RUN TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []
M_list = []
V_list = []

print("NONLINEAR DIFFUSION MODEL")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, u = solve_nonlinear_diffusion(rho_r, rho_vals)

        U = cumulative_trapezoid(u, r, initial=0)
        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

        # mass proxy
        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

print("\nMODEL FIT")
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

# BTFR slope
M = np.array(M_list)
V = np.array(V_list)
slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("\nBTFR slope (V^4 vs M):", slope)

NONLINEAR DIFFUSION MODEL

MODEL FIT
Median RMSE: nan
Mean RMSE: nan
P90 RMSE: nan

BTFR slope (V^4 vs M): 0.745432302582731


In [ ]:
# ============================================================
# U_inf vs Mass Scaling Test
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
Uinf_list = []
V_list = []

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        M_list.append(M_bar)
        Uinf_list.append(U_inf)
        V_list.append(V_flat)

    except:
        pass

M = np.array(M_list)
Uinf = np.array(Uinf_list)
V = np.array(V_list)

alpha = np.polyfit(np.log(M), np.log(Uinf), 1)[0]
beta = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("U_inf vs M slope:", alpha)
print("V^4 vs M slope:", beta)

U_inf vs M slope: nan
V^4 vs M slope: 0.745432302582731


In [ ]:
# ============================================================
# MASS-TO-LIGHT SWEEP + BTFR SLOPE TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

UPS_DISK_LIST = [0.3, 0.5, 0.7, 1.0]
UPS_BUL_LIST  = [0.5, 1.0, 1.5]

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot, UPS_DISK, UPS_BUL):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff, vbar2

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("MASS-TO-LIGHT SWEEP (BTFR SLOPE)")
print("==========================================")

for UPS_DISK in UPS_DISK_LIST:
    for UPS_BUL in UPS_BUL_LIST:

        M_list = []
        V_list = []

        for f in files:
            try:
                rot = read_rotmod_file(f)
                rho_r, rho_vals, vbar2 = build_source(rot, UPS_DISK, UPS_BUL)
                r, m = solve_field(rho_r, rho_vals)

                u = np.maximum(m - m_inf, 0)
                U = cumulative_trapezoid(u, r, initial=0)
                if np.max(U) <= 0:
                    continue

                M_bar = np.trapz(vbar2 * rot["r"], rot["r"])
                V_flat = np.max(rot["vobs"])

                M_list.append(M_bar)
                V_list.append(V_flat)

            except:
                pass

        M = np.array(M_list)
        V = np.array(V_list)

        slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

        print(f"UPS_DISK={UPS_DISK:.1f}, UPS_BUL={UPS_BUL:.1f} | BTFR slope = {slope:.3f}")

MASS-TO-LIGHT SWEEP (BTFR SLOPE)
UPS_DISK=0.3, UPS_BUL=0.5 | BTFR slope = 0.799
UPS_DISK=0.3, UPS_BUL=1.0 | BTFR slope = 0.785
UPS_DISK=0.3, UPS_BUL=1.5 | BTFR slope = 0.776
UPS_DISK=0.5, UPS_BUL=0.5 | BTFR slope = 0.788
UPS_DISK=0.5, UPS_BUL=1.0 | BTFR slope = 0.779
UPS_DISK=0.5, UPS_BUL=1.5 | BTFR slope = 0.770
UPS_DISK=0.7, UPS_BUL=0.5 | BTFR slope = 0.780
UPS_DISK=0.7, UPS_BUL=1.0 | BTFR slope = 0.773
UPS_DISK=0.7, UPS_BUL=1.5 | BTFR slope = 0.767
UPS_DISK=1.0, UPS_BUL=0.5 | BTFR slope = 0.772
UPS_DISK=1.0, UPS_BUL=1.0 | BTFR slope = 0.767
UPS_DISK=1.0, UPS_BUL=1.5 | BTFR slope = 0.762


In [ ]:
# ============================================================
# MEASURE EFFECTIVE Rs FROM NONLINEAR FIELD
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10

# Nonlinear screening parameters
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 0.5
UPS_BUL  = 0.7

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff, vbar2

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
Rs_list = []

print("MEASURING Rs vs MASS")
print("====================================")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals, vbar2 = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 1e-8)

        # Measure Rs from outer slope
        outer = r > (0.6 * np.max(r))
        if np.sum(outer) < 10:
            continue

        slope = np.polyfit(r[outer], np.log(u[outer]), 1)[0]
        Rs_eff = -1.0 / slope

        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        if np.isfinite(Rs_eff) and Rs_eff > 0:
            M_list.append(M_bar)
            Rs_list.append(Rs_eff)

    except:
        pass

M = np.array(M_list)
Rs = np.array(Rs_list)

slope = np.polyfit(np.log(M), np.log(Rs), 1)[0]

print(f"Rs vs M slope = {slope:.3f}")

MEASURING Rs vs MASS
Rs vs M slope = 0.055


In [ ]:
# ============================================================
# GAMMA RESPONSE SWEEP
# V(r) = Vflat * (U/Uinf)^gamma
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

# Field model parameters (keep fixed)
m_inf = 0.02
A_src = 0.10
GAMMA_SCREEN = 0.5
U0 = 1.0

UPS_DISK = 0.5
UPS_BUL  = 0.7

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

GAMMA_LIST = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.ones_like(vobs)*5
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA_SCREEN / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("GAMMA SWEEP RESULTS")
print("============================================")

for gamma in GAMMA_LIST:

    rmse_list = []
    M_list = []
    V_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field(rho_r, rho_vals)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            U_obs = interp1d(r, U, fill_value="extrapolate")(rot["r"])
            U_obs = np.maximum(U_obs, 0)

            V_flat = np.max(rot["vobs"])
            V_pred = V_flat * (U_obs / np.max(U))**gamma

            rmse = np.sqrt(np.mean((rot["vobs"] - V_pred)**2))
            rmse_list.append(rmse)

            # BTFR
            M_bar = np.trapz(rot["vgas"]**2 + rot["vdisk"]**2 + rot["vbul"]**2, rot["r"])
            M_list.append(M_bar)
            V_list.append(V_flat)

        except:
            pass

    rmse_med = np.median(rmse_list)

    M = np.array(M_list)
    V = np.array(V_list)
    slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

    print(f"gamma={gamma:.2f} | Median RMSE={rmse_med:.2f} | BTFR slope={slope:.3f}")

GAMMA SWEEP RESULTS
gamma=0.15 | Median RMSE=18.67 | BTFR slope=0.994
gamma=0.20 | Median RMSE=17.60 | BTFR slope=0.994
gamma=0.25 | Median RMSE=16.20 | BTFR slope=0.994
gamma=0.30 | Median RMSE=15.53 | BTFR slope=0.994
gamma=0.35 | Median RMSE=14.94 | BTFR slope=0.994
gamma=0.40 | Median RMSE=14.35 | BTFR slope=0.994


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# WORST-GALAXY RESIDUAL ANATOMY
#
# Frozen best model:
#   lambda_b = 0.675
#   r_b      = 3.25
#
# Diagnose for the worst galaxies:
#   - where residual peaks in x = r/Rs
#   - whether miss is inner / mid / outer
#   - whether miss is mainly amplitude or shape
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

        rho_eff = rho * (1.0 + boost)

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Mbar": Mtot,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "compactness": M_enc[r <= 0.3*np.max(r)][-1] / max(Mtot, EPS) if np.any(r <= 0.3*np.max(r)) else np.nan,
            "bulge_frac": vb / denom,
            "gas_frac": vg / denom,
            "boost_med": med(boost),
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

# global amplitude
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = row["Vmax_obs"]**2
    num += x * y
    den += x * x
A_global = num / den

# evaluate
for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    v_bul_s = row["v_bul_s"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat2 = A_global * (Uinf / Rs)
    v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)
    v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
    v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

    resid = v_pred - v_obs
    frac_signed = resid / np.maximum(v_obs, 1.0)
    x = r / Rs

    row["rmse"] = np.sqrt(np.mean(resid**2))
    row["mae"] = np.mean(np.abs(resid))
    row["s_in"] = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    row["s_mid"] = np.median(frac_signed[(x >= 0.7) & (x < 1.5)]) if np.any((x >= 0.7) & (x < 1.5)) else np.nan
    row["s_out"] = np.median(frac_signed[x >= 1.5]) if np.any(x >= 1.5) else np.nan

    # anatomy
    i_abs = int(np.argmax(np.abs(resid)))
    i_neg = int(np.argmin(resid))
    i_pos = int(np.argmax(resid))

    row["x_peak_abs"] = x[i_abs]
    row["resid_peak_abs"] = resid[i_abs]
    row["x_peak_neg"] = x[i_neg]
    row["resid_peak_neg"] = resid[i_neg]
    row["x_peak_pos"] = x[i_pos]
    row["resid_peak_pos"] = resid[i_pos]

    # region-integrated deficits/excesses
    inner = x < 0.7
    mid = (x >= 0.7) & (x < 1.5)
    outer = x >= 1.5

    def neg_power(mask):
        vals = -resid[mask]
        vals = vals[vals > 0]
        return np.sum(vals) if len(vals) else 0.0

    def pos_power(mask):
        vals = resid[mask]
        vals = vals[vals > 0]
        return np.sum(vals) if len(vals) else 0.0

    row["neg_in"] = neg_power(inner)
    row["neg_mid"] = neg_power(mid)
    row["neg_out"] = neg_power(outer)
    row["pos_in"] = pos_power(inner)
    row["pos_mid"] = pos_power(mid)
    row["pos_out"] = pos_power(outer)

    # dominant failure mode
    negs = {"inner": row["neg_in"], "mid": row["neg_mid"], "outer": row["neg_out"]}
    poss = {"inner": row["pos_in"], "mid": row["pos_mid"], "outer": row["pos_out"]}
    dom_neg_region = max(negs, key=negs.get)
    dom_pos_region = max(poss, key=poss.get)

    total_neg = sum(negs.values())
    total_pos = sum(poss.values())

    if total_neg > 1.5 * total_pos:
        if dom_neg_region == "inner":
            row["anatomy"] = "underpowered_inner"
        elif dom_neg_region == "mid":
            row["anatomy"] = "underpowered_mid"
        else:
            row["anatomy"] = "underpowered_outer"
    elif total_pos > 1.5 * total_neg:
        if dom_pos_region == "inner":
            row["anatomy"] = "overpowered_inner"
        elif dom_pos_region == "mid":
            row["anatomy"] = "overpowered_mid"
        else:
            row["anatomy"] = "overpowered_outer"
    else:
        if row["neg_in"] > row["neg_mid"] and row["pos_out"] > row["pos_in"]:
            row["anatomy"] = "early-rise_timing"
        elif row["neg_mid"] > row["neg_in"] and row["neg_mid"] > row["neg_out"]:
            row["anatomy"] = "mid_support_missing"
        else:
            row["anatomy"] = "mixed_shape"

rows = sorted(rows, key=lambda z: z["rmse"])
worst = rows[-15:]

from collections import Counter
cnt = Counter([g["anatomy"] for g in worst])

print("WORST-GALAXY RESIDUAL ANATOMY")
print("="*140)
print(f"Frozen model: lambda_b={LAMBDA_B}, r_b={R_B}, A_global={A_global:.6f}")
print()

print("WORST 15 ANATOMY TABLE")
print("="*140)
print(f'{"name":<22} {"RMSE":>8} {"logM":>8} {"bulge":>8} {"compact":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"x|res|pk":>9} {"xnegpk":>8} {"neg_in":>9} {"neg_mid":>9} {"neg_out":>9} {"anatomy":>22}')
for g in worst:
    print(f'{g["name"]:<22} {g["rmse"]:8.2f} {np.log10(g["Mbar"]):8.3f} {g["bulge_frac"]:8.3f} {g["compactness"]:8.3f} {g["s_in"]:8.3f} {g["s_mid"]:8.3f} {g["s_out"]:8.3f} {g["x_peak_abs"]:9.3f} {g["x_peak_neg"]:8.3f} {g["neg_in"]:9.2f} {g["neg_mid"]:9.2f} {g["neg_out"]:9.2f} {g["anatomy"]:>22}')

print()
print("ANATOMY COUNTS IN WORST 15")
print("="*140)
for k, v in cnt.items():
    print(f"{k}: {v}")

print()
print("MEDIAN ANATOMY FEATURES BY TYPE")
print("="*140)
for k in cnt:
    chunk = [g for g in worst if g["anatomy"] == k]
    print(f"{k}:")
    print(f"  n              = {len(chunk)}")
    print(f"  median logM    = {med([np.log10(g['Mbar']) for g in chunk]):.3f}")
    print(f"  median bulge   = {med([g['bulge_frac'] for g in chunk]):.3f}")
    print(f"  median compact = {med([g['compactness'] for g in chunk]):.3f}")
    print(f"  median x|res|  = {med([g['x_peak_abs'] for g in chunk]):.3f}")
    print(f"  median x_neg   = {med([g['x_peak_neg'] for g in chunk]):.3f}")
    print(f"  median s_in    = {med([g['s_in'] for g in chunk]):.3f}")
    print(f"  median s_mid   = {med([g['s_mid'] for g in chunk]):.3f}")
    print(f"  median s_out   = {med([g['s_out'] for g in chunk]):.3f}")
    print()

WORST-GALAXY RESIDUAL ANATOMY
Frozen model: lambda_b=0.675, r_b=3.25, A_global=1.288393

WORST 15 ANATOMY TABLE
name                       RMSE     logM    bulge  compact     s_in    s_mid    s_out  x|res|pk   xnegpk    neg_in   neg_mid   neg_out                anatomy
IC4202_rotmod.dat         53.66   10.944    0.859    0.209   -0.247   -0.240   -0.229     0.846    0.846    317.34    655.93    652.11       underpowered_mid
UGC12506_rotmod.dat       56.64   10.994    0.733    0.508   -0.269   -0.264   -0.199     0.048    0.048    417.00    585.02    711.79     underpowered_outer
ESO563-G021_rotmod.dat    61.57   11.253    0.782    0.581    0.022   -0.220   -0.192     0.147    0.794    220.15    486.89    737.95     underpowered_outer
NGC5033_rotmod.dat        62.30   10.688    0.883    0.843   -0.321   -0.259   -0.152     0.051    0.051    578.33    288.76    360.15     underpowered_inner
UGC05253_rotmod.dat       62.76   10.805    0.914    0.736   -0.202    0.025    0.222     0.008   

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TWO-SCALE BULGE CHANNEL TEST
#
# Frozen broad bulge term:
#   lambda_b = 0.675
#   r_b      = 3.25
#
# Add compact bulge-core term:
#   lambda_c * Vbul^2 / (1 + r/r_c)
#
# Goal:
#   fix the dominant remaining class: underpowered_inner
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen broad bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
                "Vmax_obs": np.max(v_obs),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = row["Vmax_obs"]**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lambda_c, r_c):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        broad = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        core  = lambda_c * (v_bul_s**2) * f_lor(r / max(r_c, 1e-6))

        v_pred = np.sqrt(np.maximum(v_field2 + broad + core, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rr = dict(row)
        rr["rmse"] = np.sqrt(np.mean(resid**2))
        rr["mae"] = np.mean(np.abs(resid))
        rr["s_in"] = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
        rr["s_mid"] = np.median(frac_signed[(x >= 0.7) & (x < 1.5)]) if np.any((x >= 0.7) & (x < 1.5)) else np.nan
        rr["s_out"] = np.median(frac_signed[x >= 1.5]) if np.any(x >= 1.5) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["s_in"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "rows": rows_sorted,
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["s_in"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("TWO-SCALE BULGE CHANNEL TEST")
print("="*122)
print("Frozen broad term:")
print(f"  lambda_b = {LAMBDA_B}")
print(f"  r_b      = {R_B}")
print("Reference frozen broad-only model:")
print("  medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.61, max_jump ~ 15.84")
print()

rows = build_rows()
A_global = fit_amplitude(rows)
print("Galaxies solved:", len(rows))
print("A_global:", A_global)
print()

grid = [
    (0.05, 0.20),
    (0.05, 0.35),
    (0.05, 0.50),
    (0.10, 0.20),
    (0.10, 0.35),
    (0.10, 0.50),
    (0.15, 0.20),
    (0.15, 0.35),
    (0.15, 0.50),
    (0.20, 0.20),
    (0.20, 0.35),
    (0.20, 0.50),
    (0.25, 0.20),
    (0.25, 0.35),
    (0.25, 0.50),
]

all_results = []

for lambda_c, r_c in grid:
    res = evaluate(rows, A_global, lambda_c, r_c)
    all_results.append({
        "lambda_c": lambda_c,
        "r_c": r_c,
        **{k: v for k, v in res.items() if k != "rows"}
    })

    print(f"(lambda_c={lambda_c:.2f}, r_c={r_c:.2f})")
    print("-"*122)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*122)
print(f'{"lam_c":>8} {"r_c":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["lambda_c"]:8.2f} {r["r_c"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*122)
    print("lambda_c =", best["lambda_c"])
    print("r_c =", best["r_c"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])

TWO-SCALE BULGE CHANNEL TEST
Frozen broad term:
  lambda_b = 0.675
  r_b      = 3.25
Reference frozen broad-only model:
  medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.61, max_jump ~ 15.84

Galaxies solved: 60
A_global: 1.2883928302627974

(lambda_c=0.05, r_c=0.20)
--------------------------------------------------------------------------------------------------------------------------
median RMSE: 26.313447518937668
median signed inner residual: 0.019533719898157373
median low-mass RMSE: 13.542908344947302
median high-mass RMSE: 45.59975489412495
largest adjacent decile jump: 15.879317423163656

(lambda_c=0.05, r_c=0.35)
--------------------------------------------------------------------------------------------------------------------------
median RMSE: 26.30261447869684
median signed inner residual: 0.019838992700791105
median low-mass RMSE: 13.534500829519313
median high-mass RMSE: 45.62004160858885
largest adjacent decile jump: 15.899766580800772

(lambda_c=0.05, r_c=0.50)
------

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# BULGE-DEPENDENT BULGE CHANNEL
#
# lambda_b_eff = lambda0 + lambda1 * bulge_fraction
#
# Frozen r_b = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Build galaxy rows
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * (1.0 + boost)

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Mbar": Mtot,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

# Fit amplitude
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = row["Vmax_obs"]**2
    num += x * y
    den += x * x
A_global = num / den

print("Galaxies solved:", len(rows))
print("A_global:", A_global)
print()

# ------------------------------------------------------------
# Sweep lambda0, lambda1
# ------------------------------------------------------------
grid = [
    (0.40, 0.20),
    (0.40, 0.30),
    (0.40, 0.40),
    (0.50, 0.20),
    (0.50, 0.30),
    (0.50, 0.40),
    (0.60, 0.10),
    (0.60, 0.20),
    (0.60, 0.30),
    (0.65, 0.10),
    (0.65, 0.20),
    (0.65, 0.30),
]

print("BULGE-DEPENDENT LAMBDA TEST")
print("="*110)

results = []

for lambda0, lambda1 in grid:

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        B = row["bulge_frac"]

        lambda_eff = lambda0 + lambda1 * B

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)
        v_bulge2 = lambda_eff * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((lambda0, lambda1, med_rmse, med_sin, med_high))

    print(f"(lambda0={lambda0:.2f}, lambda1={lambda1:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[4], z[2]))[0]
print("lambda0 =", best[0])
print("lambda1 =", best[1])
print("median RMSE =", best[2])
print("median signed inner =", best[3])
print("median high-mass RMSE =", best[4])

Galaxies solved: 60
A_global: 1.2883928302627974

BULGE-DEPENDENT LAMBDA TEST
(lambda0=0.40, lambda1=0.20) -> medRMSE=27.028, s_in=0.007, high-mass=47.014
(lambda0=0.40, lambda1=0.30) -> medRMSE=26.404, s_in=0.012, high-mass=45.740
(lambda0=0.40, lambda1=0.40) -> medRMSE=26.187, s_in=0.017, high-mass=46.607
(lambda0=0.50, lambda1=0.20) -> medRMSE=26.357, s_in=0.016, high-mass=45.584
(lambda0=0.50, lambda1=0.30) -> medRMSE=26.142, s_in=0.022, high-mass=46.715
(lambda0=0.50, lambda1=0.40) -> medRMSE=25.946, s_in=0.030, high-mass=48.429
(lambda0=0.60, lambda1=0.10) -> medRMSE=26.311, s_in=0.019, high-mass=45.664
(lambda0=0.60, lambda1=0.20) -> medRMSE=26.098, s_in=0.024, high-mass=46.824
(lambda0=0.60, lambda1=0.30) -> medRMSE=25.905, s_in=0.038, high-mass=48.541
(lambda0=0.65, lambda1=0.10) -> medRMSE=26.180, s_in=0.023, high-mass=46.042
(lambda0=0.65, lambda1=0.20) -> medRMSE=25.978, s_in=0.033, high-mass=47.731
(lambda0=0.65, lambda1=0.30) -> medRMSE=25.795, s_in=0.043, high-mass=49.36

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SATURATING BULGE-STRENGTH LAW TEST
#
# lambda_b(B) = lambda_min + (lambda_max - lambda_min) * B / (B + B0)
#
# Frozen:
#   r_b = 3.25
#   field backbone = data-fitted constitutive law
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def lambda_sat(B, lam_min, lam_max, B0):
    return lam_min + (lam_max - lam_min) * (B / (B + B0 + EPS))

# ------------------------------------------------------------
# Build rows once
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * (1.0 + boost)

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Mbar": Mtot,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = row["Vmax_obs"]**2
    num += x * y
    den += x * x
A_global = num / den

print("Galaxies solved:", len(rows))
print("A_global:", A_global)
print()

grid = [
    (0.45, 0.65, 0.10),
    (0.45, 0.65, 0.20),
    (0.45, 0.65, 0.30),
    (0.45, 0.70, 0.10),
    (0.45, 0.70, 0.20),
    (0.45, 0.70, 0.30),
    (0.50, 0.65, 0.10),
    (0.50, 0.65, 0.20),
    (0.50, 0.65, 0.30),
    (0.50, 0.70, 0.10),
    (0.50, 0.70, 0.20),
    (0.50, 0.70, 0.30),
    (0.55, 0.70, 0.10),
    (0.55, 0.70, 0.20),
    (0.55, 0.70, 0.30),
]

print("SATURATING BULGE-STRENGTH TEST")
print("="*118)

results = []

for lam_min, lam_max, B0 in grid:
    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        B = row["bulge_frac"]

        lam_eff = lambda_sat(B, lam_min, lam_max, B0)

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)
        v_bulge2 = lam_eff * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((lam_min, lam_max, B0, med_rmse, med_sin, med_high))

    print(f"(lam_min={lam_min:.2f}, lam_max={lam_max:.2f}, B0={B0:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[5], z[3]))[0]
print("lam_min =", best[0])
print("lam_max =", best[1])
print("B0 =", best[2])
print("median RMSE =", best[3])
print("median signed inner =", best[4])
print("median high-mass RMSE =", best[5])

Galaxies solved: 60
A_global: 1.2883928302627974

SATURATING BULGE-STRENGTH TEST
(lam_min=0.45, lam_max=0.65, B0=0.10) -> medRMSE=26.463, s_in=0.015, high-mass=46.257
(lam_min=0.45, lam_max=0.65, B0=0.20) -> medRMSE=26.598, s_in=0.014, high-mass=46.495
(lam_min=0.45, lam_max=0.65, B0=0.30) -> medRMSE=26.838, s_in=0.012, high-mass=46.693
(lam_min=0.45, lam_max=0.70, B0=0.10) -> medRMSE=26.340, s_in=0.019, high-mass=45.617
(lam_min=0.45, lam_max=0.70, B0=0.20) -> medRMSE=26.400, s_in=0.017, high-mass=45.910
(lam_min=0.45, lam_max=0.70, B0=0.30) -> medRMSE=26.450, s_in=0.015, high-mass=46.156
(lam_min=0.50, lam_max=0.65, B0=0.10) -> medRMSE=26.447, s_in=0.016, high-mass=46.184
(lam_min=0.50, lam_max=0.65, B0=0.20) -> medRMSE=26.484, s_in=0.014, high-mass=46.362
(lam_min=0.50, lam_max=0.65, B0=0.30) -> medRMSE=26.620, s_in=0.013, high-mass=46.511
(lam_min=0.50, lam_max=0.70, B0=0.10) -> medRMSE=26.325, s_in=0.019, high-mass=45.610
(lam_min=0.50, lam_max=0.70, B0=0.20) -> medRMSE=26.373, s_

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# RESPONSE-LAW TEST
#
# Frozen field backbone + frozen broad bulge channel
#
# Test:
#   V_field^2 = Vinf^2 * (U/Uinf)^q
#   V_pred^2  = V_field^2 + lambda_b * Vbul^2 / (1 + r/r_b)
#
# Only q changes.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
PHI_S  = 40000.0
N_SCR  = 1.0

# Constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen broad bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Build rows once
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * (1.0 + boost)

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Mbar": Mtot,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

print("Galaxies solved:", len(rows))
print()

# ------------------------------------------------------------
# Sweep q
# ------------------------------------------------------------
grid_q = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]

results = []

print("RESPONSE-LAW q SWEEP")
print("="*118)

for q in grid_q:
    # fit amplitude for this q
    num = 0.0
    den = 0.0
    for row in rows:
        x = (row["Uinf"] / row["Rs"])
        y = row["Vmax_obs"]**2
        num += x * y
        den += x * x
    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), q)
        v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((q, med_rmse, med_sin, med_high))

    print(f"(q={q:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[3], z[1]))[0]
print("q =", best[0])
print("median RMSE =", best[1])
print("median signed inner =", best[2])
print("median high-mass RMSE =", best[3])

Galaxies solved: 60

RESPONSE-LAW q SWEEP
(q=0.25) -> medRMSE=25.110, s_in=0.107, high-mass=49.459
(q=0.30) -> medRMSE=25.618, s_in=0.081, high-mass=49.032
(q=0.35) -> medRMSE=25.233, s_in=0.049, high-mass=49.671
(q=0.40) -> medRMSE=26.332, s_in=0.019, high-mass=45.613
(q=0.45) -> medRMSE=27.418, s_in=-0.010, high-mass=46.662
(q=0.50) -> medRMSE=28.073, s_in=-0.043, high-mass=47.696
(q=0.60) -> medRMSE=28.049, s_in=-0.088, high-mass=49.708

BEST ROW
q = 0.4
median RMSE = 26.332060122115585
median signed inner = 0.019081117421516434
median high-mass RMSE = 45.612599486052986


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SOURCE-LAW ETA SWEEP
#
# Test:
#   rho_eff = rho * (1 + boost(g_b))^eta
#
# Frozen:
#   p = 3.5
#   kappa = 0.10
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field/operator
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
Q_RESP = 0.40
PHI_S  = 40000.0
N_SCR  = 1.0

# Frozen constitutive boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen broad bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Read galaxies once
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "Vmax_obs": np.max(v_obs),
            "bulge_frac": vb / denom,
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
print()

etas = [0.70, 0.85, 1.00, 1.15, 1.30, 1.50]

print("SOURCE-LAW ETA SWEEP")
print("="*118)

results = []

for eta in etas:
    rows = []

    for g in galaxies:
        r = g["r"]
        v_obs = g["v_obs"]
        v_bul_s = g["v_bul_s"]
        v_bar2 = g["v_bar2"]
        rho = g["rho"]
        g_b = g["g_b"]

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

        rho_eff = rho * np.power(1.0 + boost, eta)

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": g["name"],
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": g["Mbar"],
        })

    # fit amplitude for this eta
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
        v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((eta, med_rmse, med_sin, med_high))

    print(f"(eta={eta:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[3], z[1]))[0]
print("eta =", best[0])
print("median RMSE =", best[1])
print("median signed inner =", best[2])
print("median high-mass RMSE =", best[3])

Galaxies solved: 60

SOURCE-LAW ETA SWEEP
(eta=0.70) -> medRMSE=25.049, s_in=0.044, high-mass=46.254
(eta=0.85) -> medRMSE=25.551, s_in=0.037, high-mass=45.903
(eta=1.00) -> medRMSE=26.332, s_in=0.019, high-mass=45.613
(eta=1.15) -> medRMSE=26.754, s_in=0.007, high-mass=45.587
(eta=1.30) -> medRMSE=62.755, s_in=-0.421, high-mass=106.985
(eta=1.50) -> medRMSE=101.613, s_in=-0.605, high-mass=159.350

BEST ROW
eta = 1.15
median RMSE = 26.753861036578137
median signed inner = 0.007164187212712825
median high-mass RMSE = 45.58656650701822


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# MIXED-OPERATOR TEST
#
# Solve:
#   div( (|grad chi|^(p-2) + eps_lin) grad chi ) = rho_eff
#
# Frozen:
#   p      = 3.5
#   kappa  = 0.10
#   eta    = 1.15
#   q      = 0.40
#   lambda_b = 0.675
#   r_b      = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen model settings
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S  = 40000.0
N_SCR  = 1.0

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies once
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
print()

eps_grid = [0.0, 1e-4, 3e-4, 1e-3, 3e-3]

print("MIXED-OPERATOR EPSILON SWEEP")
print("="*118)

results = []

for eps_lin in eps_grid:
    rows = []

    for g in galaxies:
        r = g["r"]
        v_obs = g["v_obs"]
        v_bul_s = g["v_bul_s"]
        v_bar2 = g["v_bar2"]
        rho = g["rho"]
        g_b = g["g_b"]

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

        rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

        chi = solve_mixed_operator(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2, eps_lin)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": g["name"],
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": g["Mbar"],
        })

    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
        v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((eps_lin, med_rmse, med_sin, med_high))

    print(f"(eps_lin={eps_lin:.1e}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[3], z[1]))[0]
print("eps_lin =", best[0])
print("median RMSE =", best[1])
print("median signed inner =", best[2])
print("median high-mass RMSE =", best[3])

Galaxies solved: 60

MIXED-OPERATOR EPSILON SWEEP
(eps_lin=0.0e+00) -> medRMSE=27.207, s_in=0.020, high-mass=46.272
(eps_lin=1.0e-04) -> medRMSE=25.397, s_in=0.022, high-mass=46.215
(eps_lin=3.0e-04) -> medRMSE=25.397, s_in=0.022, high-mass=46.214
(eps_lin=1.0e-03) -> medRMSE=25.397, s_in=0.022, high-mass=46.215
(eps_lin=3.0e-03) -> medRMSE=25.397, s_in=0.022, high-mass=46.202

BEST ROW
eps_lin = 0.003
median RMSE = 25.39732636338576
median signed inner = 0.021961148008514512
median high-mass RMSE = 46.20187114222539


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# DENSITY-DEPENDENT STIFFNESS FLOOR TEST
#
# Solve:
#   div( (|grad chi|^(p-2) + eps(r)) grad chi ) = rho_eff
#
# with
#   eps(r) = eps0 * [1 + C_rho * (rho/rho_med)^m]
#
# Frozen:
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen model settings
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator_density_floor(r, rho_eff, v_bar2, rho_raw, rho_med, phi_s, n_scr, p_op, mu2, eps0, C_rho, m_rho, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    rho_ratio = np.clip(rho_raw / max(rho_med, 1e-30), 0.0, None)
    eps_r = eps0 * (1.0 + C_rho * np.power(rho_ratio, m_rho))
    eps_r = np.nan_to_num(eps_r, nan=eps0, posinf=eps0*(1+C_rho*10), neginf=eps0)
    eps_r = np.clip(eps_r, eps0, 1e3)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_r) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi, eps_r

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies once
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
print()

# Use a small floor since constant eps already saturated quickly
eps0_grid = [1e-4]
grid = [
    (1e-4, 0.0, 1.0),   # control: constant floor
    (1e-4, 0.5, 0.5),
    (1e-4, 0.5, 1.0),
    (1e-4, 0.5, 2.0),
    (1e-4, 1.0, 0.5),
    (1e-4, 1.0, 1.0),
    (1e-4, 1.0, 2.0),
    (1e-4, 2.0, 0.5),
    (1e-4, 2.0, 1.0),
    (1e-4, 2.0, 2.0),
]

print("DENSITY-DEPENDENT STIFFNESS FLOOR SWEEP")
print("="*118)

results = []

for eps0, C_rho, m_rho in grid:
    rows = []
    eps_meds = []

    for g in galaxies:
        r = g["r"]
        v_obs = g["v_obs"]
        v_bul_s = g["v_bul_s"]
        v_bar2 = g["v_bar2"]
        rho = g["rho"]
        g_b = g["g_b"]

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

        rho_pos = rho[rho > 0]
        rho_med = np.median(rho_pos) if len(rho_pos) else 1.0

        out = solve_mixed_operator_density_floor(
            r, rho_eff, v_bar2, rho, rho_med,
            PHI_S, N_SCR, P_OP, MU2,
            eps0, C_rho, m_rho
        )
        if out is None:
            continue
        chi, eps_r = out

        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": g["name"],
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": g["Mbar"],
        })
        eps_meds.append(np.median(eps_r))

    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
        v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)
    med_eps = np.median(eps_meds) if len(eps_meds) else np.nan

    results.append((eps0, C_rho, m_rho, med_rmse, med_sin, med_high, med_eps))

    print(f"(eps0={eps0:.1e}, C_rho={C_rho:.2f}, m={m_rho:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}, med_eps={med_eps:.2e}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[5], z[3]))[0]
print("eps0 =", best[0])
print("C_rho =", best[1])
print("m_rho =", best[2])
print("median RMSE =", best[3])
print("median signed inner =", best[4])
print("median high-mass RMSE =", best[5])
print("median eps(r) =", best[6])

Galaxies solved: 60

DENSITY-DEPENDENT STIFFNESS FLOOR SWEEP
(eps0=1.0e-04, C_rho=0.00, m=1.00) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_eps=1.00e-04
(eps0=1.0e-04, C_rho=0.50, m=0.50) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_eps=1.43e-04
(eps0=1.0e-04, C_rho=0.50, m=1.00) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_eps=1.36e-04
(eps0=1.0e-04, C_rho=0.50, m=2.00) -> medRMSE=25.461, s_in=0.025, high-mass=50.269, med_eps=1.26e-04
(eps0=1.0e-04, C_rho=1.00, m=0.50) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_eps=1.85e-04
(eps0=1.0e-04, C_rho=1.00, m=1.00) -> medRMSE=26.238, s_in=0.005, high-mass=50.272, med_eps=1.72e-04
(eps0=1.0e-04, C_rho=1.00, m=2.00) -> medRMSE=25.460, s_in=0.025, high-mass=50.260, med_eps=1.52e-04
(eps0=1.0e-04, C_rho=2.00, m=0.50) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_eps=2.70e-04
(eps0=1.0e-04, C_rho=2.00, m=1.00) -> medRMSE=25.461, s_in=0.024, high-mass=50.292, med_eps=2.45e-04
(eps0=1.0e-04, C_rho=2.00, m=2

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FOUNDATION DIAGNOSTIC:
# FIND WHICH STRUCTURAL VARIABLE TRACKS THE TRANSITION
#
# This does NOT add new model patches.
# It measures, for each galaxy:
#   - RMSE under the current frozen best backbone
#   - mass
#   - bulge fraction
#   - compactness
#   - mean surface density inside Rs
#   - mean volume density inside Rs
#   - g_b at 0.5 Rs
#   - M(<Rs)/Rs^2
#   - M(<Rs)/Rs^3
#   - dg_b/dr at 0.5 Rs
#
# Then it:
#   1. sorts by RMSE
#   2. prints decile summaries
#   3. prints correlations with RMSE
#   4. scans for the sharpest transition split variable
#
# Frozen model:
#   mixed operator with constant eps floor
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen model settings
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def interp_at(x, y, x0):
    x = np.asarray(x)
    y = np.asarray(y)
    if x0 <= x[0]:
        return y[0]
    if x0 >= x[-1]:
        return y[-1]
    return np.interp(x0, x, y)

def pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 3:
        return np.nan
    x = x - np.mean(x)
    y = y - np.mean(y)
    sx = np.sqrt(np.sum(x*x))
    sy = np.sqrt(np.sum(y*y))
    if sx <= 0 or sy <= 0:
        return np.nan
    return np.sum(x*y) / (sx*sy)

# ------------------------------------------------------------
# Load and solve frozen model
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

        chi = solve_mixed_operator(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2, EPS_LIN)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS
        bulge_frac = vb / denom

        # amplitude fit ingredient
        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Mbar": Mtot,
            "M_enc": M_enc,
            "rho": rho,
            "g_b": g_b,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "bulge_frac": bulge_frac,
        })
    except Exception:
        continue

# fit global amplitude
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = np.max(row["v_obs"])**2
    num += x * y
    den += x * x
A_global = num / den

# evaluate frozen model and derive diagnostic variables
for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    v_bul_s = row["v_bul_s"]
    M_enc = row["M_enc"]
    rho = row["rho"]
    g_b = row["g_b"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat2 = A_global * (Uinf / Rs)
    v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
    v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
    v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

    resid = v_pred - v_obs
    frac_signed = resid / np.maximum(v_obs, 1.0)
    x = r / Rs

    rmse = np.sqrt(np.mean(resid**2))
    s_in = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan

    M_Rs = interp_at(r, M_enc, Rs)
    rho_mean_Rs = M_Rs / ((4.0/3.0) * np.pi * Rs**3 + EPS)
    Sigma_mean_Rs = M_Rs / (np.pi * Rs**2 + EPS)

    g_half = interp_at(r, g_b, 0.5 * Rs)
    dgdr = np.gradient(g_b, r)
    dgdr_half = interp_at(r, dgdr, 0.5 * Rs)

    compactness = interp_at(r, M_enc, 0.3 * np.max(r)) / max(row["Mbar"], EPS)

    row["rmse"] = rmse
    row["s_in"] = s_in
    row["logM"] = np.log10(row["Mbar"])
    row["compact"] = compactness
    row["Sigma_mean_Rs"] = Sigma_mean_Rs
    row["rho_mean_Rs"] = rho_mean_Rs
    row["logSigma_mean_Rs"] = np.log10(max(Sigma_mean_Rs, EPS))
    row["logrho_mean_Rs"] = np.log10(max(rho_mean_Rs, EPS))
    row["g_half"] = g_half
    row["logg_half"] = np.log10(max(g_half, EPS))
    row["M_over_R2"] = M_Rs / (Rs**2 + EPS)
    row["M_over_R3"] = M_Rs / (Rs**3 + EPS)
    row["logM_over_R2"] = np.log10(max(row["M_over_R2"], EPS))
    row["logM_over_R3"] = np.log10(max(row["M_over_R3"], EPS))
    row["dgdr_half"] = dgdr_half
    row["log_abs_dgdr_half"] = np.log10(max(abs(dgdr_half), EPS))

rows = sorted(rows, key=lambda z: z["rmse"])

print("FOUNDATION DIAGNOSTIC: TRANSITION VARIABLE SEARCH")
print("="*130)
print("Frozen model:")
print(f"  p={P_OP}, kappa={KAPPA}, eta={ETA_SRC}, q={Q_RESP}, eps_lin={EPS_LIN}, lambda_b={LAMBDA_B}, r_b={R_B}")
print("Galaxies:", len(rows))
print("A_global:", A_global)
print("Median RMSE:", med([r["rmse"] for r in rows]))
print()

# ------------------------------------------------------------
# Deciles
# ------------------------------------------------------------
edges = np.linspace(0, len(rows), 11, dtype=int)

print("DECILE SUMMARY (best -> worst)")
print("="*130)
print(f'{"dec":<4} {"n":>3} {"RMSE":>9} {"logM":>8} {"bulge":>8} {"compact":>9} {"logSig":>9} {"logRho":>9} {"logg@0.5Rs":>11} {"logM/R^2":>10} {"logM/R^3":>10} {"log|dg/dr|":>11}')
for i in range(10):
    chunk = rows[edges[i]:edges[i+1]]
    print(f'{i+1:<4} {len(chunk):3d} {med([c["rmse"] for c in chunk]):9.3f} {med([c["logM"] for c in chunk]):8.3f} {med([c["bulge_frac"] for c in chunk]):8.3f} {med([c["compact"] for c in chunk]):9.3f} {med([c["logSigma_mean_Rs"] for c in chunk]):9.3f} {med([c["logrho_mean_Rs"] for c in chunk]):9.3f} {med([c["logg_half"] for c in chunk]):11.3f} {med([c["logM_over_R2"] for c in chunk]):10.3f} {med([c["logM_over_R3"] for c in chunk]):10.3f} {med([c["log_abs_dgdr_half"] for c in chunk]):11.3f}')
print()

# ------------------------------------------------------------
# Correlations with RMSE
# ------------------------------------------------------------
vars_to_test = [
    ("logM", "log10(Mbar)"),
    ("bulge_frac", "bulge fraction"),
    ("compact", "compactness"),
    ("logSigma_mean_Rs", "log10(mean surface density inside Rs)"),
    ("logrho_mean_Rs", "log10(mean volume density inside Rs)"),
    ("logg_half", "log10(g_b at 0.5 Rs)"),
    ("logM_over_R2", "log10(M(<Rs)/Rs^2)"),
    ("logM_over_R3", "log10(M(<Rs)/Rs^3)"),
    ("log_abs_dgdr_half", "log10(|dg_b/dr| at 0.5 Rs)"),
]

rmse_all = np.array([r["rmse"] for r in rows])

print("CORRELATION WITH RMSE")
print("="*130)
corr_rows = []
for key, label in vars_to_test:
    x = np.array([r[key] for r in rows], dtype=float)
    c = pearson(x, rmse_all)
    corr_rows.append((abs(c), c, key, label))
corr_rows.sort(reverse=True)
for _, c, key, label in corr_rows:
    print(f"{label:<38} r = {c:+.4f}")
print()

# ------------------------------------------------------------
# Sharp transition scan
# For each variable, sort by that variable and find split maximizing
# difference in median RMSE between lower and upper groups.
# ------------------------------------------------------------
print("SHARPEST RMSE SPLIT BY VARIABLE")
print("="*130)
best_splits = []

for key, label in vars_to_test:
    arr = np.array([r[key] for r in rows], dtype=float)
    ord_idx = np.argsort(arr)
    arr_s = arr[ord_idx]
    rmse_s = rmse_all[ord_idx]

    best_gap = -np.inf
    best_thr = np.nan
    best_lo = np.nan
    best_hi = np.nan
    best_i = None

    for i in range(8, len(arr_s) - 8):
        lo = rmse_s[:i]
        hi = rmse_s[i:]
        gap = np.median(hi) - np.median(lo)
        if gap > best_gap:
            best_gap = gap
            best_thr = 0.5 * (arr_s[i-1] + arr_s[i])
            best_lo = np.median(lo)
            best_hi = np.median(hi)
            best_i = i

    best_splits.append((best_gap, key, label, best_thr, best_lo, best_hi))

best_splits.sort(reverse=True)
for gap, key, label, thr, lo, hi in best_splits:
    print(f"{label:<38} thr={thr:10.4f}   medRMSE_lo={lo:8.3f}   medRMSE_hi={hi:8.3f}   gap={gap:8.3f}")
print()

# ------------------------------------------------------------
# Transition cohort: deciles 5-7
# Compare to good (1-4) and bad (8-10)
# ------------------------------------------------------------
good = rows[:edges[4]]
trans = rows[edges[4]:edges[7]]
bad = rows[edges[7]:]

print("GOOD / TRANSITION / BAD SUMMARY")
print("="*130)
print(f'{"group":<12} {"n":>3} {"RMSE":>9} {"logM":>8} {"bulge":>8} {"compact":>9} {"logSig":>9} {"logRho":>9} {"logg@0.5Rs":>11} {"logM/R^2":>10} {"logM/R^3":>10} {"log|dg/dr|":>11}')
for name, chunk in [("GOOD", good), ("TRANS", trans), ("BAD", bad)]:
    print(f'{name:<12} {len(chunk):3d} {med([c["rmse"] for c in chunk]):9.3f} {med([c["logM"] for c in chunk]):8.3f} {med([c["bulge_frac"] for c in chunk]):8.3f} {med([c["compact"] for c in chunk]):9.3f} {med([c["logSigma_mean_Rs"] for c in chunk]):9.3f} {med([c["logrho_mean_Rs"] for c in chunk]):9.3f} {med([c["logg_half"] for c in chunk]):11.3f} {med([c["logM_over_R2"] for c in chunk]):10.3f} {med([c["logM_over_R3"] for c in chunk]):10.3f} {med([c["log_abs_dgdr_half"] for c in chunk]):11.3f}')
print()

print("TOP 12 WORST GALAXIES WITH FOUNDATION VARIABLES")
print("="*130)
print(f'{"name":<22} {"RMSE":>8} {"logM":>8} {"bulge":>8} {"compact":>9} {"logSig":>9} {"logRho":>9} {"logg@0.5Rs":>11} {"logM/R^2":>10} {"logM/R^3":>10}')
for r in rows[-12:]:
    print(f'{r["name"]:<22} {r["rmse"]:8.2f} {r["logM"]:8.3f} {r["bulge_frac"]:8.3f} {r["compact"]:9.3f} {r["logSigma_mean_Rs"]:9.3f} {r["logrho_mean_Rs"]:9.3f} {r["logg_half"]:11.3f} {r["logM_over_R2"]:10.3f} {r["logM_over_R3"]:10.3f}')

FOUNDATION DIAGNOSTIC: TRANSITION VARIABLE SEARCH
Frozen model:
  p=3.5, kappa=0.1, eta=1.15, q=0.4, eps_lin=0.0001, lambda_b=0.675, r_b=3.25
Galaxies: 60
A_global: 1.2344936006554394
Median RMSE: 25.461813748480438

DECILE SUMMARY (best -> worst)
dec    n      RMSE     logM    bulge   compact    logSig    logRho  logg@0.5Rs   logM/R^2   logM/R^3  log|dg/dr|
1      6     8.919    8.826    0.568     0.465     7.145     6.548       2.502      7.643      7.170       2.263
2      6    12.087    9.254    0.485     0.443     7.279     6.587       2.555      7.776      7.209       1.594
3      6    13.728    9.565    0.496     0.360     7.093     6.289       2.404      7.590      6.911       1.724
4      6    16.391    9.375    0.606     0.214     7.219     6.445       2.585      7.717      7.067       1.742
5      6    21.631    9.987    0.716     0.599     7.471     6.729       2.991      7.968      7.351       2.495
6      6    31.672   10.440    0.833     0.904     7.980     6.800       3

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FOUNDATION TEST:
# SURFACE-DENSITY-TRIGGERED CONSTITUTIVE SWITCH IN p
#
# Idea:
#   Let the transport exponent switch with the coarse structural
#   variable
#
#       Sigma_* = M(<Rs_guess) / Rs_guess^2
#
#   via
#
#       p_eff(Sigma_*) = p_lo + (p_hi - p_lo) * S
#       S = 1 / (1 + exp(-(logSigma - logSigma_c)/Delta))
#
#   Then solve with galaxy-wise p_eff held fixed per galaxy.
#
# This is a regime-switch test in the backbone itself,
# not another output patch.
#
# Frozen:
#   mixed operator with constant eps floor
#   eta = 1.15
#   q = 0.40
#   kappa = 0.10
#   lambda_b = 0.675
#   r_b = 3.25
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen model settings
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# ------------------------------------------------------------
# Load galaxies once and build coarse structural variable
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        # structural scale guess from half-baryonic-mass radius
        frac = M_enc / max(Mtot, EPS)
        ihalf = int(np.searchsorted(frac, 0.5))
        ihalf = min(ihalf, len(r) - 1)
        Rs_guess = r[ihalf] if np.isfinite(r[ihalf]) and r[ihalf] > 0 else np.median(r)

        M_Rsg = np.interp(Rs_guess, r, M_enc)
        Sigma_star = M_Rsg / (Rs_guess**2 + EPS)
        logSigma_star = np.log10(max(Sigma_star, EPS))

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
            "Rs_guess": Rs_guess,
            "Sigma_star": Sigma_star,
            "logSigma_star": logSigma_star,
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
print()

# threshold from your diagnostic ~ 8.1185
logSigma_c_grid = [7.95, 8.10, 8.25]
Delta_grid = [0.08, 0.15, 0.25]
p_lo_hi_grid = [
    (3.5, 3.5),   # control
    (3.5, 3.7),
    (3.5, 3.9),
    (3.5, 4.1),
    (3.7, 4.1),
]

print("SURFACE-DENSITY-TRIGGERED p SWITCH")
print("="*126)
print("p_eff = p_lo + (p_hi - p_lo) * sigmoid((logSigma* - logSigma_c)/Delta)")
print()

results = []

for p_lo, p_hi in p_lo_hi_grid:
    for logSigma_c in logSigma_c_grid:
        for Delta in Delta_grid:

            rows = []
            p_vals = []

            for g in galaxies:
                r = g["r"]
                v_obs = g["v_obs"]
                v_bul_s = g["v_bul_s"]
                v_bar2 = g["v_bar2"]
                rho = g["rho"]
                g_b = g["g_b"]

                S = sigmoid((g["logSigma_star"] - logSigma_c) / Delta)
                p_eff = p_lo + (p_hi - p_lo) * S
                p_vals.append(p_eff)

                boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                          * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
                boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
                rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

                chi = solve_mixed_operator(r, rho_eff, v_bar2, PHI_S, N_SCR, p_eff, MU2, EPS_LIN)
                if chi is None or not np.all(np.isfinite(chi)):
                    continue

                chi = chi - np.min(chi)
                chi[chi < 0] = 0.0

                Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
                Uinf = Uchi[-1]
                if not np.isfinite(Uinf) or Uinf <= 0:
                    continue

                target = (1.0 - np.exp(-1.0)) * Uinf
                idx = np.searchsorted(Uchi, target)
                idx = min(idx, len(r) - 1)
                Rs = r[idx]
                if not np.isfinite(Rs) or Rs <= 0:
                    continue

                rows.append({
                    "name": g["name"],
                    "r": r,
                    "v_obs": v_obs,
                    "v_bul_s": v_bul_s,
                    "Uchi": Uchi,
                    "Uinf": Uinf,
                    "Rs": Rs,
                    "Mbar": g["Mbar"],
                    "logSigma_star": g["logSigma_star"],
                    "p_eff": p_eff,
                })

            num = 0.0
            den = 0.0
            for row in rows:
                x = row["Uinf"] / row["Rs"]
                y = np.max(row["v_obs"])**2
                num += x * y
                den += x * x
            A_global = num / den if den > 0 else np.nan

            rmse_list = []
            s_in_list = []
            high_mass = []

            for row in rows:
                r = row["r"]
                v_obs = row["v_obs"]
                v_bul_s = row["v_bul_s"]
                Uchi = row["Uchi"]
                Uinf = row["Uinf"]
                Rs = row["Rs"]

                Vflat2 = A_global * (Uinf / Rs)
                v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
                v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
                v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

                resid = v_pred - v_obs
                frac_signed = resid / np.maximum(v_obs, 1.0)
                x = r / Rs

                rmse = np.sqrt(np.mean(resid**2))
                rmse_list.append(rmse)

                if np.any(x < 0.7):
                    s_in_list.append(np.median(frac_signed[x < 0.7]))

                if row["Mbar"] >= 1e10:
                    high_mass.append(rmse)

            med_rmse = np.median(rmse_list)
            med_sin = np.median(s_in_list)
            med_high = np.median(high_mass)
            med_p = np.median(p_vals)
            p_span = (np.min(p_vals), np.max(p_vals))

            results.append((p_lo, p_hi, logSigma_c, Delta, med_rmse, med_sin, med_high, med_p, p_span))

            print(
                f"(p_lo={p_lo:.2f}, p_hi={p_hi:.2f}, logSc={logSigma_c:.2f}, D={Delta:.2f}) "
                f"-> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}, "
                f"med_p={med_p:.3f}, p_span=({p_span[0]:.3f},{p_span[1]:.3f})"
            )

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[6], z[4]))[0]
print("p_lo =", best[0])
print("p_hi =", best[1])
print("logSigma_c =", best[2])
print("Delta =", best[3])
print("median RMSE =", best[4])
print("median signed inner =", best[5])
print("median high-mass RMSE =", best[6])
print("median p_eff =", best[7])
print("p_eff span =", best[8])

Galaxies solved: 60

SURFACE-DENSITY-TRIGGERED p SWITCH
p_eff = p_lo + (p_hi - p_lo) * sigmoid((logSigma* - logSigma_c)/Delta)

(p_lo=3.50, p_hi=3.50, logSc=7.95, D=0.08) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=7.95, D=0.15) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=7.95, D=0.25) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=8.10, D=0.08) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=8.10, D=0.15) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=8.10, D=0.25) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.500,3.500)
(p_lo=3.50, p_hi=3.50, logSc=8.25, D=0.08) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_p=3.500, p_span=(3.5

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FOUNDATION TEST:
# REPLACE SOURCE VARIABLE g_b WITH ENCLOSED-LOAD VARIABLE
#
# Test source-loading variable families:
#
#   Y_mass(r)  = M(<r) / r^2
#   Y_mix(r)   = g_b(r)^a * [M(<r)/r^2]^(1-a)
#
# Then use the SAME boost law shape, but evaluated on Y instead of g_b:
#
#   boost(Y) = A_g * (Y/g_ref)^beta_lo * (1 + Y/g_t)^-(beta_lo-beta_hi)
#
# Frozen backbone:
#   mixed operator with constant eps floor
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
#
# Goal:
#   See whether the missing foundation variable is really an enclosed-load /
#   surface-density-like invariant rather than pure local g_b.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies once
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)
        y_mass = M_enc / (r**2 + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "y_mass": y_mass,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
print()

def boost_from_Y(Y, gref_scale=1.0):
    # keep the same functional shape, only change the source variable
    Yscaled = Y / max(gref_scale, EPS)
    return A_G * np.power(np.clip(Yscaled / GREF, 0.0, None), BETA_LO) * \
           np.power(1.0 + np.clip(Yscaled / GT_FIT, 0.0, None), -DELTA_BETA)

# ------------------------------------------------------------
# Sweep source-variable families
# ------------------------------------------------------------
# controls:
#   GB      : original g_b
#   YMASS   : M(<r)/r^2
#   MIX_a   : g_b^a * y_mass^(1-a)
#
# Because the raw scales differ, each galaxy gets a scale-normalization
# so that the median of Y roughly matches median g_b in that galaxy.
# That keeps the test about SHAPE, not arbitrary unit mismatch.
# ------------------------------------------------------------

families = [
    ("GB", None),
    ("YMASS", None),
    ("MIX", 0.75),
    ("MIX", 0.50),
    ("MIX", 0.25),
]

print("SOURCE-VARIABLE REPLACEMENT TEST")
print("="*122)
print("Variables:")
print("  GB     = g_b")
print("  YMASS  = M(<r)/r^2")
print("  MIX_a  = g_b^a * (M(<r)/r^2)^(1-a)")
print()

results = []

for fam, a in families:
    rows = []

    for g in galaxies:
        r = g["r"]
        v_obs = g["v_obs"]
        v_bul_s = g["v_bul_s"]
        v_bar2 = g["v_bar2"]
        rho = g["rho"]
        g_b = g["g_b"]
        y_mass = g["y_mass"]

        if fam == "GB":
            Y_raw = g_b.copy()
        elif fam == "YMASS":
            Y_raw = y_mass.copy()
        else:
            # mixed geometric mean variable
            Y_raw = np.power(np.clip(g_b, EPS, None), a) * np.power(np.clip(y_mass, EPS, None), 1.0 - a)

        # normalize Y to the local g_b scale for a fair shape test
        med_gb = np.median(np.clip(g_b, EPS, None))
        med_Y  = np.median(np.clip(Y_raw, EPS, None))
        scale  = med_Y / max(med_gb, EPS)
        Y = Y_raw / max(scale, EPS)

        boost = boost_from_Y(Y, gref_scale=1.0)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

        chi = solve_mixed_operator(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2, EPS_LIN)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": g["name"],
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": g["Mbar"],
        })

    # fit amplitude for this source family
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A_global * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
        v_bulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    label = fam if fam != "MIX" else f"MIX_a={a:.2f}"
    results.append((label, med_rmse, med_sin, med_high))

    print(f"({label}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[2]**2 + 0.0, z[3], z[1]))[0]
print("label =", best[0])
print("median RMSE =", best[1])
print("median signed inner =", best[2])
print("median high-mass RMSE =", best[3])

Galaxies solved: 60

SOURCE-VARIABLE REPLACEMENT TEST
Variables:
  GB     = g_b
  YMASS  = M(<r)/r^2
  MIX_a  = g_b^a * (M(<r)/r^2)^(1-a)

(GB) -> medRMSE=25.462, s_in=0.025, high-mass=50.269
(YMASS) -> medRMSE=25.462, s_in=0.025, high-mass=50.269
(MIX_a=0.75) -> medRMSE=25.462, s_in=0.025, high-mass=50.269
(MIX_a=0.50) -> medRMSE=25.462, s_in=0.025, high-mass=50.269
(MIX_a=0.25) -> medRMSE=25.462, s_in=0.025, high-mass=50.269

BEST ROW
label = MIX_a=0.50
median RMSE = 25.461809661228557
median signed inner = 0.024684631988208562
median high-mass RMSE = 50.26921233484374


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# MINIMAL FOUNDATION BRANCH:
# TWO-COMPONENT FIELD OBSERVABLE
#
# Test the smallest real departure from the exhausted one-field-readout family:
#
#   V_pred^2(r) =
#       A * (Uchi/Uinf)^q
#     + B_loc * chi(r)^nu
#     + lambda_b * Vbul^2 / (1 + r/r_b)
#
# This does NOT change the solved field equation class.
# It changes the OBSERVABLE class:
#   - cumulative term  -> broad structure
#   - local chi term   -> compact inner support
#
# Frozen field backbone:
#   mixed operator with constant eps floor
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
#
# Sweep only:
#   B_loc, nu
#
# Goal:
#   see whether adding a local field readout fixes the stubborn inner failures
#   without wrecking the global fit.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen field backbone
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies and solve frozen field backbone once
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                  * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
        boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

        chi = solve_mixed_operator(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2, EPS_LIN)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        # normalize chi by its own galaxy max so local term is shape-based,
        # not dominated by raw absolute scale differences.
        chi_max = np.max(chi)
        chi_norm = chi / max(chi_max, EPS)

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "chi": chi,
            "chi_norm": chi_norm,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": Mtot,
        })
    except Exception:
        continue

print("Galaxies solved:", len(rows))
print()

# ------------------------------------------------------------
# Grid on local observable term
#
# V_pred^2 = A*(U/Uinf)^q + B_loc * chi_norm^nu + bulge_term
#
# A is re-fit globally for each row of (B_loc, nu)
# B_loc is in velocity^2 units, so test modest values first.
# ------------------------------------------------------------
grid = [
    (0.0,  1.0),   # control
    (200.0, 0.5),
    (200.0, 1.0),
    (200.0, 2.0),
    (500.0, 0.5),
    (500.0, 1.0),
    (500.0, 2.0),
    (1000.0, 0.5),
    (1000.0, 1.0),
    (1000.0, 2.0),
    (2000.0, 0.5),
    (2000.0, 1.0),
    (2000.0, 2.0),
]

print("TWO-COMPONENT FIELD OBSERVABLE TEST")
print("="*122)
print("V_pred^2 = A*(U/Uinf)^q + B_loc*chi_norm^nu + lambda_b*Vbul^2/(1+r/r_b)")
print()

results = []

for B_loc, nu in grid:
    # fit A globally for this observable family
    num = 0.0
    den = 0.0
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        chi_norm = row["chi_norm"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        # use Vmax point to fit asymptotic amplitude term after subtracting local+bulge parts
        i_vmax = int(np.argmax(v_obs))
        x_u = np.clip(row["Uchi"][i_vmax] / max(Uinf, EPS), 0.0, None)**Q_RESP
        local_term = B_loc * (chi_norm[i_vmax] ** nu)
        bulge_term = LAMBDA_B * (row["v_bul_s"][i_vmax]**2) * f_lor(r[i_vmax] / R_B)

        y = max(v_obs[i_vmax]**2 - local_term - bulge_term, 0.0)
        x = (Uinf / max(Rs, EPS)) * x_u

        num += x * y
        den += x * x

    A_global = num / den if den > 0 else np.nan

    rmse_list = []
    s_in_list = []
    high_mass = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        chi_norm = row["chi_norm"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vfield2_cum = A_global * (Uinf / Rs) * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
        Vfield2_loc = B_loc * np.power(np.clip(chi_norm, 0.0, None), nu)
        Vbulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)

        v_pred = np.sqrt(np.maximum(Vfield2_cum + Vfield2_loc + Vbulge2, 0.0))

        resid = v_pred - v_obs
        frac_signed = resid / np.maximum(v_obs, 1.0)
        x = r / Rs

        rmse = np.sqrt(np.mean(resid**2))
        rmse_list.append(rmse)

        if np.any(x < 0.7):
            s_in_list.append(np.median(frac_signed[x < 0.7]))

        if row["Mbar"] >= 1e10:
            high_mass.append(rmse)

    med_rmse = np.median(rmse_list)
    med_sin = np.median(s_in_list)
    med_high = np.median(high_mass)

    results.append((B_loc, nu, med_rmse, med_sin, med_high, A_global))

    print(f"(B_loc={B_loc:7.1f}, nu={nu:.2f}) -> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}, A={A_global:.3f}")

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[4], z[2], abs(z[3])))[0]
print("B_loc =", best[0])
print("nu =", best[1])
print("median RMSE =", best[2])
print("median signed inner =", best[3])
print("median high-mass RMSE =", best[4])
print("A_global =", best[5])

Galaxies solved: 60

TWO-COMPONENT FIELD OBSERVABLE TEST
V_pred^2 = A*(U/Uinf)^q + B_loc*chi_norm^nu + lambda_b*Vbul^2/(1+r/r_b)

(B_loc=    0.0, nu=1.00) -> medRMSE=28.656, s_in=0.125, high-mass=52.559, A=1.529
(B_loc=  200.0, nu=0.50) -> medRMSE=29.062, s_in=0.131, high-mass=52.775, A=1.525
(B_loc=  200.0, nu=1.00) -> medRMSE=28.997, s_in=0.131, high-mass=52.760, A=1.526
(B_loc=  200.0, nu=2.00) -> medRMSE=28.930, s_in=0.131, high-mass=52.746, A=1.527
(B_loc=  500.0, nu=0.50) -> medRMSE=30.024, s_in=0.145, high-mass=51.990, A=1.520
(B_loc=  500.0, nu=1.00) -> medRMSE=29.623, s_in=0.143, high-mass=51.985, A=1.523
(B_loc=  500.0, nu=2.00) -> medRMSE=29.396, s_in=0.139, high-mass=51.997, A=1.525
(B_loc= 1000.0, nu=0.50) -> medRMSE=30.668, s_in=0.163, high-mass=50.795, A=1.512
(B_loc= 1000.0, nu=1.00) -> medRMSE=30.125, s_in=0.161, high-mass=50.784, A=1.517
(B_loc= 1000.0, nu=2.00) -> medRMSE=29.656, s_in=0.155, high-mass=50.809, A=1.521
(B_loc= 2000.0, nu=0.50) -> medRMSE=32.322, s_in=0

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FOUNDATION TEST:
# COMPACTNESS-DEPENDENT PROPAGATION / SCREENING SCALE
#
# Keep the same backbone, but make the screening scale depend on
# the galaxy-level surface-density-like structural variable:
#
#   Sigma_* = M(<Rs_guess) / Rs_guess^2
#
# and test
#
#   phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma)
#
# so that:
#   gamma > 0  -> high-Sigma systems have smaller phi_s_eff
#               -> stronger screening / shorter propagation scale
#
# Frozen:
#   mixed operator with constant eps floor
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
#
# Goal:
#   test whether the real missing foundation variable is
#   compactness-dependent propagation length.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s_eff, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / max(phi_s_eff, EPS), 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies once and compute Sigma_* proxy
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        # same proxy used before: half-baryonic-mass radius
        frac = M_enc / max(Mtot, EPS)
        ihalf = int(np.searchsorted(frac, 0.5))
        ihalf = min(ihalf, len(r) - 1)
        Rs_guess = r[ihalf] if np.isfinite(r[ihalf]) and r[ihalf] > 0 else np.median(r)
        M_Rsg = np.interp(Rs_guess, r, M_enc)
        Sigma_star = M_Rsg / (Rs_guess**2 + EPS)

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
            "Rs_guess": Rs_guess,
            "Sigma_star": Sigma_star,
            "logSigma_star": np.log10(max(Sigma_star, EPS)),
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
Sigma0 = np.median([g["Sigma_star"] for g in galaxies])
print("Sigma0 (median Sigma_*) =", Sigma0)
print("log10 Sigma0 =", np.log10(Sigma0))
print()

# ------------------------------------------------------------
# Sweep gamma in phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma)
# gamma > 0 => high-Sigma galaxies get smaller phi_s_eff
# ------------------------------------------------------------
gamma_grid = [-0.50, -0.25, 0.00, 0.10, 0.20, 0.30, 0.40, 0.60]
# optional clipping to prevent absurd values
clip_grid = [(0.5, 2.0), (0.33, 3.0)]

print("COMPACTNESS-DEPENDENT PROPAGATION SCALE TEST")
print("="*126)
print("phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma), then clipped to [clip_lo*phi_s, clip_hi*phi_s]")
print()

results = []

for clip_lo, clip_hi in clip_grid:
    for gamma in gamma_grid:
        rows = []
        phi_list = []

        for g in galaxies:
            r = g["r"]
            v_obs = g["v_obs"]
            v_bul_s = g["v_bul_s"]
            v_bar2 = g["v_bar2"]
            rho = g["rho"]
            g_b = g["g_b"]

            scale = (g["Sigma_star"] / max(Sigma0, EPS)) ** (-gamma)
            phi_s_eff = PHI_S * scale
            phi_s_eff = np.clip(phi_s_eff, PHI_S * clip_lo, PHI_S * clip_hi)
            phi_list.append(phi_s_eff)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

            chi = solve_mixed_operator(r, rho_eff, v_bar2, phi_s_eff, N_SCR, P_OP, MU2, EPS_LIN)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": g["name"],
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "Mbar": g["Mbar"],
            })

        # fit amplitude for this propagation law
        num = 0.0
        den = 0.0
        for row in rows:
            x = row["Uinf"] / row["Rs"]
            y = np.max(row["v_obs"])**2
            num += x * y
            den += x * x
        A_global = num / den if den > 0 else np.nan

        rmse_list = []
        s_in_list = []
        high_mass = []

        for row in rows:
            r = row["r"]
            v_obs = row["v_obs"]
            v_bul_s = row["v_bul_s"]
            Uchi = row["Uchi"]
            Uinf = row["Uinf"]
            Rs = row["Rs"]

            Vfield2 = A_global * (Uinf / Rs) * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
            Vbulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
            v_pred = np.sqrt(np.maximum(Vfield2 + Vbulge2, 0.0))

            resid = v_pred - v_obs
            frac_signed = resid / np.maximum(v_obs, 1.0)
            x = r / Rs

            rmse = np.sqrt(np.mean(resid**2))
            rmse_list.append(rmse)

            if np.any(x < 0.7):
                s_in_list.append(np.median(frac_signed[x < 0.7]))

            if row["Mbar"] >= 1e10:
                high_mass.append(rmse)

        med_rmse = np.median(rmse_list)
        med_sin = np.median(s_in_list)
        med_high = np.median(high_mass)
        med_phi = np.median(phi_list)

        results.append((clip_lo, clip_hi, gamma, med_rmse, med_sin, med_high, med_phi, A_global))

        print(
            f"(clip=[{clip_lo:.2f},{clip_hi:.2f}], gamma={gamma:+.2f}) "
            f"-> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}, "
            f"med_phi={med_phi:.1f}, A={A_global:.3f}"
        )

print("\nBEST ROW")
# prioritize high-mass, then global RMSE, then small absolute inner bias
best = sorted(results, key=lambda z: (z[5], z[3], abs(z[4])))[0]
print("clip_lo =", best[0])
print("clip_hi =", best[1])
print("gamma =", best[2])
print("median RMSE =", best[3])
print("median signed inner =", best[4])
print("median high-mass RMSE =", best[5])
print("median phi_s_eff =", best[6])
print("A_global =", best[7])

Galaxies solved: 60
Sigma0 (median Sigma_*) = 362913755.16406107
log10 Sigma0 = 8.559803429150236

COMPACTNESS-DEPENDENT PROPAGATION SCALE TEST
phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma), then clipped to [clip_lo*phi_s, clip_hi*phi_s]

(clip=[0.50,2.00], gamma=-0.50) -> medRMSE=24.764, s_in=0.031, high-mass=48.546, med_phi=39999.9, A=1.335
(clip=[0.50,2.00], gamma=-0.25) -> medRMSE=26.040, s_in=0.034, high-mass=49.133, med_phi=40000.0, A=1.280
(clip=[0.50,2.00], gamma=+0.00) -> medRMSE=25.462, s_in=0.025, high-mass=50.269, med_phi=40000.0, A=1.234
(clip=[0.50,2.00], gamma=+0.10) -> medRMSE=25.482, s_in=-0.000, high-mass=50.799, med_phi=40000.0, A=1.214
(clip=[0.50,2.00], gamma=+0.20) -> medRMSE=25.539, s_in=0.015, high-mass=51.417, med_phi=40000.1, A=1.192
(clip=[0.50,2.00], gamma=+0.30) -> medRMSE=25.651, s_in=0.009, high-mass=52.135, med_phi=40000.1, A=1.167
(clip=[0.50,2.00], gamma=+0.40) -> medRMSE=25.750, s_in=0.004, high-mass=52.679, med_phi=40000.1, A=1.146
(clip=[0.50,2.00],

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# REFINEMENT AROUND THE FIRST FOUNDATION BASIN THAT MOVED
#
# Refine the compactness-dependent propagation-scale branch:
#
#   phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma)
#
# Previous best direction:
#   gamma = -0.5  (equivalently phi_s_eff grows with Sigma_*)
#
# Frozen:
#   mixed operator with constant eps floor
#   p = 3.5
#   kappa = 0.10
#   eta = 1.15
#   q = 0.40
#   lambda_b = 0.675
#   r_b = 3.25
#
# Goal:
#   check whether gamma ~ -0.5 is a real basin or just noise
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP    = 3.5
MU2     = 1e-7
KAPPA   = 0.10
A_G     = 0.40
GREF    = 1000.0
ETA_SRC = 1.15
Q_RESP  = 0.40
PHI_S   = 40000.0
N_SCR   = 1.0
EPS_LIN = 1e-4

# Boost shape
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

# Frozen bulge channel
LAMBDA_B = 0.675
R_B = 3.25

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_mixed_operator(r, rho_eff, v_bar2, phi_s_eff, n_scr, p_op, mu2, eps_lin, n_iter=180, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / max(phi_s_eff, EPS), 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        nonlinear = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)
        sigma = (nonlinear + eps_lin) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

# ------------------------------------------------------------
# Load galaxies once and compute Sigma_* proxy
# ------------------------------------------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        if not np.isfinite(Mtot) or Mtot <= 0:
            continue

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        rho = np.clip(rho, 0.0, None)
        g_b = v_bar2 / (r + EPS)

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS

        frac = M_enc / max(Mtot, EPS)
        ihalf = int(np.searchsorted(frac, 0.5))
        ihalf = min(ihalf, len(r) - 1)
        Rs_guess = r[ihalf] if np.isfinite(r[ihalf]) and r[ihalf] > 0 else np.median(r)
        M_Rsg = np.interp(Rs_guess, r, M_enc)
        Sigma_star = M_Rsg / (Rs_guess**2 + EPS)

        galaxies.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bul_s": v_bul_s,
            "v_bar2": v_bar2,
            "rho": rho,
            "g_b": g_b,
            "Mbar": Mtot,
            "bulge_frac": vb / denom,
            "Vmax_obs": np.max(v_obs),
            "Rs_guess": Rs_guess,
            "Sigma_star": Sigma_star,
            "logSigma_star": np.log10(max(Sigma_star, EPS)),
        })
    except Exception:
        continue

print("Galaxies solved:", len(galaxies))
Sigma0 = np.median([g["Sigma_star"] for g in galaxies])
print("Sigma0 (median Sigma_*) =", Sigma0)
print("log10 Sigma0 =", np.log10(Sigma0))
print()

gamma_grid = [-0.70, -0.60, -0.55, -0.50, -0.45, -0.40, -0.30]
clip_grid = [
    (0.50, 2.00),
    (0.40, 2.50),
    (0.33, 3.00),
]

print("REFINEMENT: COMPACTNESS-DEPENDENT PROPAGATION SCALE")
print("="*126)
print("phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma), clipped to [clip_lo*phi_s, clip_hi*phi_s]")
print()

results = []

for clip_lo, clip_hi in clip_grid:
    for gamma in gamma_grid:
        rows = []
        phi_list = []

        for g in galaxies:
            r = g["r"]
            v_obs = g["v_obs"]
            v_bul_s = g["v_bul_s"]
            v_bar2 = g["v_bar2"]
            rho = g["rho"]
            g_b = g["g_b"]

            scale = (g["Sigma_star"] / max(Sigma0, EPS)) ** (-gamma)
            phi_s_eff = PHI_S * scale
            phi_s_eff = np.clip(phi_s_eff, PHI_S * clip_lo, PHI_S * clip_hi)
            phi_list.append(phi_s_eff)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * np.power(1.0 + boost, ETA_SRC)

            chi = solve_mixed_operator(r, rho_eff, v_bar2, phi_s_eff, N_SCR, P_OP, MU2, EPS_LIN)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": g["name"],
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "Mbar": g["Mbar"],
            })

        num = 0.0
        den = 0.0
        for row in rows:
            x = row["Uinf"] / row["Rs"]
            y = np.max(row["v_obs"])**2
            num += x * y
            den += x * x
        A_global = num / den if den > 0 else np.nan

        rmse_list = []
        s_in_list = []
        high_mass = []

        for row in rows:
            r = row["r"]
            v_obs = row["v_obs"]
            v_bul_s = row["v_bul_s"]
            Uchi = row["Uchi"]
            Uinf = row["Uinf"]
            Rs = row["Rs"]

            Vfield2 = A_global * (Uinf / Rs) * np.power(np.clip(Uchi / Uinf, 0.0, None), Q_RESP)
            Vbulge2 = LAMBDA_B * (v_bul_s**2) * f_lor(r / R_B)
            v_pred = np.sqrt(np.maximum(Vfield2 + Vbulge2, 0.0))

            resid = v_pred - v_obs
            frac_signed = resid / np.maximum(v_obs, 1.0)
            x = r / Rs

            rmse = np.sqrt(np.mean(resid**2))
            rmse_list.append(rmse)

            if np.any(x < 0.7):
                s_in_list.append(np.median(frac_signed[x < 0.7]))

            if row["Mbar"] >= 1e10:
                high_mass.append(rmse)

        med_rmse = np.median(rmse_list)
        med_sin = np.median(s_in_list)
        med_high = np.median(high_mass)
        med_phi = np.median(phi_list)
        phi_lo = np.min(phi_list)
        phi_hi = np.max(phi_list)

        results.append((clip_lo, clip_hi, gamma, med_rmse, med_sin, med_high, med_phi, phi_lo, phi_hi, A_global))

        print(
            f"(clip=[{clip_lo:.2f},{clip_hi:.2f}], gamma={gamma:+.2f}) "
            f"-> medRMSE={med_rmse:.3f}, s_in={med_sin:.3f}, high-mass={med_high:.3f}, "
            f"med_phi={med_phi:.1f}, phi_span=({phi_lo:.1f},{phi_hi:.1f}), A={A_global:.3f}"
        )

print("\nBEST ROW")
best = sorted(results, key=lambda z: (z[5], z[3], abs(z[4])))[0]
print("clip_lo =", best[0])
print("clip_hi =", best[1])
print("gamma =", best[2])
print("median RMSE =", best[3])
print("median signed inner =", best[4])
print("median high-mass RMSE =", best[5])
print("median phi_s_eff =", best[6])
print("phi_s_eff span =", (best[7], best[8]))
print("A_global =", best[9])

Galaxies solved: 60
Sigma0 (median Sigma_*) = 362913755.16406107
log10 Sigma0 = 8.559803429150236

REFINEMENT: COMPACTNESS-DEPENDENT PROPAGATION SCALE
phi_s_eff = phi_s * (Sigma_*/Sigma0)^(-gamma), clipped to [clip_lo*phi_s, clip_hi*phi_s]

(clip=[0.50,2.00], gamma=-0.70) -> medRMSE=24.876, s_in=0.032, high-mass=48.112, med_phi=40000.0, phi_span=(20000.0,80000.0), A=1.342
(clip=[0.50,2.00], gamma=-0.60) -> medRMSE=24.819, s_in=0.030, high-mass=48.214, med_phi=39999.9, phi_span=(20000.0,80000.0), A=1.338
(clip=[0.50,2.00], gamma=-0.55) -> medRMSE=24.782, s_in=0.030, high-mass=48.379, med_phi=39999.9, phi_span=(20000.0,80000.0), A=1.337
(clip=[0.50,2.00], gamma=-0.50) -> medRMSE=24.764, s_in=0.031, high-mass=48.546, med_phi=39999.9, phi_span=(20000.0,80000.0), A=1.335
(clip=[0.50,2.00], gamma=-0.45) -> medRMSE=24.782, s_in=0.031, high-mass=48.719, med_phi=39999.9, phi_span=(20000.0,80000.0), A=1.333
(clip=[0.50,2.00], gamma=-0.40) -> medRMSE=24.826, s_in=0.031, high-mass=48.761, med_phi=

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# METRIC-COMPRESSION ROTATION LAW TEST
# ------------------------------------------------------------
# V_pred^2 = V_bar^2 * E(r)
#
# E(r) = (1 + x)^(delta - 1) * [1 + (1 + delta*gamma) * x]
# x    = (r0 / r)^gamma
#
# This is the direct observable law implied by the compressed
# radial metric picture.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7

# quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

EPS = 1e-8

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def enhancement(r, r0, gamma, delta):
    x = np.power(np.clip(r0 / np.clip(r, EPS, None), 0.0, None), gamma)
    return np.power(1.0 + x, delta - 1.0) * (1.0 + (1.0 + delta * gamma) * x)

def vpred_from_params(r, vbar2, p):
    log_r0, gamma, delta = p
    r0 = np.exp(log_r0)
    E = enhancement(r, r0, gamma, delta)
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def residuals(p, r, vbar2, v_obs):
    v_pred = vpred_from_params(r, vbar2, p)
    # relative-style weighting so large galaxies don't dominate too hard
    w = 1.0 / np.maximum(v_obs, 20.0)
    return (v_pred - v_obs) * w

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    vbar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(v_gas[i_vmax])
    vd = abs(v_disk_s[i_vmax])
    vb = abs(v_bul_s[i_vmax])
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit each galaxy
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]

    # initial guess:
    # r0 ~ radius where baryons begin to underpredict
    r0_init = np.median(r)
    p0 = np.array([np.log(r0_init), 0.15, 0.60])  # log_r0, gamma, delta

    # bounds: broad but sensible
    lo = np.array([np.log(np.min(r) * 0.2), 0.01, 0.05])
    hi = np.array([np.log(np.max(r) * 20.0), 1.50, 2.50])

    try:
        res = least_squares(
            residuals, p0,
            bounds=(lo, hi),
            args=(r, vbar2, v_obs),
            max_nfev=3000,
            xtol=1e-10, ftol=1e-10, gtol=1e-10,
        )
    except Exception:
        continue

    if not res.success:
        continue

    log_r0, gamma, delta = res.x
    r0 = np.exp(log_r0)

    v_pred = vpred_from_params(r, vbar2, res.x)
    resid = v_pred - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae  = np.mean(np.abs(resid))

    frac_signed = resid / np.maximum(v_obs, 1.0)
    x = r / max(r0, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x < 2.0)]) if np.any((x >= 0.7) & (x < 2.0)) else np.nan
    s_out = np.median(frac_signed[x >= 2.0]) if np.any(x >= 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "r0": r0,
        "gamma": gamma,
        "delta": delta,
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

print("Galaxies fit:", len(rows))
print()

rows = sorted(rows, key=lambda z: z["rmse"])

# -----------------------------
# summaries
# -----------------------------
rmse_list = [r["rmse"] for r in rows]
mae_list  = [r["mae"]  for r in rows]
s_in_list = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
gamma_list = [r["gamma"] for r in rows]
delta_list = [r["delta"] for r in rows]
r0_list    = [r["r0"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("METRIC-COMPRESSION LAW TEST")
print("="*120)
print("GLOBAL SUMMARY")
print("="*120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median gamma:             ", med(gamma_list))
print("Median delta:             ", med(delta_list))
print("Median r0:                ", med(r0_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("="*120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"r0":>8} {"gamma":>8} {"delta":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["r0"]:8.3f} {row["gamma"]:8.3f} {row["delta"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("="*120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"r0":>8} {"gamma":>8} {"delta":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["r0"]:8.3f} {row["gamma"]:8.3f} {row["delta"]:8.3f}')

Galaxies loaded: 60

Galaxies fit: 60

METRIC-COMPRESSION LAW TEST
GLOBAL SUMMARY
Median RMSE:               27.17553462867702
Mean RMSE:                 32.37833826754329
Median MAE:                22.286793782431992
Median signed inner resid: -0.07528628036293782
Median signed mid resid:   -0.12915369258707912
Median signed outer resid: -0.0318581435974476
Median gamma:              0.010000000000000002
Median delta:              1.8834470785961646
Median r0:                 251.69999999901802
Median low-Vmax RMSE:      17.358563228303286
Median high-Vmax RMSE:     40.060756439351586

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out       r0    gamma    delta
UGCA444_rotmod.dat           3.17     2.46   -0.006      nan      nan   52.400    0.089    2.500
UGC04305_rotmod.dat          3.38     2.78    0.003      nan      nan  110.400    0.010    0.856
NGC2976_rotmod.dat           5.19     3.84   -0.031      nan      nan   45.400    0.010    0.810
U

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# METRIC-COMPRESSION LAW + COMPACTNESS SCALING
# ------------------------------------------------------------
# V_pred^2 = V_bar^2 * E(r)
#
# E(r) = (1 + x)^(delta - 1) * [1 + (1 + delta*gamma) * x]
# x    = (r0 / r)^gamma * (Sigma_* / Sigma0)^eta
#
# Same law as before, with one structural correction only:
# compact galaxies get stronger geometric compression.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7

# quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

EPS = 1e-8
G = 4.30091e-6

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def enhancement(r, r0, gamma, delta, sigma_star, sigma0, eta):
    x = np.power(np.clip(r0 / np.clip(r, EPS, None), 0.0, None), gamma)
    x *= np.power(np.clip(sigma_star / max(sigma0, EPS), EPS, None), eta)
    return np.power(1.0 + x, delta - 1.0) * (1.0 + (1.0 + delta * gamma) * x)

def vpred_from_params(r, vbar2, sigma_star, sigma0, p):
    log_r0, gamma, delta, eta = p
    r0 = np.exp(log_r0)
    E = enhancement(r, r0, gamma, delta, sigma_star, sigma0, eta)
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def residuals(p, r, vbar2, v_obs, sigma_star, sigma0):
    v_pred = vpred_from_params(r, vbar2, sigma_star, sigma0, p)
    w = 1.0 / np.maximum(v_obs, 20.0)
    return (v_pred - v_obs) * w

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    vbar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

    # baryonic enclosed mass and structural surface density
    M_enc = r * vbar2 / G
    Mtot = M_enc[-1]
    frac = M_enc / max(Mtot, EPS)
    ihalf = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[ihalf]
    sigma_star = M_enc[ihalf] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(v_gas[i_vmax])
    vd = abs(v_disk_s[i_vmax])
    vb = abs(v_bul_s[i_vmax])
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "sigma_star": sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
sigma0 = np.median([g["sigma_star"] for g in galaxies])
print("Sigma0 =", sigma0)
print("log10 Sigma0 =", np.log10(sigma0))
print()

# -----------------------------
# fit each galaxy
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]
    sigma_star = g["sigma_star"]

    # same baseline as before, with eta added
    p0 = np.array([np.log(np.median(r)), 0.10, 1.80, 0.50])  # log_r0, gamma, delta, eta

    lo = np.array([np.log(np.min(r) * 0.2), 0.005, 0.05, 0.00])
    hi = np.array([np.log(np.max(r) * 30.0), 1.50, 3.50, 1.50])

    try:
        res = least_squares(
            residuals, p0,
            bounds=(lo, hi),
            args=(r, vbar2, v_obs, sigma_star, sigma0),
            max_nfev=4000,
            xtol=1e-10, ftol=1e-10, gtol=1e-10,
        )
    except Exception:
        continue

    if not res.success:
        continue

    log_r0, gamma, delta, eta = res.x
    r0 = np.exp(log_r0)

    v_pred = vpred_from_params(r, vbar2, sigma_star, sigma0, res.x)
    resid = v_pred - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae  = np.mean(np.abs(resid))

    frac_signed = resid / np.maximum(v_obs, 1.0)
    xgeom = np.power(np.clip(r0 / np.clip(r, EPS, None), 0.0, None), gamma) * np.power(sigma_star / max(sigma0, EPS), eta)
    s_in  = np.median(frac_signed[xgeom > 1.0]) if np.any(xgeom > 1.0) else np.nan
    s_mid = np.median(frac_signed[(xgeom <= 1.0) & (xgeom > 0.3)]) if np.any((xgeom <= 1.0) & (xgeom > 0.3)) else np.nan
    s_out = np.median(frac_signed[xgeom <= 0.3]) if np.any(xgeom <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "r0": r0,
        "gamma": gamma,
        "delta": delta,
        "eta": eta,
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
        "logSigma": np.log10(sigma_star),
    })

print("Galaxies fit:", len(rows))
print()

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
gamma_list = [r["gamma"] for r in rows]
delta_list = [r["delta"] for r in rows]
eta_list   = [r["eta"] for r in rows]
r0_list    = [r["r0"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("METRIC-COMPRESSION + COMPACTNESS TEST")
print("="*120)
print("GLOBAL SUMMARY")
print("="*120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median gamma:             ", med(gamma_list))
print("Median delta:             ", med(delta_list))
print("Median eta:               ", med(eta_list))
print("Median r0:                ", med(r0_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("="*120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"r0":>8} {"gamma":>8} {"delta":>8} {"eta":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["r0"]:8.3f} {row["gamma"]:8.3f} {row["delta"]:8.3f} {row["eta"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("="*120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"r0":>8} {"gamma":>8} {"delta":>8} {"eta":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["r0"]:8.3f} {row["gamma"]:8.3f} {row["delta"]:8.3f} {row["eta"]:8.3f}')

Galaxies loaded: 60
Sigma0 = 362913764.3374412
log10 Sigma0 = 8.559803440127908

Galaxies fit: 60

METRIC-COMPRESSION + COMPACTNESS TEST
GLOBAL SUMMARY
Median RMSE:               27.13118789508951
Mean RMSE:                 31.977037385578754
Median MAE:                22.254734996561126
Median signed inner resid: -0.062175768709169535
Median signed mid resid:   0.007156919981177917
Median signed outer resid: -0.10706280708233738
Median gamma:              0.005000000000000001
Median delta:              1.454269704639486
Median eta:                1.8584661652073904e-07
Median r0:                 377.5499999999543
Median low-Vmax RMSE:      16.413524945196663
Median high-Vmax RMSE:     39.72147533230445

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out       r0    gamma    delta      eta
UGCA444_rotmod.dat           2.13     1.69    0.008      nan      nan   78.600    0.005    3.328    0.000
UGC04305_rotmod.dat          3.38     2.77    0.003      

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# UNIVERSAL METRIC COMPRESSION LAW (V3 - PHYSICAL SCALE)
# 1. r0 is NO LONGER a free parameter; it is tied to Rh (Half-Mass)
# 2. Delta scales with Baryonic Mass (Mb)
# 3. Gamma is a Universal Constant (The "Vortex" Slope)
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# Standard Stellar Mass-to-Light Ratios
UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
M0 = 1e10  # Reference Mass (10 billion solar masses)

# Quality Controls
MIN_POINTS = 15
EPS = 1e-8

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try: rows.append([float(x) for x in parts[:5]])
                    except: pass
    if not rows: return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (np.isfinite(r) & (v_obs > 0) & (r > 0))
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def enhancement(r, r0, gamma, delta):
    # The "Vortex" logic: x represents the metric compression ratio
    x = np.power(np.clip(r0 / np.clip(r, EPS, None), 0.0, None), gamma)
    return np.power(1.0 + x, delta - 1.0) * (1.0 + (1.0 + delta * gamma) * x)

# -----------------------------
# 1. Load and Pre-Process
# -----------------------------
galaxies = []
for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None or len(data[0]) < MIN_POINTS: continue
    r, v_obs, v_gas, v_disk, v_bul = data

    # Calculate Baryonic Velocity Squared
    vbar2 = np.maximum(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2, 0.0)

    # Estimate Total Baryonic Mass and Half-Mass Radius (Rh)
    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = np.searchsorted(frac, 0.5)
    Rh = r[min(idx_h, len(r)-1)]

    galaxies.append({
        "name": os.path.basename(path),
        "r": r, "v_obs": v_obs, "vbar2": vbar2,
        "Mb": Mb, "Rh": Rh
    })

print(f"Testing logic on {len(galaxies)} galaxies...")

# -----------------------------
# 2. Global Solver
# -----------------------------
def global_residuals(params):
    delta0, delta1, gamma, alpha = params
    all_errs = []

    for g in galaxies:
        # Predict Delta based on Mass
        delta = delta0 + delta1 * np.log10(g["Mb"] / M0)

        # Predict r0 based on physical scale of the stars (Rh)
        # alpha is the scaling factor: does the vortex start at 1Rh or 5Rh?
        r0 = alpha * g["Rh"]

        E = enhancement(g["r"], r0, gamma, delta)
        v_pred = np.sqrt(np.maximum(g["vbar2"] * E, 0.0))

        # Weight by velocity to prevent high-speed galaxies from dominating
        err = (v_pred - g["v_obs"]) / np.maximum(g["v_obs"], 10.0)
        all_errs.extend(err)

    return np.array(all_errs)

# Initial Guesses: [delta0, delta1, gamma, alpha]
p_start = [1.2, -0.2, 0.2, 2.0]
p_bounds = ([0.1, -1.0, 0.01, 0.1], [5.0, 1.0, 2.0, 20.0])

res = least_squares(global_residuals, p_start, bounds=p_bounds, ftol=1e-4)
d0, d1, gam, alp = res.x

print("\n--- UNIVERSAL PARAMETERS FOUND ---")
print(f"Base Delta (d0): {d0:.4f}")
print(f"Mass Scaling (d1): {d1:.4f}")
print(f"Vortex Slope (gamma): {gam:.4f}")
print(f"Scale Multiplier (alpha): {alp:.4f}")

# -----------------------------
# 3. Final Evaluation
# -----------------------------
results = []
for g in galaxies:
    delta = d0 + d1 * np.log10(g["Mb"] / M0)
    r0 = alp * g["Rh"]
    v_p = np.sqrt(np.maximum(g["vbar2"] * enhancement(g["r"], r0, gam, delta), 0.0))

    rmse = np.sqrt(np.mean((v_p - g["v_obs"])**2))
    results.append({"name": g["name"], "rmse": rmse, "Mb": g["Mb"]})

results = sorted(results, key=lambda x: x["rmse"])
print(f"\nGlobal Median RMSE: {np.median([x['rmse'] for x in results]):.2f} km/s")

print("\nTOP 5 SUCCESSES (Logic Holds):")
for r in results[:5]: print(f"{r['name']:<25} RMSE: {r['rmse']:.2f}")

print("\nBOTTOM 5 FAILURES (The 'Holes'):")
for r in results[-5:]: print(f"{r['name']:<25} RMSE: {r['rmse']:.2f}")


Testing logic on 84 galaxies...

--- UNIVERSAL PARAMETERS FOUND ---
Base Delta (d0): 1.4716
Mass Scaling (d1): -0.2975
Vortex Slope (gamma): 0.0100
Scale Multiplier (alpha): 14.2125

Global Median RMSE: 37.85 km/s

TOP 5 SUCCESSES (Logic Holds):
KK98-251_rotmod.dat       RMSE: 4.57
UGCA444_rotmod.dat        RMSE: 10.17
UGC04305_rotmod.dat       RMSE: 12.12
UGC01281_rotmod.dat       RMSE: 12.86
NGC2366_rotmod.dat        RMSE: 13.19

BOTTOM 5 FAILURES (The 'Holes'):
NGC2841_rotmod.dat        RMSE: 109.86
UGC05253_rotmod.dat       RMSE: 112.82
NGC7814_rotmod.dat        RMSE: 114.49
UGC06787_rotmod.dat       RMSE: 124.91
UGC02487_rotmod.dat       RMSE: 134.23


In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# FIRST PRINCIPLES: ACCELERATION-DEPENDENT METRIC COMPRESSION
# Logic: If gravity is "converted motion," the metric
# compresses when the "motion pressure" (acceleration) is low.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try: rows.append([float(x) for x in parts[:5]])
                    except: pass
    if not rows: return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (np.isfinite(r) & (v_obs > 0) & (r > 0))
    # Using standard M/L ratios to keep it "First Principles"
    vbar2 = v_gas**2 + (0.5*v_disk)**2 + (0.7*v_bul)**2
    return r[mask], v_obs[mask], vbar2[mask]

# -----------------------------
# The Metric Law
# -----------------------------
def v_predict(r, vbar2, a0, gamma):
    # g_newton is the "local motion pressure"
    g_n = vbar2 / r

    # The "Vortex" Factor (E)
    # Triggered by the ratio of a0 / g_n
    x = np.power(a0 / np.maximum(g_n, 1e-15), gamma)
    E = np.power(1.0 + x, 1.0) # Simple linear scaling

    return np.sqrt(vbar2 * E)

# -----------------------------
# Load and Fit
# -----------------------------
galaxies = []
for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data:
        galaxies.append({"name": os.path.basename(path), "r": data[0], "v_obs": data[1], "vbar2": data[2]})

def global_residuals(p):
    a0, gamma = p
    errs = []
    for g in galaxies:
        vp = v_predict(g['r'], g['vbar2'], a0, gamma)
        errs.extend((vp - g['v_obs']) / np.maximum(g['v_obs'], 10.0))
    return np.array(errs)

# Start with a0 ~ 1.2e-10 m/s^2 (converted to kpc/km/s units)
# 1.2e-10 m/s^2 is approx 3700 in these units
res = least_squares(global_residuals, [3700.0, 0.5], bounds=([100.0, 0.1], [10000.0, 1.0]))

a0_fit, gam_fit = res.x
print(f"Universal a0: {a0_fit:.2f}")
print(f"Universal Gamma: {gam_fit:.4f}")

# Evaluate
results = []
for g in galaxies:
    vp = v_predict(g['r'], g['vbar2'], a0_fit, gam_fit)
    rmse = np.sqrt(np.mean((vp - g['v_obs'])**2))
    results.append({"name": g["name"], "rmse": rmse})

results = sorted(results, key=lambda x: x["rmse"])
print(f"\nMedian RMSE: {np.median([x['rmse'] for x in results]):.2f} km/s")
print("\nTop Successes:")
for r in results[:5]: print(f"{r['name']:<25} RMSE: {r['rmse']:.2f}")


Universal a0: 1569.69
Universal Gamma: 0.5747

Median RMSE: 15.41 km/s

Top Successes:
UGCA281_rotmod.dat        RMSE: 1.15
UGC04483_rotmod.dat       RMSE: 2.00
F583-4_rotmod.dat         RMSE: 2.10
UGC05918_rotmod.dat       RMSE: 3.18
KK98-251_rotmod.dat       RMSE: 3.60


In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# FIRST PRINCIPLES: SPIN-COUPLED METRIC COMPRESSION
# Logic: a0 triggers the vortex, but the "Twist" (gamma)
# is powered by the Angular Momentum of the Baryonic Mass.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try: rows.append([float(x) for x in parts[:5]])
                    except: pass
    if not rows: return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (np.isfinite(r) & (v_obs > 0) & (r > 0))
    # Standard M/L Ratios
    vbar2 = v_gas**2 + (0.5*v_disk)**2 + (0.7*v_bul)**2
    return r[mask], v_obs[mask], vbar2[mask]

# -----------------------------
# The Metric Law with Spin Scaling
# -----------------------------
def v_predict(r, vbar2, a0, gamma_base, L_scale):
    g_n = vbar2 / r

    # Calculate an estimate of specific angular momentum (r * v)
    # We use the baryonic velocity as the "source" of the spin
    L_specific = np.mean(r * np.sqrt(vbar2))

    # Gamma (the twist) now scales with the spin of the system
    # L_scale determines how much the spin affects the metric
    gamma = gamma_base + L_scale * np.log10(max(L_specific, 1.0) / 1000.0)
    gamma = np.clip(gamma, 0.01, 2.0)

    x = np.power(a0 / np.maximum(g_n, 1e-15), gamma)
    E = 1.0 + x # Metric Enhancement

    return np.sqrt(vbar2 * E)

# -----------------------------
# Load and Global Fit
# -----------------------------
galaxies = []
for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data:
        galaxies.append({"name": os.path.basename(path), "r": data[0], "v_obs": data[1], "vbar2": data[2]})

def global_residuals(p):
    a0, gamma_base, L_scale = p
    errs = []
    for g in galaxies:
        vp = v_predict(g['r'], g['vbar2'], a0, gamma_base, L_scale)
        # Weight by observation to keep errors relative
        errs.extend((vp - g['v_obs']) / np.maximum(g['v_obs'], 10.0))
    return np.array(errs)

# Initial Guesses: [a0, gamma_base, L_scale]
# We're testing if L_scale is non-zero (meaning spin matters)
p_start = [1570.0, 0.57, 0.0]
res = least_squares(global_residuals, p_start, bounds=([100.0, 0.1, -1.0], [10000.0, 1.5, 1.0]))

a0_f, gam_f, l_f = res.x
print(f"Universal a0: {a0_f:.2f}")
print(f"Base Gamma: {gam_f:.4f}")
print(f"Spin Coupling (L_scale): {l_f:.4f}")

# Final Evaluation
results = []
for g in galaxies:
    vp = v_predict(g['r'], g['vbar2'], a0_f, gam_f, l_f)
    rmse = np.sqrt(np.mean((vp - g['v_obs'])**2))
    results.append({"name": g["name"], "rmse": rmse})

results = sorted(results, key=lambda x: x["rmse"])
print(f"\nMedian RMSE: {np.median([x['rmse'] for x in results]):.2f} km/s")

print("\nTop Successes:")
for r in results[:5]: print(f"{r['name']:<25} RMSE: {r['rmse']:.2f}")


Universal a0: 1666.57
Base Gamma: 0.6778
Spin Coupling (L_scale): 0.1316

Median RMSE: 15.08 km/s

Top Successes:
D512-2_rotmod.dat         RMSE: 1.46
UGC07866_rotmod.dat       RMSE: 1.59
F583-4_rotmod.dat         RMSE: 2.20
KK98-251_rotmod.dat       RMSE: 3.15
UGCA281_rotmod.dat        RMSE: 3.34


In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION-TRIGGERED METRIC COMPRESSION
# ------------------------------------------------------------
# g_obs = g_N * [1 + (a0 / g_N)^gamma]
# V_pred^2 = V_bar^2 * [1 + (a0 / g_N)^gamma]
#
# This version:
# - uses the same style of quality cuts as before
# - fits one GLOBAL a0 and gamma
# - reports:
#     median RMSE / MAE
#     low-mass / high-mass RMSE
#     signed inner / mid / outer residuals
#     best/worst galaxies
#     correlations of RMSE with logM, bulge fraction, logSigma
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-8

# -----------------------------
# quality cuts
# -----------------------------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    vbar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

    # baryonic mass estimate and structural quantities
    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r) - 1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(v_gas[i_vmax])
    vd = abs(v_disk_s[i_vmax])
    vb = abs(v_bul_s[i_vmax])
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# model
# -----------------------------
def v_predict(r, vbar2, a0, gamma):
    gN = vbar2 / np.maximum(r, EPS)
    x = np.power(a0 / np.maximum(gN, 1e-15), gamma)
    E = 1.0 + x
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], a0, gamma)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start near Gemini result
p0 = [1600.0, 0.57]
bounds_lo = [100.0, 0.05]
bounds_hi = [10000.0, 1.50]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma_fit = res.x

print("Fitted global parameters")
print("a0    =", a0_fit)
print("gamma =", gamma_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]

    vp = v_predict(r, vbar2, a0_fit, gamma_fit)
    resid = vp - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    # use gN/a0 as the natural regime variable
    gN = vbar2 / np.maximum(r, EPS)
    y = gN / a0_fit

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": np.max(g["Vmax_obs"]),
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION-TRIGGERED COMPRESSION TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted global parameters
a0    = 2511.6391613420924
gamma = 0.5383589810634039

ACCELERATION-TRIGGERED COMPRESSION TEST
GLOBAL SUMMARY
Median RMSE:               15.817854398440584
Mean RMSE:                 30.797434564264403
Median MAE:                14.49399526797433
Median signed inner resid: -0.02521157277551773
Median signed mid resid:   -0.03872976416079514
Median signed outer resid: -0.06577309800575733
Median low-Vmax RMSE:      9.981781927367797
Median high-Vmax RMSE:     41.92236334199501

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGCA444_rotmod.dat           2.76     2.45      nan      nan   -0.084    7.870    7.075    0.200
NGC4183_rotmod.dat           3.89     2.80      nan   -0.002   -0.004    9.910    8.004    0.694
UGC07524_rotmod.dat          4.13     3.34      nan      nan   -0.018    9.407    7.396    0.563
NGC0247_rotmod.dat           4.23     3.55      nan      nan   -0.0

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION + COMPACTNESS RESPONSE LAW
# ------------------------------------------------------------
# g_obs = g_N * [1 + (a0/g_N)^gamma * (Sigma_*/Sigma0)^beta]
#
# Global parameters:
#   a0, gamma, beta
#
# Same diagnostics style as before.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-8

# -----------------------------
# quality cuts
# -----------------------------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    vbar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r) - 1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(v_gas[i_vmax])
    vd = abs(v_disk_s[i_vmax])
    vb = abs(v_bul_s[i_vmax])
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
Sigma0 = np.median([g["Sigma_star"] for g in galaxies])
print("Sigma0 =", Sigma0)
print("log10 Sigma0 =", np.log10(Sigma0))
print()

# -----------------------------
# model
# -----------------------------
def v_predict(r, vbar2, sigma_star, sigma0, a0, gamma, beta):
    gN = vbar2 / np.maximum(r, EPS)
    compact = np.power(np.clip(sigma_star / max(sigma0, EPS), EPS, None), beta)
    x = np.power(a0 / np.maximum(gN, 1e-15), gamma) * compact
    E = 1.0 + x
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma, beta = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], g["Sigma_star"], Sigma0, a0, gamma, beta)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from previous good solution with beta=0
p0 = [2511.6, 0.538, 0.0]
bounds_lo = [100.0, 0.05, -2.0]
bounds_hi = [10000.0, 1.50, 2.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma_fit, beta_fit = res.x

print("Fitted global parameters")
print("a0    =", a0_fit)
print("gamma =", gamma_fit)
print("beta  =", beta_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]
    vp = v_predict(r, vbar2, g["Sigma_star"], Sigma0, a0_fit, gamma_fit, beta_fit)

    resid = vp - v_obs
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    gN = vbar2 / np.maximum(r, EPS)
    y = gN / a0_fit

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION + COMPACTNESS RESPONSE TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60
Sigma0 = 362913764.3374412
log10 Sigma0 = 8.559803440127908

Fitted global parameters
a0    = 2128.96713518715
gamma = 0.6368029780391834
beta  = 0.1397326416908919

ACCELERATION + COMPACTNESS RESPONSE TEST
GLOBAL SUMMARY
Median RMSE:               18.009543394637063
Mean RMSE:                 31.60180344264683
Median MAE:                16.849601728689777
Median signed inner resid: -0.061439995464822976
Median signed mid resid:   0.003655733169602838
Median signed outer resid: -0.02933623309135225
Median low-Vmax RMSE:      10.225304780358483
Median high-Vmax RMSE:     45.16662174286952

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGCA444_rotmod.dat           4.72     4.18      nan      nan   -0.161    7.870    7.075    0.200
NGC0055_rotmod.dat           5.01     4.21      nan      nan   -0.010    9.656    7.559    0.618
DDO161_rotmod.dat            5.30     4.62      nan      nan    0.006    8.9

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION LAW WITH COMPACTNESS-SHIFTED THRESHOLD
# ------------------------------------------------------------
# a0_eff = a0 * (Sigma_*/Sigma0)^beta
#
# g_obs = g_N * [1 + (a0_eff / g_N)^gamma]
# V_pred^2 = V_bar^2 * [1 + (a0_eff / g_N)^gamma]
#
# Global fit only:
#   a0, gamma, beta
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-8

# -----------------------------
# quality cuts
# -----------------------------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vgas2  = np.maximum(v_gas**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
Sigma0 = np.median([g["Sigma_star"] for g in galaxies])
print("Sigma0 =", Sigma0)
print("log10 Sigma0 =", np.log10(Sigma0))
print()

# -----------------------------
# model
# -----------------------------
def v_predict(r, vbar2, sigma_star, sigma0, a0, gamma, beta):
    gN = vbar2 / np.maximum(r, EPS)
    a0_eff = a0 * np.power(np.clip(sigma_star / max(sigma0, EPS), EPS, None), beta)
    E = 1.0 + np.power(a0_eff / np.maximum(gN, 1e-15), gamma)
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma, beta = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], g["Sigma_star"], Sigma0, a0, gamma, beta)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from best plain acceleration fit
p0 = [2511.6, 0.538, 0.0]
bounds_lo = [100.0, 0.05, -2.0]
bounds_hi = [10000.0, 1.50, 2.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma_fit, beta_fit = res.x

print("Fitted global parameters")
print("a0    =", a0_fit)
print("gamma =", gamma_fit)
print("beta  =", beta_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]

    vp = v_predict(r, vbar2, g["Sigma_star"], Sigma0, a0_fit, gamma_fit, beta_fit)
    resid = vp - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    gN = vbar2 / np.maximum(r, EPS)
    a0_eff = a0_fit * np.power(np.clip(g["Sigma_star"] / max(Sigma0, EPS), EPS, None), beta_fit)
    y = gN / a0_eff

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION LAW WITH SHIFTED a0 TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60
Sigma0 = 362913764.3374412
log10 Sigma0 = 8.559803440127908

Fitted global parameters
a0    = 2128.9429836804284
gamma = 0.6368083309083343
beta  = 0.21943736072573589

ACCELERATION LAW WITH SHIFTED a0 TEST
GLOBAL SUMMARY
Median RMSE:               18.00958513916801
Mean RMSE:                 31.60191402346766
Median MAE:                16.84962231384661
Median signed inner resid: -0.026100148465284974
Median signed mid resid:   0.017475562109134056
Median signed outer resid: -0.03379927929922592
Median low-Vmax RMSE:      10.225238880704113
Median high-Vmax RMSE:     45.16700705111312

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGCA444_rotmod.dat           4.72     4.19      nan      nan   -0.161    7.870    7.075    0.200
NGC0055_rotmod.dat           5.01     4.21      nan      nan   -0.010    9.656    7.559    0.618
DDO161_rotmod.dat            5.30     4.62      nan      nan    0.006    8.948

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION LAW WITH GLOBAL MASS-AMPLITUDE COUPLING
# ------------------------------------------------------------
# V_pred^2 = V_bar^2 * [1 + (a0 / gN)^gamma * (Mb / M0)^eta]
#
# where:
#   gN = V_bar^2 / r
#   Mb = total baryonic mass of the galaxy
#   M0 = 1e10 Msun
#
# Global fit:
#   a0, gamma, eta
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
M0 = 1e10
EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# model
# -----------------------------
def v_predict(r, vbar2, Mb, a0, gamma, eta):
    gN = vbar2 / np.maximum(r, EPS)
    mass_amp = np.power(np.clip(Mb / M0, EPS, None), eta)
    E = 1.0 + np.power(a0 / np.maximum(gN, 1e-15), gamma) * mass_amp
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma, eta = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], g["Mb"], a0, gamma, eta)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from the best plain acceleration fit
p0 = [2511.6, 0.538, 0.0]
bounds_lo = [100.0, 0.05, -2.0]
bounds_hi = [10000.0, 1.50, 2.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma_fit, eta_fit = res.x

print("Fitted global parameters")
print("a0    =", a0_fit)
print("gamma =", gamma_fit)
print("eta   =", eta_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]

    vp = v_predict(r, vbar2, g["Mb"], a0_fit, gamma_fit, eta_fit)
    resid = vp - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    gN = vbar2 / np.maximum(r, EPS)
    a0_eff = a0_fit * np.power(np.clip(g["Mb"] / M0, EPS, None), eta_fit)
    y = gN / a0_eff

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION LAW WITH MASS-AMPLITUDE TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted global parameters
a0    = 1976.5604897379621
gamma = 0.6289692250127583
eta   = 0.13220946143755558

ACCELERATION LAW WITH MASS-AMPLITUDE TEST
GLOBAL SUMMARY
Median RMSE:               18.423643078988505
Mean RMSE:                 30.63730531334519
Median MAE:                16.704273650747794
Median signed inner resid: -0.03280736786424279
Median signed mid resid:   -0.02434640328640739
Median signed outer resid: -0.02690851778470215
Median low-Vmax RMSE:      9.162807265796513
Median high-Vmax RMSE:     43.02900555480106

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC0247_rotmod.dat           3.99     3.23      nan      nan   -0.031    9.729    7.694    0.590
NGC6503_rotmod.dat           4.85     3.58    0.059   -0.024   -0.022    9.892    9.008    0.908
NGC2976_rotmod.dat           4.99     4.17   -0.049    0.079      nan    9.186    8.639    0.733
UGC07524_rotmod.dat          5.02    

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION LAW WITH GLOBAL WELL-DEPTH COUPLING
# ------------------------------------------------------------
# V_pred^2 = V_bar^2 * [1 + (a0 / gN)^gamma * (gbar_max / g0)^xi]
#
# where:
#   gN(r)      = V_bar^2 / r
#   gbar_max   = max_r(V_bar^2 / r) for each galaxy
#   g0         = sample median of gbar_max
#
# Global fit:
#   a0, gamma, xi
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    gN_profile = vbar2 / np.maximum(r, EPS)
    gbar_max = np.max(gN_profile)

    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "gbar_max": gbar_max,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
g0 = np.median([g["gbar_max"] for g in galaxies])
print("g0 =", g0)
print("log10 g0 =", np.log10(g0))
print()

# -----------------------------
# model
# -----------------------------
def v_predict(r, vbar2, gbar_max, g0, a0, gamma, xi):
    gN = vbar2 / np.maximum(r, EPS)
    depth_amp = np.power(np.clip(gbar_max / max(g0, EPS), EPS, None), xi)
    E = 1.0 + np.power(a0 / np.maximum(gN, 1e-15), gamma) * depth_amp
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma, xi = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], g["gbar_max"], g0, a0, gamma, xi)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from best plain acceleration fit
p0 = [2511.6, 0.538, 0.0]
bounds_lo = [100.0, 0.05, -2.0]
bounds_hi = [10000.0, 1.50, 2.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma_fit, xi_fit = res.x

print("Fitted global parameters")
print("a0    =", a0_fit)
print("gamma =", gamma_fit)
print("xi    =", xi_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]

    vp = v_predict(r, vbar2, g["gbar_max"], g0, a0_fit, gamma_fit, xi_fit)
    resid = vp - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    gN = vbar2 / np.maximum(r, EPS)
    a0_eff = a0_fit * np.power(np.clip(g["gbar_max"] / max(g0, EPS), EPS, None), xi_fit)
    y = gN / a0_eff

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "loggmax": np.log10(g["gbar_max"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION LAW WITH WELL-DEPTH TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"loggmx":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["loggmax"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"loggmx":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["loggmax"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("log10(gbar_max):   ", corr([r["loggmax"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60
g0 = 3160.9080078324228
log10 g0 = 3.499811856715217

Fitted global parameters
a0    = 2047.6309812773582
gamma = 0.6483906068165525
xi    = 0.1631352824852911

ACCELERATION LAW WITH WELL-DEPTH TEST
GLOBAL SUMMARY
Median RMSE:               19.49834559395386
Mean RMSE:                 31.87158419947673
Median MAE:                17.88330946021597
Median signed inner resid: -0.11354624244521497
Median signed mid resid:   -0.0070595249424321264
Median signed outer resid: -0.03706532652790423
Median low-Vmax RMSE:      9.312055513604822
Median high-Vmax RMSE:     48.379207963934476

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig   loggmx    bulge
IC2574_rotmod.dat            5.09     4.52      nan      nan    0.071    9.119    7.197    1.910    0.423
NGC2976_rotmod.dat           5.14     4.24    0.003    0.120      nan    9.186    8.639    3.300    0.733
DDO161_rotmod.dat            5.16     4.70      nan      n

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION LAW WITH MASS-DEPENDENT TRANSITION SHARPNESS
# ------------------------------------------------------------
# gamma(Mb) = gamma0 + gamma1 * log10(Mb / M0)
#
# V_pred^2 = V_bar^2 * [1 + (a0 / gN)^gamma(Mb)]
# gN       = V_bar^2 / r
#
# Global fit:
#   a0, gamma0, gamma1
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
M0 = 1e10
EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "vbar2": vbar2,
        "Mb": Mb,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# model
# -----------------------------
def gamma_of_mass(Mb, gamma0, gamma1):
    return gamma0 + gamma1 * np.log10(np.clip(Mb / M0, EPS, None))

def v_predict(r, vbar2, Mb, a0, gamma0, gamma1):
    gN = vbar2 / np.maximum(r, EPS)
    gam = gamma_of_mass(Mb, gamma0, gamma1)
    # keep exponent in a sane positive range numerically
    gam = np.clip(gam, 0.02, 2.5)
    E = 1.0 + np.power(a0 / np.maximum(gN, 1e-15), gam)
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma0, gamma1 = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["vbar2"], g["Mb"], a0, gamma0, gamma1)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from best plain acceleration fit
# if gamma1=0 this reduces to the current best model
p0 = [2511.6, 0.538, 0.0]
bounds_lo = [100.0, 0.05, -1.0]
bounds_hi = [10000.0, 1.50, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
a0_fit, gamma0_fit, gamma1_fit = res.x

print("Fitted global parameters")
print("a0     =", a0_fit)
print("gamma0 =", gamma0_fit)
print("gamma1 =", gamma1_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    vbar2 = g["vbar2"]
    gam = np.clip(gamma_of_mass(g["Mb"], gamma0_fit, gamma1_fit), 0.02, 2.5)

    vp = v_predict(r, vbar2, g["Mb"], a0_fit, gamma0_fit, gamma1_fit)
    resid = vp - v_obs

    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    gN = vbar2 / np.maximum(r, EPS)
    y = gN / a0_fit

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "gamma_eff": gam,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
gamma_eff_list = [r["gamma_eff"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION LAW WITH MASS-DEPENDENT GAMMA TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median gamma_eff:         ", med(gamma_eff_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"gamEff":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["gamma_eff"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"gamEff":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["gamma_eff"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))
print("gamma_eff:         ", corr([r["gamma_eff"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted global parameters
a0     = 2558.114847669816
gamma0 = 0.5637201411657234
gamma1 = 0.0537405872792352

ACCELERATION LAW WITH MASS-DEPENDENT GAMMA TEST
GLOBAL SUMMARY
Median RMSE:               18.406149995922092
Mean RMSE:                 30.660124224513883
Median MAE:                17.073397703049935
Median signed inner resid: -0.02511323601167545
Median signed mid resid:   -0.02269723016331362
Median signed outer resid: -0.03322327537570359
Median gamma_eff:          0.5798032841372376
Median low-Vmax RMSE:      9.903786155412508
Median high-Vmax RMSE:     38.90824140669322

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out   gamEff     logM   logSig    bulge
NGC0247_rotmod.dat           3.39     2.81      nan      nan   -0.021    0.549    9.729    7.694    0.590
UGC07524_rotmod.dat          4.28     3.52      nan      nan   -0.023    0.532    9.407    7.396    0.563
NGC4183_rotmod.dat           4.33     3.80      nan  

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# ACCELERATION LAW + GLOBAL STELLAR M/L FIT
# ------------------------------------------------------------
# g_obs = g_N * [1 + (a0 / g_N)^gamma]
#
# where
#   g_N = V_bar^2 / r
#   V_bar^2 = V_gas^2 + (ups_disk * V_disk)^2 + (ups_bul * V_bul)^2
#
# Global fit:
#   a0, gamma, ups_disk, ups_bul
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

G = 4.30091e-6
EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "v_gas": v_gas,
        "v_disk": v_disk,
        "v_bul": v_bul,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# model
# -----------------------------
def build_vbar2(v_gas, v_disk, v_bul, ups_disk, ups_bul):
    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((ups_disk * v_disk)**2, 0.0)
    vbul2  = np.maximum((ups_bul  * v_bul )**2, 0.0)
    return vgas2 + vdisk2 + vbul2, vgas2, vdisk2, vbul2

def v_predict(r, v_gas, v_disk, v_bul, a0, gamma, ups_disk, ups_bul):
    vbar2, _, _, _ = build_vbar2(v_gas, v_disk, v_bul, ups_disk, ups_bul)
    gN = vbar2 / np.maximum(r, EPS)
    E = 1.0 + np.power(a0 / np.maximum(gN, 1e-15), gamma)
    return np.sqrt(np.maximum(vbar2 * E, 0.0))

def global_residuals(p):
    a0, gamma, ups_disk, ups_bul = p
    errs = []
    for g in galaxies:
        vp = v_predict(g["r"], g["v_gas"], g["v_disk"], g["v_bul"], a0, gamma, ups_disk, ups_bul)
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near current best law and standard SPARC-ish M/L guesses
p0 = [2511.6, 0.538, 0.50, 0.70]
bounds_lo = [100.0, 0.05, 0.05, 0.05]
bounds_hi = [10000.0, 1.50, 2.50, 3.50]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=8000)
a0_fit, gamma_fit, ups_d_fit, ups_b_fit = res.x

print("Fitted global parameters")
print("a0       =", a0_fit)
print("gamma    =", gamma_fit)
print("ups_disk =", ups_d_fit)
print("ups_bul  =", ups_b_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    r = g["r"]
    v_obs = g["v_obs"]
    v_gas = g["v_gas"]
    v_disk = g["v_disk"]
    v_bul = g["v_bul"]

    vbar2, vgas2, vdisk2, vbul2 = build_vbar2(v_gas, v_disk, v_bul, ups_d_fit, ups_b_fit)
    vp = v_predict(r, v_gas, v_disk, v_bul, a0_fit, gamma_fit, ups_d_fit, ups_b_fit)

    resid = vp - v_obs
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(v_obs, 1.0)

    # structural quantities
    M_enc = r * vbar2 / G
    Mb = M_enc[-1]
    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    gN = vbar2 / np.maximum(r, EPS)
    y = gN / a0_fit

    s_in  = np.median(frac_signed[y > 1.0]) if np.any(y > 1.0) else np.nan
    s_mid = np.median(frac_signed[(y <= 1.0) & (y > 0.3)]) if np.any((y <= 1.0) & (y > 0.3)) else np.nan
    s_out = np.median(frac_signed[y <= 0.3]) if np.any(y <= 0.3) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(Mb),
        "logSigma": np.log10(Sigma_star),
        "bulge_frac": vb / denom,
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("ACCELERATION LAW + GLOBAL M/L TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted global parameters
a0       = 2353.9322163681736
gamma    = 0.49135488616099565
ups_disk = 0.7897399854812461
ups_bul  = 0.7084845386257866

ACCELERATION LAW + GLOBAL M/L TEST
GLOBAL SUMMARY
Median RMSE:               15.370967254194419
Mean RMSE:                 31.038889074466603
Median MAE:                14.466495672310025
Median signed inner resid: -0.0010721312192866635
Median signed mid resid:   -0.04596263792080656
Median signed outer resid: -0.0759026591801298
Median low-Vmax RMSE:      9.7455841840783
Median high-Vmax RMSE:     42.38028663050266

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGCA444_rotmod.dat           1.31     1.03      nan      nan    0.013    8.071    7.424    0.166
UGC07524_rotmod.dat          4.53     3.86      nan      nan   -0.053    9.569    7.514    0.470
NGC4183_rotmod.dat           4.55     3.41      nan   -0.059   -0.001   10.024    8.051    0.615
NGC02

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES RADIAL STIFFNESS FIELD
# ------------------------------------------------------------
# Field equation from radial action:
#
#   Rs(M)^2 * m''(r) - (m(r) - m_inf) = -lambda * j_b(r)
#
# with
#   j_b(r) = (1/Mb) dM_enc/dr
#
# Cumulative stiffness:
#   S(r) = ∫_0^r m(s) * (s/Rs)^kappa ds
#
# Response law:
#   V(r) = Vinf(M) * [1 - exp(-S(r)/Sc(M))]
#
# where
#   Rs(M)   = AR * (M9)^beta_R
#   Vinf(M) = AV * (M9)^alpha_V
#
# This is the minimal first-principles closure consistent with
# the MTS document you pasted.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ---------- fixed mass scalings from your document ----------
AR = 1.402      # kpc
BETA_R = 0.279
AV = 50.542     # km/s
ALPHA_V = 0.311

# ---------- baryonic M/L ----------
UPS_DISK = 0.79
UPS_BUL  = 0.71

# ---------- constants ----------
G = 4.30091e-6  # kpc (km/s)^2 / Msun
EPS = 1e-12

# ---------- quality cuts ----------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    on the observed radial grid.

    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann: m0 - m1 = 0
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet: m(rmax)=m_inf
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_predict(r, Mb, jb, params):
    """
    params = [lam, m_inf, sc0, kappa]
    """
    lam, m_inf, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    # dimensional consistency: Sc carries the same radial scaling as the weighted integral
    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m, S, Rs, Vinf

# -----------------------------
# load sample
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    # enclosed baryonic mass inferred from baryonic circular support
    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r) - 1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# global fit of field-law parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _ = mts_predict(g["r"], g["Mb"], g["jb"], [lam, m_inf, sc0, kappa])
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# sensible starting point
p0 = [5.0, 0.5, 2.0, -0.05]
bounds_lo = [0.01, 0.001, 0.01, -1.0]
bounds_hi = [100.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=4000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit = res.x

print("Fitted first-principles field parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf = mts_predict(g["r"], g["Mb"], g["jb"], [lam_fit, m_inf_fit, sc0_fit, kappa_fit])
    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS FIRST-PRINCIPLES FIELD TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted first-principles field parameters
lambda = 12.720470668072167
m_inf  = 1.0208999017527156
sc0    = 3.0860198854064285
kappa  = -0.6232871727344884

MTS FIRST-PRINCIPLES FIELD TEST
GLOBAL SUMMARY
Median RMSE:               24.80995553400856
Mean RMSE:                 38.0033471862173
Median MAE:                21.577396952629115
Median signed inner resid: -0.31722682877449415
Median signed mid resid:   -0.14125475875060803
Median signed outer resid: -0.15306723202882022
Median low-Vinf RMSE:      14.092710416787995
Median high-Vinf RMSE:     59.49854381309224

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
IC2574_rotmod.dat            6.38     5.65   -0.605    0.236    0.043    9.299    7.268    0.356
UGC07524_rotmod.dat          6.38     5.53    0.013    0.188   -0.064    9.570    7.515    0.471
UGCA444_rotmod.dat           6.48     5.33    0.156   -0.079   -0.254    8.071    7.425    0.166
D

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES p-LAPLACIAN BACKBONE  (p fixed to 4)
# ------------------------------------------------------------
# Governing field:
#
#   (1/r^2) d/dr [ r^2 |dS/dr|^(p-2) dS/dr ] = alpha * rho_b(r)
#
# with p = 4  ->  quartic transport / marginal log regime.
#
# Constitutive relation:
#   m(r) = dS/du,   with   du = (r/Rs)^kappa dr
#
# Response law:
#   V(r) = Vinf(M) * [1 - exp( - S(r) / Sc(M) )]
#
# Global fixed mass scalings from your MTS draft:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fitted field-law parameters:
#   alpha, sc0, kappa
#
# No extra phenomenology branch. This is the direct first-principles
# p=4 cumulative-stiffness test.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS global laws from your document -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- fixed p-Laplacian choice -----
P_OP = 4.0

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6   # kpc (km/s)^2 / Msun
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def cumtrapz(y, x):
    out = np.zeros_like(x)
    for i in range(1, len(x)):
        out[i] = out[i-1] + 0.5*(y[i] + y[i-1])*(x[i] - x[i-1])
    return out

def solve_p4_Sprime(r, rho_b, alpha):
    """
    For spherical p-Laplacian:
      (1/r^2) d/dr [ r^2 |S'|^(p-2) S' ] = alpha rho_b
    with p = 4.

    Integrate once:
      r^2 |S'|^2 S' = alpha * Msrc(r)
    where Msrc(r)=∫_0^r rho_b(s) s^2 ds

    Since source is nonnegative, take S' >= 0:
      S' = [ alpha * Msrc(r) / r^2 ]^(1/3)
    """
    Msrc = cumtrapz(rho_b * r**2, r)
    rhs = alpha * Msrc / np.maximum(r**2, EPS)
    Sprime = np.power(np.clip(rhs, 0.0, None), 1.0/3.0)
    return Sprime, Msrc

def mts_p4_predict(r, Mb, rho_b, params):
    """
    params = [alpha, sc0, kappa]
    """
    alpha, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    Sprime_r, Msrc = solve_p4_Sprime(r, rho_b, alpha)

    # convert dS/dr into stiffness density m = dS/du
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    m_eff = Sprime_r / np.maximum(weight, EPS)

    # cumulative stiffness in weighted coordinate:
    # dS = m du = m * weight * dr = Sprime_r * dr
    S = cumtrapz(Sprime_r, r)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m_eff, S, Sprime_r, Rs, Vinf

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)

    # effective spherical source density from enclosed-mass derivative
    rho_b = dMdr / (4.0 * np.pi * np.maximum(r**2, EPS))
    rho_b = np.clip(rho_b, 0.0, None)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "rho_b": rho_b,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global first-principles parameters
# -----------------------------
def global_residuals(p):
    alpha, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_p4_predict(g["r"], g["Mb"], g["rho_b"], [alpha, sc0, kappa])
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

p0 = [2000.0, 2.0, -0.2]
bounds_lo = [1e-6, 1e-4, -1.5]
bounds_hi = [1e8, 1e3, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=4000)
alpha_fit, sc0_fit, kappa_fit = res.x

print("Fitted p=4 first-principles parameters")
print("alpha =", alpha_fit)
print("sc0   =", sc0_fit)
print("kappa =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, m_eff, S, Sprime_r, Rs, Vinf = mts_p4_predict(g["r"], g["Mb"], g["rho_b"], [alpha_fit, sc0_fit, kappa_fit])
    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS p=4 FIRST-PRINCIPLES FIELD TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted p=4 first-principles parameters
alpha = 1022.7994444390379
sc0   = 990.9813612452248
kappa = -0.3556142475976364

MTS p=4 FIRST-PRINCIPLES FIELD TEST
GLOBAL SUMMARY
Median RMSE:               26.308545463308327
Mean RMSE:                 36.47432450887528
Median MAE:                19.203098574043487
Median signed inner resid: -0.4497412919498739
Median signed mid resid:   -0.08407191580872288
Median signed outer resid: -0.138704978940017
Median low-Vinf RMSE:      14.114260193914177
Median high-Vinf RMSE:     51.8772000348788

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC2366_rotmod.dat           4.77     4.22   -0.025    0.109   -0.074    8.891    7.555    0.345
UGC01281_rotmod.dat          8.31     6.63    0.193    0.091   -0.248    8.712    7.578    0.391
UGCA444_rotmod.dat           8.38     8.16   -0.676   -0.279   -0.281    8.071    7.424    0.166
UGC07524_rotmod.dat          9.22

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES p=4 FIELD WITH CURVATURE-STRENGTH SOURCE
# ------------------------------------------------------------
# Governing field:
#
#   (1/r^2) d/dr [ r^2 |S'|^(p-2) S' ] = alpha * rho_b(r) * g_b(r)^gamma_src
#
# with p = 4.
#
# Geometry:
#   du = (r/Rs)^kappa dr
#   m = dS/du
#
# Response:
#   V(r) = Vinf(M) * [1 - exp(-S(r)/Sc(M))]
#
# Fixed global laws from your MTS draft:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fitted parameters:
#   alpha, gamma_src, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

P_OP = 4.0

# Use the globally fitted M/L that gave your best acceleration-law run
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

G = 4.30091e-6
EPS = 1e-12

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def cumtrapz(y, x):
    out = np.zeros_like(x)
    for i in range(1, len(x)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def solve_p4_Sprime(r, rho_eff, alpha):
    """
    For p=4:
      (1/r^2) d/dr [r^2 |S'|^2 S'] = alpha * rho_eff(r)

    Integrate once:
      r^2 |S'|^2 S' = alpha * Msrc(r),
      Msrc(r)=∫_0^r rho_eff(s) s^2 ds

    Since source is nonnegative, take S' >= 0:
      S' = [ alpha * Msrc(r) / r^2 ]^(1/3)
    """
    Msrc = cumtrapz(rho_eff * r**2, r)
    rhs = alpha * Msrc / np.maximum(r**2, EPS)
    Sprime = np.power(np.clip(rhs, 0.0, None), 1.0/3.0)
    return Sprime, Msrc

def mts_p4_predict(r, Mb, rho_b, g_b, params):
    """
    params = [alpha, gamma_src, sc0, kappa]
    """
    alpha, gamma_src, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    # Effective source term: density times curvature strength
    rho_eff = rho_b * np.power(np.clip(g_b, EPS, None), gamma_src)

    Sprime_r, Msrc = solve_p4_Sprime(r, rho_eff, alpha)

    # Geometry / constitutive map
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    m_eff = Sprime_r / np.maximum(weight, EPS)

    # Since dS = S' dr
    S = cumtrapz(Sprime_r, r)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m_eff, S, Sprime_r, Rs, Vinf, rho_eff

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)

    # spherical effective density
    rho_b = dMdr / (4.0 * np.pi * np.maximum(r**2, EPS))
    rho_b = np.clip(rho_b, 0.0, None)

    # baryonic field strength
    g_b = G * M_enc / np.maximum(r**2, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "rho_b": rho_b,
        "g_b": g_b,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global first-principles parameters
# -----------------------------
def global_residuals(p):
    alpha, gamma_src, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _ = mts_p4_predict(
            g["r"], g["Mb"], g["rho_b"], g["g_b"],
            [alpha, gamma_src, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start from previous p=4 solution, with mild source-curvature coupling
p0 = [1022.8, 0.30, 991.0, -0.36]
bounds_lo = [1e-6, 0.0, 1e-4, -1.5]
bounds_hi = [1e8, 2.0, 1e4, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
alpha_fit, gamma_src_fit, sc0_fit, kappa_fit = res.x

print("Fitted p=4 curvature-coupled field parameters")
print("alpha      =", alpha_fit)
print("gamma_src  =", gamma_src_fit)
print("sc0        =", sc0_fit)
print("kappa      =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, m_eff, S, Sprime_r, Rs, Vinf, rho_eff = mts_p4_predict(
        g["r"], g["Mb"], g["rho_b"], g["g_b"],
        [alpha_fit, gamma_src_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS p=4 CURVATURE-COUPLED FIELD TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted p=4 curvature-coupled field parameters
alpha      = 723.0899517508658
gamma_src  = 0.8820845767596209
sc0        = 3632.781910907681
kappa      = 0.2879355939533613

MTS p=4 CURVATURE-COUPLED FIELD TEST
GLOBAL SUMMARY
Median RMSE:               26.80253158334353
Mean RMSE:                 36.125486147668994
Median MAE:                18.640114753609765
Median signed inner resid: -0.4060970762423996
Median signed mid resid:   -0.10785441367740825
Median signed outer resid: -0.13870490605023109
Median low-Vinf RMSE:      13.953686785005079
Median high-Vinf RMSE:     51.097833334373114

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC2366_rotmod.dat           4.66     4.14   -0.078    0.108   -0.074    8.891    7.555    0.345
UGC07524_rotmod.dat          6.84     5.55   -0.373    0.216   -0.013    9.569    7.514    0.470
F583-1_rotmod.dat            7.51     6.37    0.034    0.226    0.023    

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES TEST 1:
# Lapse-weighted cumulative stiffness
# ------------------------------------------------------------
# Field equation (kept simple, from earlier radial action):
#
#   Rs(M)^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Geometry / cumulative variable:
#
#   S(r) = ∫_0^r m(s) * (s/Rs)^kappa * N(s) ds
#
# Lapse profile:
#
#   N(r) = 1 - eps_t * exp(-r / Rt)
#   Rt   = tau_t * Rs
#
# Response:
#
#   V(r) = Vinf(M) * [1 - exp(-S(r)/Sc(M))]
#
# Fixed global laws from your MTS draft:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fitted parameters:
#   lambda, m_inf, sc0, kappa, eps_t, tau_t
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS global laws from your document -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- baryonic M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness_lapse(r, m, Rs, kappa, eps_t, tau_t):
    Rt = max(tau_t * Rs, EPS)
    N = 1.0 - eps_t * np.exp(-r / Rt)
    N = np.clip(N, 1e-4, None)

    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa) * N
    integrand = m * weight

    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S, N

def mts_lapse_predict(r, Mb, jb, params):
    """
    params = [lam, m_inf, sc0, kappa, eps_t, tau_t]
    """
    lam, m_inf, sc0, kappa, eps_t, tau_t = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S, N = cumulative_stiffness_lapse(r, m, Rs, kappa, eps_t, tau_t)

    # keep same dimensional convention as before
    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m, S, N, Rs, Vinf

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, eps_t, tau_t = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_lapse_predict(
            g["r"], g["Mb"], g["jb"],
            [lam, m_inf, sc0, kappa, eps_t, tau_t]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start near previous simple-field solution, with mild lapse
p0 = [12.7, 1.02, 3.09, -0.35, 0.20, 1.5]
bounds_lo = [0.01, 0.001, 0.01, -1.5, 0.0, 0.05]
bounds_hi = [100.0, 10.0, 100.0, 1.0, 0.95, 20.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=6000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, eps_t_fit, tau_t_fit = res.x

print("Fitted lapse-weighted field parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("eps_t  =", eps_t_fit)
print("tau_t  =", tau_t_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Nsol, Rs, Vinf = mts_lapse_predict(
        g["r"], g["Mb"], g["jb"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, eps_t_fit, tau_t_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS LAPSE-WEIGHTED FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted lapse-weighted field parameters
lambda = 30.64273885841893
m_inf  = 0.8103573493911441
sc0    = 2.3078110484241776
kappa  = -1.0753923170910014
eps_t  = 0.9499999998629344
tau_t  = 2.139206867316325

MTS LAPSE-WEIGHTED FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               24.823102320378812
Mean RMSE:                 35.75929593791045
Median MAE:                19.455839758963464
Median signed inner resid: -0.32259145561747415
Median signed mid resid:   -0.11244616568730463
Median signed outer resid: -0.14051779681826754
Median low-Vinf RMSE:      13.853043461586285
Median high-Vinf RMSE:     51.14636959605478

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGC07524_rotmod.dat          5.14     4.15   -0.195    0.095   -0.045    9.569    7.514    0.470
F583-1_rotmod.dat            5.82     4.95    0.162   -0.036   -0.086    9.655    7.239    0.256
NGC2366_rotmod.dat           6.29  

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS LAPSE RE-TEST (CLEAN AUDIT)
# ------------------------------------------------------------
# Purpose:
#   Isolate the lapse idea cleanly.
#
# Changes from previous lapse test:
#   1) Fix kappa = -0.35   (remove degeneracy with lapse)
#   2) Keep same simple field equation
#   3) Use gentler lapse profile:
#
#        N(r) = 1 / (1 + eps_t * exp(-r / Rt))
#        Rt   = tau_t * Rs
#
#   4) Fit only:
#        lambda, m_inf, sc0, eps_t, tau_t
#
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa * N(r) dr
#
# Response:
#   V(r) = Vinf(M) * [1 - exp(-S/Sc)]
#
# Fixed MTS scalings from your document:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- fixed stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- fixed geometric measure from repeated field tests -----
KAPPA_FIXED = -0.35

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior second derivative on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness_lapse(r, m, Rs, kappa, eps_t, tau_t):
    """
    Gentler lapse:
        N(r) = 1 / (1 + eps_t * exp(-r/Rt))
        Rt = tau_t * Rs
    """
    Rt = max(tau_t * Rs, EPS)
    N = 1.0 / (1.0 + eps_t * np.exp(-r / Rt))
    N = np.clip(N, 1e-6, None)

    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa) * N
    integrand = m * weight

    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S, N, integrand

def mts_lapse_predict(r, Mb, jb, params):
    """
    params = [lam, m_inf, sc0, eps_t, tau_t]
    """
    lam, m_inf, sc0, eps_t, tau_t = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S, N, integrand = cumulative_stiffness_lapse(r, m, Rs, KAPPA_FIXED, eps_t, tau_t)

    # keep same Sc convention, but now with fixed kappa
    Sc = sc0 * (Rs ** (1.0 + KAPPA_FIXED))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m, S, N, integrand, Rs, Vinf

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, eps_t, tau_t = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _ = mts_lapse_predict(
            g["r"], g["Mb"], g["jb"],
            [lam, m_inf, sc0, eps_t, tau_t]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near previous simple-field solution, but no extreme lapse assumption
p0 = [12.7, 1.02, 3.09, 0.25, 1.5]
bounds_lo = [0.01, 0.001, 0.01, 0.0, 0.05]
bounds_hi = [100.0, 10.0, 100.0, 5.0, 20.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=6000)
lam_fit, m_inf_fit, sc0_fit, eps_t_fit, tau_t_fit = res.x

print("Fitted clean lapse-audit parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", KAPPA_FIXED, "  # fixed")
print("eps_t  =", eps_t_fit)
print("tau_t  =", tau_t_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Nsol, Isol, Rs, Vinf = mts_lapse_predict(
        g["r"], g["Mb"], g["jb"],
        [lam_fit, m_inf_fit, sc0_fit, eps_t_fit, tau_t_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS CLEAN LAPSE-AUDIT TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted clean lapse-audit parameters
lambda = 21.34811975172858
m_inf  = 2.5170091316402554
sc0    = 0.6089153186966825
kappa  = -0.35   # fixed
eps_t  = 4.999820462584323
tau_t  = 15.08357062699566

MTS CLEAN LAPSE-AUDIT TEST
GLOBAL SUMMARY
Median RMSE:               25.23230457312369
Mean RMSE:                 38.71556145270403
Median MAE:                21.131537206319038
Median signed inner resid: -0.42000204394971263
Median signed mid resid:   -0.12057943827544734
Median signed outer resid: -0.14264125383500958
Median low-Vinf RMSE:      13.831736478643979
Median high-Vinf RMSE:     57.531791043825564

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGCA444_rotmod.dat           6.10     4.96    0.278   -0.003   -0.232    8.071    7.424    0.166
NGC2366_rotmod.dat           7.60     6.06    0.944    0.171   -0.074    8.891    7.555    0.345
UGC07524_rotmod.dat          8.12     6.43    0.066    0.

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES TEST:
# CURVATURE-SOURCED LINEAR STIFFNESS FIELD
# ------------------------------------------------------------
# Field equation:
#
#   Rs(M)^2 m'' - (m - m_inf) = -lambda * gsrc(r)
#
# where
#   gsrc(r) = g_b(r) / <g_b>
#   g_b(r)  = G M_enc(<r) / r^2
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V(r) = Vinf(M) * [1 - exp(-S/Sc(M))]
#
# Fixed MTS laws from your document:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fitted parameters:
#   lambda, m_inf, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS global laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, gsrc, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * gsrc(r)
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * gsrc[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_curvature_source_predict(r, Mb, gsrc, params):
    """
    params = [lam, m_inf, sc0, kappa]
    """
    lam, m_inf, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, gsrc, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    v_pred = Vinf * y

    return v_pred, m, S, Rs, Vinf

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vbar2 / G, 0.0)
    Mb = M_enc[-1]

    # curvature-strength source
    g_b = G * M_enc / np.maximum(r**2, EPS)

    # normalize by galaxy-mean g_b over observed grid so source is dimensionless
    g_mean = np.trapz(g_b, r) / max(r[-1] - r[0], EPS)
    gsrc = g_b / max(g_mean, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "gsrc": gsrc,
        "Mb": Mb,
        "Rh": Rh,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _ = mts_curvature_source_predict(
            g["r"], g["Mb"], g["gsrc"],
            [lam, m_inf, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near earlier simple-field solution
p0 = [12.7, 1.0, 3.1, -0.35]
bounds_lo = [0.01, 0.001, 0.01, -1.5]
bounds_hi = [100.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=6000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit = res.x

print("Fitted curvature-source field parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf = mts_curvature_source_predict(
        g["r"], g["Mb"], g["gsrc"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS CURVATURE-SOURCE FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted curvature-source field parameters
lambda = 6.318469070599025
m_inf  = 7.522191736202073
sc0    = 16.95403133852616
kappa  = -0.42305500011096125

MTS CURVATURE-SOURCE FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               27.20816541757529
Mean RMSE:                 37.277003945311485
Median MAE:                20.736963804212387
Median signed inner resid: -0.40202469289101384
Median signed mid resid:   -0.12266720275944902
Median signed outer resid: -0.14657068717390312
Median low-Vinf RMSE:      13.872055631208848
Median high-Vinf RMSE:     55.97691281864232

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC2366_rotmod.dat           6.30     5.80    0.541    0.017   -0.123    8.891    7.555    0.345
UGC07524_rotmod.dat          7.48     6.01    0.086    0.294   -0.036    9.569    7.514    0.470
NGC0055_rotmod.dat           7.72     4.48   -1.000    0.127    0.001    9.774    7.751

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + CUMULATIVE MEMORY
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# where:
#   V_b^2 = V_gas^2 + (ups_d V_disk)^2 + (ups_b V_bul)^2
#
# Fixed MTS laws from your document:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_local_plus_memory_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa]
    """
    lam, m_inf, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)

    # local curvature + cumulative memory
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_local_plus_memory_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near earlier field solution
p0 = [12.7, 1.0, 3.0, -0.35]
bounds_lo = [0.01, 0.001, 0.01, -1.5]
bounds_hi = [100.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=6000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit = res.x

print("Fitted local+memory bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_local_plus_memory_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS LOCAL + MEMORY FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted local+memory bridge parameters
lambda = 12.058743801112733
m_inf  = 1.8136698069196893
sc0    = 5.769116014395077
kappa  = -0.5420338644899587

MTS LOCAL + MEMORY FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.44028621176028
Mean RMSE:                 29.37555120964001
Median MAE:                17.662914923710424
Median signed inner resid: -0.1583910306794522
Median signed mid resid:   -0.016327554967237163
Median signed outer resid: -0.04264655313704034
Median low-Vinf RMSE:      9.49291545616982
Median high-Vinf RMSE:     41.84928011312053

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC0055_rotmod.dat           2.81     2.40   -0.215    0.042    0.030    9.774    7.751    0.521
UGC07524_rotmod.dat          2.86     2.69   -0.100    0.054   -0.042    9.569    7.514    0.470
IC2574_rotmod.dat            4.37     3.81   -0.219    0.130    0.042    9.298    7.267    0.3

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + MEMORY OVER PROPER RADIAL DISTANCE
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Proper-radial accumulation:
#   S(r) = ∫ m(r) * (r/Rs)^kappa * sqrt(g_rr) dr
#
# Weak-field metric factor (phenomenological GR-inspired proxy):
#   sqrt(g_rr) = 1 + eps_g * V_b(r)^2 / Vref^2
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa, eps_g
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12
VREF2 = 300.0**2  # km^2 / s^2

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness_proper(r, m, Rs, kappa, vb2, eps_g):
    """
    Proper-radial accumulation:
      S = ∫ m * (r/Rs)^kappa * sqrt(g_rr) dr

    Weak-field proxy:
      sqrt(g_rr) = 1 + eps_g * vb2 / VREF2
    """
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    sqrt_grr = 1.0 + eps_g * np.clip(vb2 / VREF2, 0.0, None)
    sqrt_grr = np.clip(sqrt_grr, 1e-6, None)

    integrand = m * weight * sqrt_grr
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S, sqrt_grr

def mts_local_plus_proper_memory_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa, eps_g]
    """
    lam, m_inf, sc0, kappa, eps_g = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S, sqrt_grr = cumulative_stiffness_proper(r, m, Rs, kappa, vb2, eps_g)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)

    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y, sqrt_grr

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, eps_g = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _ = mts_local_plus_proper_memory_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa, eps_g]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start near previous local+memory bridge
p0 = [12.06, 1.81, 5.77, -0.54, 0.10]
bounds_lo = [0.01, 0.001, 0.01, -1.5, 0.0]
bounds_hi = [100.0, 10.0, 100.0, 1.0, 50.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=7000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, eps_g_fit = res.x

print("Fitted proper-memory bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("eps_g  =", eps_g_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y, sqrt_grr = mts_local_plus_proper_memory_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, eps_g_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Rs_pred": Rs,
        "Vinf_pred": Vinf,
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vinf_pred"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vinf_pred"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vinf_pred"] >= v_split]

print("MTS LOCAL + PROPER-MEMORY FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vinf RMSE:     ", med(low_rmse))
print("Median high-Vinf RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted proper-memory bridge parameters
lambda = 11.037570113789947
m_inf  = 1.6600741729138215
sc0    = 5.280557500554631
kappa  = -0.5420344265394558
eps_g  = 2.032286658208354e-13

MTS LOCAL + PROPER-MEMORY FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.440279798021912
Mean RMSE:                 29.3755517499868
Median MAE:                17.66293222268265
Median signed inner resid: -0.15839103717058534
Median signed mid resid:   -0.01632762663293226
Median signed outer resid: -0.04264674639734224
Median low-Vinf RMSE:      9.492918477679364
Median high-Vinf RMSE:     41.8492617897127

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC0055_rotmod.dat           2.81     2.40   -0.215    0.042    0.030    9.774    7.751    0.521
UGC07524_rotmod.dat          2.86     2.69   -0.100    0.054   -0.042    9.569    7.514    0.470
IC2574_rotmod.dat            4.37     3.81   -0.219    0

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# AUDIT: DOES THE PROPER-MEMORY FACTOR ACTUALLY DO ANYTHING?
# ------------------------------------------------------------
# We keep the previously fitted local+memory bridge parameters fixed,
# then manually vary eps_g and inspect:
#
#   sqrt(g_rr) = 1 + eps_g * V_b^2 / VREF^2
#
# and measure:
#   - median RMSE across galaxies
#   - mean RMSE across galaxies
#   - median change in S_end
#   - median max(sqrt(g_rr))
#
# This checks that the code path is active and that the scaling is sane.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

G = 4.30091e-6
EPS = 1e-12
VREF2 = 300.0**2

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

# ---- best-fit local+memory bridge parameters from previous run ----
LAM0   = 12.058743801112733
MINF0  = 1.8136698069196893
SC00   = 5.769116014395077
KAPPA0 = -0.5420338644899587

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness_plain(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def cumulative_stiffness_proper(r, m, Rs, kappa, vb2, eps_g):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    sqrt_grr = 1.0 + eps_g * np.clip(vb2 / VREF2, 0.0, None)
    sqrt_grr = np.clip(sqrt_grr, 1e-8, None)
    integrand = m * weight * sqrt_grr
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S, sqrt_grr

def predict_bridge(r, Mb, jb, vb2, lam, m_inf, sc0, kappa, eps_g=0.0):
    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)

    if eps_g == 0.0:
        S = cumulative_stiffness_plain(r, m, Rs, kappa)
        sqrt_grr = np.ones_like(r)
    else:
        S, sqrt_grr = cumulative_stiffness_proper(r, m, Rs, kappa, vb2, eps_g)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    vp = np.sqrt(np.maximum(vpred2, 0.0))
    return vp, S, sqrt_grr, Rs, Vinf

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# baseline no-proper-memory solution
# -----------------------------
base_rmse = []
base_Send = {}
for g in galaxies:
    vp, S, sqrt_grr, Rs, Vinf = predict_bridge(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        LAM0, MINF0, SC00, KAPPA0, eps_g=0.0
    )
    rmse = np.sqrt(np.mean((vp - g["v_obs"])**2))
    base_rmse.append(rmse)
    base_Send[g["name"]] = S[-1]

print("BASELINE (eps_g = 0)")
print("Median RMSE:", med(base_rmse))
print("Mean RMSE:  ", np.mean(base_rmse))
print()

# -----------------------------
# audit fixed eps_g values
# -----------------------------
eps_list = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0]

print("AUDIT TABLE")
print("=" * 120)
print(f'{"eps_g":>8} {"medRMSE":>10} {"meanRMSE":>10} {"med_dS_end%":>12} {"med_max_sgrr":>14} {"max_max_sgrr":>14}')

for eps_g in eps_list:
    rmses = []
    dS_pct = []
    max_sgrr = []

    for g in galaxies:
        vp, S, sqrt_grr, Rs, Vinf = predict_bridge(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            LAM0, MINF0, SC00, KAPPA0, eps_g=eps_g
        )
        rmse = np.sqrt(np.mean((vp - g["v_obs"])**2))
        rmses.append(rmse)

        Send0 = base_Send[g["name"]]
        Send1 = S[-1]
        dS_pct.append(100.0 * (Send1 - Send0) / max(abs(Send0), EPS))
        max_sgrr.append(np.max(sqrt_grr))

    print(f'{eps_g:8.3f} {med(rmses):10.3f} {np.mean(rmses):10.3f} {med(dS_pct):12.3f} {med(max_sgrr):14.3f} {np.max(max_sgrr):14.3f}')

print()
print("CHECK A FEW HIGH-MASS GALAXIES DIRECTLY")
print("=" * 120)

targets = [
    "NGC2841_rotmod.dat",
    "UGC11914_rotmod.dat",
    "UGC08699_rotmod.dat",
    "UGC05253_rotmod.dat",
]

for name in targets:
    g = next((x for x in galaxies if x["name"] == name), None)
    if g is None:
        continue

    print(f"\n{name}")
    vp0, S0, sgrr0, Rs0, Vinf0 = predict_bridge(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        LAM0, MINF0, SC00, KAPPA0, eps_g=0.0
    )
    rmse0 = np.sqrt(np.mean((vp0 - g["v_obs"])**2))
    print(f"  baseline RMSE   = {rmse0:.3f}")
    print(f"  baseline S_end  = {S0[-1]:.6f}")

    for eps_g in [0.1, 0.5, 1.0, 2.0, 5.0]:
        vp, S, sgrr, Rs, Vinf = predict_bridge(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            LAM0, MINF0, SC00, KAPPA0, eps_g=eps_g
        )
        rmse = np.sqrt(np.mean((vp - g["v_obs"])**2))
        dS = 100.0 * (S[-1] - S0[-1]) / max(abs(S0[-1]), EPS)
        print(f"  eps_g={eps_g:>4.1f} | RMSE={rmse:8.3f} | dS_end={dS:8.3f}% | max(sqrt_grr)={np.max(sgrr):8.3f}")

Galaxies loaded: 60

BASELINE (eps_g = 0)
Median RMSE: 19.44028621176028
Mean RMSE:   29.37555120964

AUDIT TABLE
   eps_g    medRMSE   meanRMSE  med_dS_end%   med_max_sgrr   max_max_sgrr
   0.010     19.449     29.381        0.065          1.001          1.008
   0.050     19.485     29.401        0.325          1.006          1.038
   0.100     19.529     29.426        0.650          1.011          1.077
   0.250     19.672     29.493        1.626          1.029          1.192
   0.500     20.565     29.590        3.252          1.057          1.385
   1.000     20.309     29.735        6.503          1.114          1.770
   2.000     20.648     29.912       13.007          1.228          2.539
   5.000     21.255     30.098       32.517          1.570          4.848
  10.000     22.550     30.100       65.035          2.141          8.695

CHECK A FEW HIGH-MASS GALAXIES DIRECTLY

NGC2841_rotmod.dat
  baseline RMSE   = 92.031
  baseline S_end  = 55.890826
  eps_g= 0.1 | RMSE=  91.337

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + MASS-COUPLED MEMORY
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Memory coupling:
#   eta(M) = (M / M0)^delta
#
# Response:
#   V_pred^2(r) = V_b^2(r) + eta(M) * Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa, delta
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12
M0 = 1e10  # Msun

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_local_plus_mass_memory_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa, delta]
    """
    lam, m_inf, sc0, kappa, delta = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    eta = np.power(np.clip(Mb / M0, EPS, None), delta)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)

    vpred2 = vb2 + eta * (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y, eta

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, delta = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _ = mts_local_plus_mass_memory_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa, delta]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near previous local+memory bridge, with neutral delta=0
p0 = [12.058743801112733, 1.8136698069196893, 5.769116014395077, -0.5420338644899587, 0.0]
bounds_lo = [0.01, 0.001, 0.01, -1.5, -2.0]
bounds_hi = [100.0, 10.0, 100.0, 1.0, 2.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=8000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, delta_fit = res.x

print("Fitted local+mass-memory bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("delta  =", delta_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y, eta = mts_local_plus_mass_memory_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, delta_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "eta": eta,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
eta_list   = [r["eta"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS LOCAL + MASS-MEMORY FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median eta:               ", med(eta_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"eta":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["eta"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"eta":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["eta"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))
print("eta:               ", corr([r["eta"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted local+mass-memory bridge parameters
lambda = 12.358827539687914
m_inf  = 2.1398936546100247
sc0    = 6.6411341321299195
kappa  = -0.5529087069134844
delta  = -0.019232116506824506

MTS LOCAL + MASS-MEMORY FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               18.62908143039937
Mean RMSE:                 29.153288213062645
Median MAE:                17.691446175811656
Median signed inner resid: -0.15866650424621856
Median signed mid resid:   -0.02326208385626723
Median signed outer resid: -0.04820081595903321
Median eta:                0.9836805806739204
Median low-Vmax RMSE:      9.703817376247505
Median high-Vmax RMSE:     39.63839979958854

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out      eta     logM   logSig    bulge
UGC07524_rotmod.dat          2.69     2.51   -0.086    0.068   -0.036    1.019    9.569    7.514    0.470
NGC0055_rotmod.dat           2.78     2.36   -0.215    0.041    0.030    1.010    9

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + THRESHOLD-ACTIVATED MEMORY SOURCE
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * A(r) * j_b(r)
#
# Activation:
#   A(r) = 1 / (1 + (Sigma_loc(r) / Sigma0)^n_act)
#
# where Sigma_loc(r) = M_enc(r) / r^2
#
# Interpretation:
#   - dense inner regions: A small  -> local curvature dominates
#   - diffuse outer regions: A ~ 1  -> memory turns on
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa, Sigma0, n_act
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, src, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * src(r)
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * src[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def activation_sigma(M_enc, r, Sigma0, n_act):
    Sigma_loc = M_enc / np.maximum(r**2, EPS)
    A = 1.0 / (1.0 + np.power(np.clip(Sigma_loc / max(Sigma0, EPS), EPS, None), n_act))
    return np.clip(A, 0.0, 1.0), Sigma_loc

def mts_activated_memory_predict(r, Mb, M_enc, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa, Sigma0, n_act]
    """
    lam, m_inf, sc0, kappa, Sigma0, n_act = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    Aact, Sigma_loc = activation_sigma(M_enc, r, Sigma0, n_act)
    src = Aact * jb

    m = solve_m_field(r, src, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y, Aact, Sigma_loc

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "M_enc": M_enc,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
Sigma0_guess = np.median([g["Sigma_star"] for g in galaxies])
print("Sigma0 initial guess =", Sigma0_guess)
print("log10 Sigma0 guess   =", np.log10(Sigma0_guess))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, Sigma0, n_act = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _, _ = mts_activated_memory_predict(
            g["r"], g["Mb"], g["M_enc"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa, Sigma0, n_act]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near best local+memory bridge, with moderate activation
p0 = [12.06, 1.81, 5.77, -0.54, Sigma0_guess, 1.0]
bounds_lo = [0.01, 0.001, 0.01, -1.5, Sigma0_guess/1000.0, 0.1]
bounds_hi = [100.0, 10.0, 100.0, 1.0, Sigma0_guess*1000.0, 8.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=12000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, Sigma0_fit, n_act_fit = res.x

print("Fitted activated-memory bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("Sigma0 =", Sigma0_fit)
print("log10 Sigma0 =", np.log10(Sigma0_fit))
print("n_act  =", n_act_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y, Aact, Sigma_loc = mts_activated_memory_predict(
        g["r"], g["Mb"], g["M_enc"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, Sigma0_fit, n_act_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "A_mid": np.median(Aact),
        "A_in": np.median(Aact[x < 0.7]) if np.any(x < 0.7) else np.nan,
        "A_out": np.median(Aact[x > 2.0]) if np.any(x > 2.0) else np.nan,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
A_mid_list = [r["A_mid"] for r in rows]

v_split = np.median([g["Vmax_obs"] for g in galaxies])
low_rmse  = [r["rmse"] for r in rows if next(g["Vmax_obs"] for g in galaxies if g["name"] == r["name"]) < v_split]
high_rmse = [r["rmse"] for r in rows if next(g["Vmax_obs"] for g in galaxies if g["name"] == r["name"]) >= v_split]

print("MTS ACTIVATED-MEMORY FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median activation A_mid:  ", med(A_mid_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"A_in":>8} {"A_mid":>8} {"A_out":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["A_in"]:8.3f} {row["A_mid"]:8.3f} {row["A_out"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"A_in":>8} {"A_mid":>8} {"A_out":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["A_in"]:8.3f} {row["A_mid"]:8.3f} {row["A_out"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))
print("A_mid:             ", corr([r["A_mid"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60
Sigma0 initial guess = 319126537.4187316
log10 Sigma0 guess   = 8.503962920067265

Fitted activated-memory bridge parameters
lambda = 14.334599435533793
m_inf  = 1.5586584060167143
sc0    = 4.929408810952818
kappa  = -0.5487516355131662
Sigma0 = 319126536815.00195
log10 Sigma0 = 11.503962919245657
n_act  = 0.10000000000000002

MTS ACTIVATED-MEMORY FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.379970790602677
Mean RMSE:                 29.359629648557835
Median MAE:                18.150707619280624
Median signed inner resid: -0.15889314406497224
Median signed mid resid:   -0.018865308150220944
Median signed outer resid: -0.042229023042683084
Median activation A_mid:   0.6792718788702523
Median low-Vmax RMSE:      9.812154266359897
Median high-Vmax RMSE:     41.3997239567451

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     A_in    A_mid    A_out
NGC0055_rotmod.dat           2.77     2.36   -0.215    0.04

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + BACKGROUND-DEPENDENT PROPAGATION
# ------------------------------------------------------------
# Field equation:
#
#   (1/r^2) d/dr [ r^2 D(r) dm/dr ] - (m - m_inf) = -lambda * j_b(r)
#
# with
#
#   D(r) = (r / Rs)^q
#
# so q controls how propagation into the background changes with radius.
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa, q_prop
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field_variable_D(r, jb, Rs, m_inf, lam, q_prop):
    """
    Solve:
      (1/r^2) d/dr [ r^2 D(r) dm/dr ] - (m - m_inf) = -lambda * jb(r)

    with
      D(r) = (r / Rs)^q_prop

    BCs:
      m'(0)=0
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # build diffusion flux coefficient F = r^2 D
    D = np.power(np.clip(r / max(Rs, EPS), EPS, None), q_prop)
    F = r**2 * D

    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]

        r_imh = 0.5 * (r[i] + r[i - 1])
        r_iph = 0.5 * (r[i] + r[i + 1])

        # interface coefficients
        F_imh = np.interp(r_imh, r, F)
        F_iph = np.interp(r_iph, r, F)

        # discretize (1/r^2) d/dr [ F dm/dr ]
        coef_L = F_imh / (hL * (0.5 * (hL + hR)))
        coef_R = F_iph / (hR * (0.5 * (hL + hR)))
        coef_C = -(coef_L + coef_R)

        rr = max(r[i]**2, EPS)

        A[i, i - 1] = coef_L / rr
        A[i, i]     = coef_C / rr - 1.0
        A[i, i + 1] = coef_R / rr
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_variable_propagation_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa, q_prop]
    """
    lam, m_inf, sc0, kappa, q_prop = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field_variable_D(r, jb, Rs, m_inf, lam, q_prop)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, q_prop = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_variable_propagation_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa, q_prop]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start from best current local+memory bridge with neutral propagation q=0
p0 = [12.06, 1.81, 5.77, -0.54, 0.0]
bounds_lo = [0.01, 0.001, 0.01, -1.5, -3.0]
bounds_hi = [100.0, 10.0, 100.0, 1.0, 3.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=12000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, q_prop_fit = res.x

print("Fitted variable-propagation bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("q_prop =", q_prop_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_variable_propagation_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, q_prop_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS VARIABLE-PROPAGATION FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted variable-propagation bridge parameters
lambda = 13.42178745855261
m_inf  = 3.5585960933329868
sc0    = 9.590701689451462
kappa  = -0.5154667610440941
q_prop = -2.999999998926963

MTS VARIABLE-PROPAGATION FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.95316562550191
Mean RMSE:                 29.409297685273824
Median MAE:                17.112387855464505
Median signed inner resid: -0.1555182397393193
Median signed mid resid:   -0.015609150146259094
Median signed outer resid: -0.03727821890423412
Median low-Vmax RMSE:      9.760993953530962
Median high-Vmax RMSE:     42.368844915806235

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGC07524_rotmod.dat          2.63     2.47   -0.103    0.054   -0.037    9.569    7.514    0.470
NGC0055_rotmod.dat           2.89     2.42   -0.215    0.027    0.026    9.774    7.751    0.521
IC2574_rotmod.dat            4.28     3.72   -0.22

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + STRETCHED MEMORY RESPONSE
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b(r)
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Stretched response:
#   y(r) = 1 - exp( - (S/Sc)^nu )
#
# Rotation law:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * y(r)^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa, nu
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field(r, jb, Rs, m_inf, lam):
    """
    Solve:
        Rs^2 m'' - (m - m_inf) = -lam * jb
    BCs:
        m'(0)=0
        m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_stretched_response_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa, nu]
    """
    lam, m_inf, sc0, kappa, nu = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field(r, jb, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    z = np.power(np.clip(S / Sc, 0.0, None), nu)
    y = 1.0 - np.exp(-z)

    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa, nu = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_stretched_response_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa, nu]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start from best local+memory bridge, with near-mastercurve nu
p0 = [12.06, 1.81, 5.77, -0.54, 0.94]
bounds_lo = [0.01, 0.001, 0.01, -1.5, 0.2]
bounds_hi = [100.0, 10.0, 100.0, 1.0, 3.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=12000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit, nu_fit = res.x

print("Fitted stretched-response bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print("nu     =", nu_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_stretched_response_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit, nu_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS STRETCHED-RESPONSE FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted stretched-response bridge parameters
lambda = 8.518511058042526
m_inf  = 1.2142022963370211
sc0    = 4.326735341501675
kappa  = -0.6002363318836414
nu     = 1.3022231661720765

MTS STRETCHED-RESPONSE FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               21.474081894035486
Mean RMSE:                 29.715313349416085
Median MAE:                17.810351428318093
Median signed inner resid: -0.19051543453584396
Median signed mid resid:   -0.0070651659507793806
Median signed outer resid: -0.023377035464263902
Median low-Vmax RMSE:      9.408363862721357
Median high-Vmax RMSE:     43.67843191736698

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGC07524_rotmod.dat          2.23     1.69   -0.191   -0.003   -0.015    9.569    7.514    0.470
IC2574_rotmod.dat            3.47     2.85   -0.292   -0.017    0.032    9.298    7.267    0.356
F583-1_rotmod.dat            3.85     3.19    0.299

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# INVERSE RECONSTRUCTION OF S_data(r) AND m_data(r)
# ------------------------------------------------------------
# Uses current best bridge:
#
#   V_obs^2 = V_b^2 + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Invert to get:
#
#   y_mem = sqrt(V_obs^2 - V_b^2) / Vinf
#   S_data = -Sc * ln(1 - y_mem)
#
# Then define weighted coordinate:
#
#   du = (r/Rs)^kappa dr
#
# and reconstruct:
#
#   m_data = dS_data / du
#
# This lets us inspect what the data is demanding directly.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- best current local+memory bridge parameters -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

LAM0   = 12.058743801112733
MINF0  = 1.8136698069196893
SC00   = 5.769116014395077
KAPPA0 = -0.5420338644899587

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def gradient_safe(y, x):
    y = np.asarray(y, float)
    x = np.asarray(x, float)
    if len(y) < 3:
        return np.full_like(y, np.nan)
    g = np.gradient(y, x)
    return g

# -----------------------------
# load galaxies and reconstruct
# -----------------------------
gal_summaries = []
all_rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]
    M9 = Mb / 1e9

    Rs   = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)
    Sc   = SC00 * (Rs ** (1.0 + KAPPA0))

    # residual memory contribution from data
    vmem2 = np.maximum(v_obs**2 - vb2, 0.0)
    ymem  = np.sqrt(vmem2) / max(Vinf, EPS)

    # avoid log singularity near 1
    ymem_clip = np.clip(ymem, 0.0, 0.999999)
    S_data = -Sc * np.log(np.maximum(1.0 - ymem_clip, EPS))

    # weighted path coordinate u
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), KAPPA0)
    u = np.zeros_like(r)
    for i in range(1, len(r)):
        u[i] = u[i-1] + 0.5 * (weight[i] + weight[i-1]) * (r[i] - r[i-1])

    m_data = gradient_safe(S_data, u)

    # source-like local quantities to compare against
    g_b = vb2 / np.maximum(r, EPS)
    sigma_loc = M_enc / np.maximum(r**2, EPS)
    dMdr = np.clip(gradient_safe(M_enc, r), 0.0, None)
    dlnM_dlnr = (r / np.maximum(M_enc, EPS)) * dMdr

    # keep finite interior points only
    keep = (
        np.isfinite(r) &
        np.isfinite(S_data) &
        np.isfinite(m_data) &
        np.isfinite(g_b) &
        np.isfinite(sigma_loc) &
        np.isfinite(dMdr) &
        (m_data > 0)
    )

    if np.sum(keep) < 8:
        continue

    r_k = r[keep]
    x_k = r_k / max(Rs, EPS)
    S_k = S_data[keep]
    m_k = m_data[keep]
    gb_k = g_b[keep]
    sig_k = sigma_loc[keep]
    dMdr_k = dMdr[keep]
    dln_k = dlnM_dlnr[keep]

    # per-galaxy correlations in log space
    logm   = np.log10(np.maximum(m_k, EPS))
    loggb  = np.log10(np.maximum(gb_k, EPS))
    logsig = np.log10(np.maximum(sig_k, EPS))
    logdM  = np.log10(np.maximum(dMdr_k, EPS))
    logx   = np.log10(np.maximum(x_k, EPS))

    c_gb   = corr(logm, loggb)
    c_sig  = corr(logm, logsig)
    c_dM   = corr(logm, logdM)
    c_x    = corr(logm, logx)
    c_dln  = corr(logm, dln_k)

    gal_summaries.append({
        "name": os.path.basename(path),
        "logM": np.log10(Mb),
        "Rs": Rs,
        "Vinf": Vinf,
        "Sc": Sc,
        "corr_logm_loggb": c_gb,
        "corr_logm_logsig": c_sig,
        "corr_logm_logdMdr": c_dM,
        "corr_logm_logx": c_x,
        "corr_logm_dlnM": c_dln,
        "m_med": med(m_k),
        "S_end": S_k[-1],
        "npts": len(m_k),
    })

    # pooled rows
    for i in range(len(r_k)):
        all_rows.append({
            "name": os.path.basename(path),
            "x": x_k[i],
            "logx": np.log10(max(x_k[i], EPS)),
            "logm": np.log10(max(m_k[i], EPS)),
            "loggb": np.log10(max(gb_k[i], EPS)),
            "logsig": np.log10(max(sig_k[i], EPS)),
            "logdMdr": np.log10(max(dMdr_k[i], EPS)),
            "dlnM_dlnr": dln_k[i],
            "logM_gal": np.log10(Mb),
        })

print("Galaxies reconstructed:", len(gal_summaries))
print("Total pooled radial points:", len(all_rows))
print()

# -----------------------------
# pooled correlations
# -----------------------------
logm_all   = np.array([r["logm"] for r in all_rows], float)
loggb_all  = np.array([r["loggb"] for r in all_rows], float)
logsig_all = np.array([r["logsig"] for r in all_rows], float)
logdM_all  = np.array([r["logdMdr"] for r in all_rows], float)
logx_all   = np.array([r["logx"] for r in all_rows], float)
dln_all    = np.array([r["dlnM_dlnr"] for r in all_rows], float)

print("POOLED CORRELATIONS: reconstructed m_data against candidate source variables")
print("=" * 120)
print("corr(log m_data, log g_b)         =", corr(logm_all, loggb_all))
print("corr(log m_data, log Sigma_loc)   =", corr(logm_all, logsig_all))
print("corr(log m_data, log dM/dr)       =", corr(logm_all, logdM_all))
print("corr(log m_data, log x=r/Rs)      =", corr(logm_all, logx_all))
print("corr(log m_data, dlnM/dlnr)       =", corr(logm_all, dln_all))
print()

# -----------------------------
# median per-galaxy correlations
# -----------------------------
print("MEDIAN PER-GALAXY CORRELATIONS")
print("=" * 120)
print("median corr(log m, log g_b)       =", med([g["corr_logm_loggb"] for g in gal_summaries]))
print("median corr(log m, log Sigma_loc) =", med([g["corr_logm_logsig"] for g in gal_summaries]))
print("median corr(log m, log dM/dr)     =", med([g["corr_logm_logdMdr"] for g in gal_summaries]))
print("median corr(log m, log x)         =", med([g["corr_logm_logx"] for g in gal_summaries]))
print("median corr(log m, dlnM/dlnr)     =", med([g["corr_logm_dlnM"] for g in gal_summaries]))
print()

# -----------------------------
# strongest / weakest galaxies by a chosen diagnostic
# -----------------------------
gal_summaries = sorted(gal_summaries, key=lambda z: (-np.nan_to_num(z["corr_logm_logsig"], nan=-999)))

print("TOP 12 GALAXIES BY corr(log m_data, log Sigma_loc)")
print("=" * 120)
print(f'{"name":<24} {"corr_gb":>9} {"corr_sig":>9} {"corr_dM":>9} {"corr_x":>9} {"corr_dlnM":>11} {"logM":>8} {"m_med":>10} {"S_end":>10}')
for g in gal_summaries[:12]:
    print(f'{g["name"]:<24} {g["corr_logm_loggb"]:9.3f} {g["corr_logm_logsig"]:9.3f} {g["corr_logm_logdMdr"]:9.3f} {g["corr_logm_logx"]:9.3f} {g["corr_logm_dlnM"]:11.3f} {g["logM"]:8.3f} {g["m_med"]:10.3f} {g["S_end"]:10.3f}')
print()

print("BOTTOM 12 GALAXIES BY corr(log m_data, log Sigma_loc)")
print("=" * 120)
print(f'{"name":<24} {"corr_gb":>9} {"corr_sig":>9} {"corr_dM":>9} {"corr_x":>9} {"corr_dlnM":>11} {"logM":>8} {"m_med":>10} {"S_end":>10}')
for g in gal_summaries[-12:]:
    print(f'{g["name"]:<24} {g["corr_logm_loggb"]:9.3f} {g["corr_logm_logsig"]:9.3f} {g["corr_logm_logdMdr"]:9.3f} {g["corr_logm_logx"]:9.3f} {g["corr_logm_dlnM"]:11.3f} {g["logM"]:8.3f} {g["m_med"]:10.3f} {g["S_end"]:10.3f}')
print()

# -----------------------------
# crude radial-bin trend for pooled m_data
# -----------------------------
xbins = np.array([0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 20.0])
x_all = np.array([r["x"] for r in all_rows], float)

print("POOLED RADIAL TREND OF reconstructed m_data")
print("=" * 120)
print(f'{"x_bin":<16} {"median logm":>12} {"median loggb":>12} {"median logsig":>12} {"n":>8}')
for lo, hi in zip(xbins[:-1], xbins[1:]):
    mask = (x_all >= lo) & (x_all < hi)
    if np.sum(mask) == 0:
        continue
    print(f'[{lo:>4.1f},{hi:<4.1f}) {med(logm_all[mask]):12.3f} {med(loggb_all[mask]):12.3f} {med(logsig_all[mask]):12.3f} {np.sum(mask):8d}')

Galaxies reconstructed: 58
Total pooled radial points: 1187

POOLED CORRELATIONS: reconstructed m_data against candidate source variables
corr(log m_data, log g_b)         = -0.0026343807450535596
corr(log m_data, log Sigma_loc)   = -0.0026343807450536225
corr(log m_data, log dM/dr)       = 0.03449921802573912
corr(log m_data, log x=r/Rs)      = -0.09820473184904369
corr(log m_data, dlnM/dlnr)       = 0.061311970262048605

MEDIAN PER-GALAXY CORRELATIONS
median corr(log m, log g_b)       = 0.2746317397315704
median corr(log m, log Sigma_loc) = 0.27463173973156996
median corr(log m, log dM/dr)     = 0.16498414957052784
median corr(log m, log x)         = -0.17811630678716553
median corr(log m, dlnM/dlnr)     = 0.007291605929285207

TOP 12 GALAXIES BY corr(log m_data, log Sigma_loc)
name                       corr_gb  corr_sig   corr_dM    corr_x   corr_dlnM     logM      m_med      S_end
UGC05721_rotmod.dat          0.949     0.949     0.367    -0.854       0.450    8.952      7.798     

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + MEMORY FIELD WITH COMPACTNESS SOURCE
# ------------------------------------------------------------
# Field equation:
#
#   Rs^2 m'' - (m - m_inf) = -lam1 * j_b(r) - lam2 * C(r)
#
# where:
#
#   j_b(r) = normalized dM/dr
#
#   C(r) = normalized positive compactness gradient
#        = normalized max(0, d(M/r)/dr)
#
# Motivation:
#   - inverse reconstruction suggests m_data is mostly geometric in shape
#   - remaining error is in compact inner bulge regions
#   - add a second source sector that responds to central compactness
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lam1, lam2, m_inf, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field_two_source(r, src1, src2, Rs, m_inf, lam1, lam2):
    """
    Solve:
      Rs^2 m'' - (m - m_inf) = -lam1 * src1(r) - lam2 * src2(r)

    BCs:
      m'(0)=0
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam1 * src1[i] - lam2 * src2[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_compact_source_predict(r, Mb, src1, src2, vb2, params):
    """
    params = [lam1, lam2, m_inf, sc0, kappa]
    """
    lam1, lam2, m_inf, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field_two_source(r, src1, src2, Rs, m_inf, lam1, lam2)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    # source 1: normalized dM/dr
    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm1 = np.trapz(dMdr, r)
    src1 = dMdr / max(norm1, EPS)

    # source 2: normalized positive d(M/r)/dr
    compact_raw = np.gradient(M_enc / np.maximum(r, EPS), r)
    compact_raw = np.clip(compact_raw, 0.0, None)
    norm2 = np.trapz(compact_raw, r)
    if norm2 <= EPS:
        src2 = np.zeros_like(r)
    else:
        src2 = compact_raw / norm2

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "Mb": Mb,
        "src1": src1,
        "src2": src2,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam1, lam2, m_inf, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_compact_source_predict(
            g["r"], g["Mb"], g["src1"], g["src2"], g["vb2"],
            [lam1, lam2, m_inf, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start from best local+memory bridge + small compactness source
p0 = [12.06, 1.0, 1.81, 5.77, -0.54]
bounds_lo = [0.01, 0.0, 0.001, 0.01, -1.5]
bounds_hi = [100.0, 100.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=12000)
lam1_fit, lam2_fit, m_inf_fit, sc0_fit, kappa_fit = res.x

print("Fitted compact-source bridge parameters")
print("lam1   =", lam1_fit)
print("lam2   =", lam2_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_compact_source_predict(
        g["r"], g["Mb"], g["src1"], g["src2"], g["vb2"],
        [lam1_fit, lam2_fit, m_inf_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS COMPACT-SOURCE FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted compact-source bridge parameters
lam1   = 16.165551730461154
lam2   = 2.1184896910164464e-11
m_inf  = 2.4313328551148516
sc0    = 7.733922255604128
kappa  = -0.5420367915566814

MTS COMPACT-SOURCE FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.440243675334855
Mean RMSE:                 29.37555956828704
Median MAE:                17.663071300963217
Median signed inner resid: -0.15839109038160887
Median signed mid resid:   -0.01632817478404876
Median signed outer resid: -0.042647410193624696
Median low-Vmax RMSE:      9.800482368857939
Median high-Vmax RMSE:     41.84917567978043

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC0055_rotmod.dat           2.81     2.40   -0.215    0.042    0.030    9.774    7.751    0.521
UGC07524_rotmod.dat          2.86     2.69   -0.100    0.054   -0.042    9.569    7.514    0.470
IC2574_rotmod.dat            4.37     3.81   -0.219    0.

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + RADIAL-LAPLACIAN MEMORY FIELD
# ------------------------------------------------------------
# Field equation:
#
#   Rs^2 [ m'' + (1/r) m' ] - (m - m_inf) = -lambda * j_b(r)
#
# This replaces the previous slab-like operator:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_b
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field_radial(r, jb, Rs, m_inf, lam):
    """
    Solve:
      Rs^2 [ m'' + (1/r) m' ] - (m - m_inf) = -lam * jb(r)

    BCs:
      m'(0)=0   -> implemented as m0 = m1
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]

        # nonuniform second derivative
        a2 =  2.0 / (hL * (hL + hR))
        b2 = -2.0 / (hL * hR)
        c2 =  2.0 / (hR * (hL + hR))

        # nonuniform first derivative
        a1 = -hR / (hL * (hL + hR))
        b1 = (hR - hL) / (hL * hR)
        c1 = hL / (hR * (hL + hR))

        rr = max(r[i], EPS)

        A[i, i - 1] = Rs**2 * (a2 + (1.0 / rr) * a1)
        A[i, i]     = Rs**2 * (b2 + (1.0 / rr) * b1) - 1.0
        A[i, i + 1] = Rs**2 * (c2 + (1.0 / rr) * c1)
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S

def mts_radial_operator_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, sc0, kappa]
    """
    lam, m_inf, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field_radial(r, jb, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_radial_operator_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# start from best bridge-ish values
p0 = [12.06, 1.81, 5.77, -0.54]
bounds_lo = [0.01, 0.001, 0.01, -1.5]
bounds_hi = [100.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=12000)
lam_fit, m_inf_fit, sc0_fit, kappa_fit = res.x

print("Fitted radial-operator bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_radial_operator_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS RADIAL-OPERATOR FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted radial-operator bridge parameters
lambda = 12.197062560138608
m_inf  = 2.2359586705399317
sc0    = 6.6513389250972
kappa  = -0.5324949205260121

MTS RADIAL-OPERATOR FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.568957238860342
Mean RMSE:                 29.37603052451636
Median MAE:                18.168817643899093
Median signed inner resid: -0.16052816880368895
Median signed mid resid:   -0.023393311035198955
Median signed outer resid: -0.04072796919396134
Median low-Vmax RMSE:      9.996891332549854
Median high-Vmax RMSE:     41.01745993344869

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
UGC07524_rotmod.dat          2.67     2.43   -0.073    0.088   -0.030    9.569    7.514    0.470
NGC0055_rotmod.dat           2.83     2.42   -0.215    0.043    0.031    9.774    7.751    0.521
DDO161_rotmod.dat            4.58     3.84   -0.142    0.187   -0.018    9.254    7.510  

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + TWO-SCALE MEMORY FIELD
# ------------------------------------------------------------
# Field equations:
#
#   Rs^2 m_s'' - (m_s - m_inf_s)           = -lam_s * j_b(r)
#   (alpha*Rs)^2 m_l'' - (m_l - m_inf_l)   = -lam_l * j_b(r)
#
# Total field:
#   m(r) = m_s(r) + m_l(r)
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lam_s, lam_l, alpha, m_inf_s, m_inf_l, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_field(r, jb, Lscale, m_inf, lam):
    """
    Solve:
      Lscale^2 m'' - (m - m_inf) = -lam * jb(r)

    BCs:
      m'(0)=0   -> implemented as m0 = m1
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Lscale**2) * a
        A[i, i]     = (Lscale**2) * d - 1.0
        A[i, i + 1] = (Lscale**2) * c
        b[i] = -m_inf - lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S

def mts_two_scale_predict(r, Mb, jb, vb2, params):
    """
    params = [lam_s, lam_l, alpha, m_inf_s, m_inf_l, sc0, kappa]
    """
    lam_s, lam_l, alpha, m_inf_s, m_inf_l, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    Ls = Rs
    Ll = alpha * Rs

    m_s = solve_screened_field(r, jb, Ls, m_inf_s, lam_s)
    m_l = solve_screened_field(r, jb, Ll, m_inf_l, lam_l)
    m_tot = m_s + m_l

    S = cumulative_stiffness(r, m_tot, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m_s, m_l, m_tot, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam_s, lam_l, alpha, m_inf_s, m_inf_l, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _, _ = mts_two_scale_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam_s, lam_l, alpha, m_inf_s, m_inf_l, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near best single-scale bridge, with modest long-scale component
p0 = [8.0, 4.0, 3.0, 1.0, 0.8, 5.77, -0.54]
bounds_lo = [0.0, 0.0, 1.1, 0.0, 0.0, 0.01, -1.5]
bounds_hi = [100.0, 100.0, 20.0, 10.0, 10.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=15000)
lam_s_fit, lam_l_fit, alpha_fit, m_inf_s_fit, m_inf_l_fit, sc0_fit, kappa_fit = res.x

print("Fitted two-scale bridge parameters")
print("lam_s   =", lam_s_fit)
print("lam_l   =", lam_l_fit)
print("alpha   =", alpha_fit)
print("m_inf_s =", m_inf_s_fit)
print("m_inf_l =", m_inf_l_fit)
print("sc0     =", sc0_fit)
print("kappa   =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, m_s, m_l, m_tot, S, Rs, Vinf, y = mts_two_scale_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_s_fit, lam_l_fit, alpha_fit, m_inf_s_fit, m_inf_l_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS TWO-SCALE FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted two-scale bridge parameters
lam_s   = 2.1754515764466964
lam_l   = 92.38420188825003
alpha   = 19.99999998274852
m_inf_s = 0.12979548154729964
m_inf_l = 0.05349499301004374
sc0     = 1.4296885349639805
kappa   = -0.563201154434566

MTS TWO-SCALE FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               21.129022560796624
Mean RMSE:                 29.152838280276036
Median MAE:                18.21154010519644
Median signed inner resid: -0.15919967586286465
Median signed mid resid:   -0.014990107983076005
Median signed outer resid: -0.056188941027657854
Median low-Vmax RMSE:      10.893490048134007
Median high-Vmax RMSE:     38.45648273755627

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC0055_rotmod.dat           2.63     1.98   -0.215   -0.000   -0.018    9.774    7.751    0.521
IC2574_rotmod.dat            4.28     3.51   -0.231    0.075   -0.008    9.298    7.267    0.356
NGC236

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL FIELD + BACKGROUND MEMORY FLOOR
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m_loc'' - (m_loc - m_inf) = -lambda * j_b(r)
#
# Background memory floor:
#   m0(M) = A0 * (M / M0)^beta
#
# Total memory density:
#   m_tot(r) = m_loc(r) + m0(M)
#
# Cumulative stiffness:
#   S(r) = ∫ m_tot(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, A0, beta, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12
M0 = 1e10  # Msun

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_field(r, jb, Lscale, m_inf, lam):
    """
    Solve:
      Lscale^2 m'' - (m - m_inf) = -lam * jb(r)

    BCs:
      m'(0)=0  -> m0 = m1
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Lscale**2) * a
        A[i, i]     = (Lscale**2) * d - 1.0
        A[i, i + 1] = (Lscale**2) * c
        b[i] = -m_inf - lam * jb[i]

    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m_tot, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m_tot * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S

def mts_background_floor_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, m_inf, A0, beta, sc0, kappa]
    """
    lam, m_inf, A0, beta, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m_loc = solve_screened_field(r, jb, Rs, m_inf, lam)

    m0 = A0 * np.power(np.clip(Mb / M0, EPS, None), beta)
    m_tot = m_loc + m0

    S = cumulative_stiffness(r, m_tot, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m_loc, m0, m_tot, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, A0, beta, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _, _ = mts_background_floor_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, m_inf, A0, beta, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near best single-scale bridge with a modest floor
p0 = [12.06, 1.81, 0.05, 0.0, 5.77, -0.54]
bounds_lo = [0.01, 0.001, 0.0, -2.0, 0.01, -1.5]
bounds_hi = [100.0, 10.0, 10.0, 2.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=15000)
lam_fit, m_inf_fit, A0_fit, beta_fit, sc0_fit, kappa_fit = res.x

print("Fitted background-floor bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("A0     =", A0_fit)
print("beta   =", beta_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, m_loc, m0, m_tot, S, Rs, Vinf, y = mts_background_floor_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, m_inf_fit, A0_fit, beta_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "m0": m0,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
m0_list    = [r["m0"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS BACKGROUND-FLOOR FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median m0:                ", med(m0_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"m0":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["m0"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"m0":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["m0"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))
print("m0:                ", corr([r["m0"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted background-floor bridge parameters
lambda = 94.53455519945787
m_inf  = 0.0010000000000000002
A0     = 9.999025423939049
beta   = 0.21207563358970036
sc0    = 28.244827519115294
kappa  = -0.40297787546436936

MTS BACKGROUND-FLOOR FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               20.531497790194578
Mean RMSE:                 29.304809336611648
Median MAE:                17.36300657414614
Median signed inner resid: -0.1722550057174611
Median signed mid resid:   -0.005621143524034603
Median signed outer resid: -0.032962999087010565
Median m0:                 11.988461030076383
Median low-Vmax RMSE:      8.83509584007026
Median high-Vmax RMSE:     46.2217504634657

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out       m0     logM   logSig    bulge
UGC07524_rotmod.dat          2.68     2.34   -0.172   -0.017   -0.037    8.099    9.569    7.514    0.470
NGC2366_rotmod.dat           3.95     3.11    0.448    0.051

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# DIRECT PDE DIAGNOSTIC ON RECONSTRUCTED FIELD
# ------------------------------------------------------------
# Uses best current local+memory bridge:
#
#   V_obs^2 = V_b^2 + Vinf^2 * [1 - exp(-S/Sc)]^2
#
# invert to get S_data, then
#
#   du = (r/Rs)^kappa dr
#   m_data = dS_data / du
#
# Then compute cylindrical radial Laplacian:
#
#   Lap_r m = m'' + (1/r) m'
#
# and test whether:
#
#   Lap_r m  ~  a * j_b - b * m
#
# This is the cleanest check of whether the reconstructed field
# behaves like a screened transport equation.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- best current local+memory bridge parameters -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866
SC00     = 5.769116014395077
KAPPA0   = -0.5420338644899587

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def deriv1_nonuniform(x, y):
    return np.gradient(y, x)

def deriv2_nonuniform(x, y):
    d1 = np.gradient(y, x)
    d2 = np.gradient(d1, x)
    return d2

def fit_ab(jb, m, lap):
    """
    Fit lap ≈ a*jb - b*m
    i.e. lap = [jb, -m] @ [a, b]
    """
    X = np.column_stack([jb, -m])
    y = lap
    mask = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    X = X[mask]
    y = y[mask]
    if len(y) < 8:
        return np.nan, np.nan, np.nan, np.nan
    coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    a_fit, b_fit = coef
    yhat = X @ coef
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return a_fit, b_fit, r2, len(y)

# -----------------------------
# reconstruct and test
# -----------------------------
gal_summaries = []
pooled_jb = []
pooled_m = []
pooled_lap = []
pooled_x = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]
    M9 = Mb / 1e9

    Rs   = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)
    Sc   = SC00 * (Rs ** (1.0 + KAPPA0))

    # normalized baryonic loading
    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    # reconstruct S_data from observed memory contribution
    vmem2 = np.maximum(v_obs**2 - vb2, 0.0)
    ymem  = np.sqrt(vmem2) / max(Vinf, EPS)
    ymem  = np.clip(ymem, 0.0, 0.999999)

    S_data = -Sc * np.log(np.maximum(1.0 - ymem, EPS))

    # weighted coordinate u
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), KAPPA0)
    u = np.zeros_like(r)
    for i in range(1, len(r)):
        u[i] = u[i-1] + 0.5 * (weight[i] + weight[i-1]) * (r[i] - r[i-1])

    # reconstruct m_data = dS/du
    m_data = deriv1_nonuniform(u, S_data)

    # cylindrical radial Laplacian in u-coordinate approximation:
    # first compute derivatives wrt r for operator diagnosis
    dm_dr = deriv1_nonuniform(r, m_data)
    d2m_dr2 = deriv2_nonuniform(r, m_data)
    lap_r = d2m_dr2 + dm_dr / np.maximum(r, EPS)

    x = r / max(Rs, EPS)

    # avoid very inner boundary and outer noisy edge
    keep = (
        np.isfinite(r) &
        np.isfinite(jb) &
        np.isfinite(m_data) &
        np.isfinite(lap_r) &
        (m_data > 0) &
        (x > 0.25) &
        (x < 8.0)
    )

    if np.sum(keep) < 10:
        continue

    rk   = r[keep]
    xk   = x[keep]
    jbk  = jb[keep]
    mk   = m_data[keep]
    lapk = lap_r[keep]

    a_fit, b_fit, r2_fit, nfit = fit_ab(jbk, mk, lapk)

    # diagnostic correlations
    c_lap_jb = corr(lapk, jbk)
    c_lap_m  = corr(lapk, mk)
    c_m_jb   = corr(mk, jbk)

    gal_summaries.append({
        "name": os.path.basename(path),
        "logM": np.log10(Mb),
        "a_fit": a_fit,
        "b_fit": b_fit,
        "r2_fit": r2_fit,
        "corr_lap_jb": c_lap_jb,
        "corr_lap_m": c_lap_m,
        "corr_m_jb": c_m_jb,
        "nfit": nfit,
        "m_med": med(mk),
    })

    pooled_jb.extend(jbk.tolist())
    pooled_m.extend(mk.tolist())
    pooled_lap.extend(lapk.tolist())
    pooled_x.extend(xk.tolist())

print("Galaxies tested:", len(gal_summaries))
print("Total pooled radial points:", len(pooled_jb))
print()

# -----------------------------
# pooled fit
# -----------------------------
pooled_jb = np.array(pooled_jb, float)
pooled_m = np.array(pooled_m, float)
pooled_lap = np.array(pooled_lap, float)
pooled_x = np.array(pooled_x, float)

a_pool, b_pool, r2_pool, n_pool = fit_ab(pooled_jb, pooled_m, pooled_lap)

print("POOLED SCREENED-TRANSPORT FIT")
print("=" * 120)
print("Fit equation: Lap_r(m_data) ≈ a * j_b - b * m_data")
print("a_pool =", a_pool)
print("b_pool =", b_pool)
print("R^2    =", r2_pool)
print("N      =", n_pool)
print()

print("POOLED CORRELATIONS")
print("=" * 120)
print("corr(Lap_r m, j_b) =", corr(pooled_lap, pooled_jb))
print("corr(Lap_r m, m)   =", corr(pooled_lap, pooled_m))
print("corr(m, j_b)       =", corr(pooled_m, pooled_jb))
print("corr(m, x=r/Rs)    =", corr(pooled_m, pooled_x))
print()

# -----------------------------
# median per-galaxy summaries
# -----------------------------
print("MEDIAN PER-GALAXY DIAGNOSTICS")
print("=" * 120)
print("median a_fit       =", med([g["a_fit"] for g in gal_summaries]))
print("median b_fit       =", med([g["b_fit"] for g in gal_summaries]))
print("median R^2         =", med([g["r2_fit"] for g in gal_summaries]))
print("median corr(Lap,j) =", med([g["corr_lap_jb"] for g in gal_summaries]))
print("median corr(Lap,m) =", med([g["corr_lap_m"] for g in gal_summaries]))
print("median corr(m,j)   =", med([g["corr_m_jb"] for g in gal_summaries]))
print()

# -----------------------------
# top and bottom galaxies by PDE fit quality
# -----------------------------
gal_summaries_sorted = sorted(gal_summaries, key=lambda g: np.nan_to_num(g["r2_fit"], nan=-999), reverse=True)

print("TOP 12 GALAXIES BY PDE FIT QUALITY (R^2)")
print("=" * 120)
print(f'{"name":<24} {"a_fit":>10} {"b_fit":>10} {"R2":>8} {"corrLapJ":>10} {"corrLapM":>10} {"corrmJ":>10} {"logM":>8} {"n":>6}')
for g in gal_summaries_sorted[:12]:
    print(f'{g["name"]:<24} {g["a_fit"]:10.4g} {g["b_fit"]:10.4g} {g["r2_fit"]:8.3f} {g["corr_lap_jb"]:10.3f} {g["corr_lap_m"]:10.3f} {g["corr_m_jb"]:10.3f} {g["logM"]:8.3f} {g["nfit"]:6d}')
print()

print("BOTTOM 12 GALAXIES BY PDE FIT QUALITY (R^2)")
print("=" * 120)
print(f'{"name":<24} {"a_fit":>10} {"b_fit":>10} {"R2":>8} {"corrLapJ":>10} {"corrLapM":>10} {"corrmJ":>10} {"logM":>8} {"n":>6}')
for g in gal_summaries_sorted[-12:]:
    print(f'{g["name"]:<24} {g["a_fit"]:10.4g} {g["b_fit"]:10.4g} {g["r2_fit"]:8.3f} {g["corr_lap_jb"]:10.3f} {g["corr_lap_m"]:10.3f} {g["corr_m_jb"]:10.3f} {g["logM"]:8.3f} {g["nfit"]:6d}')
print()

# -----------------------------
# crude pooled radial-bin PDE diagnostics
# -----------------------------
xbins = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])

print("POOLED RADIAL-BIN DIAGNOSTICS")
print("=" * 120)
print(f'{"x_bin":<16} {"corr(Lap,j)":>12} {"corr(Lap,m)":>12} {"corr(m,j)":>12} {"n":>8}')
for lo, hi in zip(xbins[:-1], xbins[1:]):
    mask = (pooled_x >= lo) & (pooled_x < hi)
    if np.sum(mask) < 10:
        continue
    print(f'[{lo:>4.2f},{hi:<4.2f}) {corr(pooled_lap[mask], pooled_jb[mask]):12.3f} {corr(pooled_lap[mask], pooled_m[mask]):12.3f} {corr(pooled_m[mask], pooled_jb[mask]):12.3f} {np.sum(mask):8d}')

Galaxies tested: 53
Total pooled radial points: 1028

POOLED SCREENED-TRANSPORT FIT
Fit equation: Lap_r(m_data) ≈ a * j_b - b * m_data
a_pool = 1460.009154728471
b_pool = 24.85542313212647
R^2    = 0.26874646551993253
N      = 1028

POOLED CORRELATIONS
corr(Lap_r m, j_b) = -0.060149520111191665
corr(Lap_r m, m)   = -0.5266633393796721
corr(m, j_b)       = 0.15431277971259708
corr(m, x=r/Rs)    = 0.04820964909318507

MEDIAN PER-GALAXY DIAGNOSTICS
median a_fit       = 28.31383067057196
median b_fit       = 1.2047344962722426
median R^2         = 0.5768691300110762
median corr(Lap,j) = 0.03686821194299042
median corr(Lap,m) = -0.7643780442679744
median corr(m,j)   = -0.08031059387953747

TOP 12 GALAXIES BY PDE FIT QUALITY (R^2)
name                          a_fit      b_fit       R2   corrLapJ   corrLapM     corrmJ     logM      n
NGC1003_rotmod.dat            54.59       1.02    0.849      0.219     -0.936     -0.347    9.984     19
UGC07524_rotmod.dat           435.5      12.24    0.823

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# LOCAL CURVATURE + RADIAL HELMHOLTZ MEMORY FIELD
# ------------------------------------------------------------
# Field equation:
#
#   Rs^2 [ m'' + (1/r) m' ] + m = lambda * j_b(r)
#
# This is a forced radial Helmholtz-type equation.
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, sc0, kappa
#
# Note:
#   We solve directly for m(r). No m_inf floor here.
#   BCs:
#     m'(0)=0  -> m0 = m1
#     m(rmax)=0
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_m_field_helmholtz(r, jb, Rs, lam):
    """
    Solve:
      Rs^2 [ m'' + (1/r) m' ] + m = lam * jb(r)

    BCs:
      m'(0)=0  -> m0 = m1
      m(rmax)=0
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]

        # nonuniform second derivative
        a2 =  2.0 / (hL * (hL + hR))
        b2 = -2.0 / (hL * hR)
        c2 =  2.0 / (hR * (hL + hR))

        # nonuniform first derivative
        a1 = -hR / (hL * (hL + hR))
        b1 = (hR - hL) / (hL * hR)
        c1 =  hL / (hR * (hL + hR))

        rr = max(r[i], EPS)

        A[i, i - 1] = Rs**2 * (a2 + (1.0 / rr) * a1)
        A[i, i]     = Rs**2 * (b2 + (1.0 / rr) * b1) + 1.0
        A[i, i + 1] = Rs**2 * (c2 + (1.0 / rr) * c1)
        b[i] = lam * jb[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = 0.0

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i-1] + 0.5 * (integrand[i] + integrand[i-1]) * (r[i] - r[i-1])
    return S

def mts_helmholtz_predict(r, Mb, jb, vb2, params):
    """
    params = [lam, sc0, kappa]
    """
    lam, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    m = solve_m_field_helmholtz(r, jb, Rs, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

print("Galaxies loaded:", len(galaxies))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _ = mts_helmholtz_predict(
            g["r"], g["Mb"], g["jb"], g["vb2"],
            [lam, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

p0 = [50.0, 5.77, -0.54]
bounds_lo = [0.001, 0.01, -1.5]
bounds_hi = [1e5, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=15000)
lam_fit, sc0_fit, kappa_fit = res.x

print("Fitted Helmholtz bridge parameters")
print("lambda =", lam_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y = mts_helmholtz_predict(
        g["r"], g["Mb"], g["jb"], g["vb2"],
        [lam_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS RADIAL-HELMHOLTZ FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60

Fitted Helmholtz bridge parameters
lambda = 51.34487164863138
sc0    = 3.485708575838189
kappa  = 0.9999999999867516

MTS RADIAL-HELMHOLTZ FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               22.61092898229859
Mean RMSE:                 39.81516252320849
Median MAE:                19.664102341087002
Median signed inner resid: -0.24896676826796263
Median signed mid resid:   -0.23782531127442613
Median signed outer resid: -0.08871390725625697
Median low-Vmax RMSE:      14.91924747351776
Median high-Vmax RMSE:     58.94793332204917

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     logM   logSig    bulge
NGC5585_rotmod.dat           5.94     5.07   -0.080    0.093   -0.089    9.512    8.023    0.510
UGCA444_rotmod.dat           6.22     5.18    0.918    0.138   -0.128    8.071    7.424    0.166
NGC0100_rotmod.dat           6.53     5.31   -0.058    0.109   -0.055    9.459    7.989    0.480
NGC6503_rotmod.dat         

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# MTS FIRST-PRINCIPLES BRIDGE:
# SCREENED FIELD + COMPACTNESS-MODULATED SOURCE
# ------------------------------------------------------------
# Field equation:
#   Rs^2 m'' - (m - m_inf) = -lambda * j_eff(r)
#
# Effective source:
#   j_eff(r) = j_b(r) * C_sigma
#
# with galaxy-level compactness factor:
#   C_sigma = exp[ delta * (ln Sigma_* - ln Sigma_ref) ]
#
# This tests whether compact galaxies source the same field
# more strongly, while keeping the same radial loading shape.
#
# Cumulative stiffness:
#   S(r) = ∫ m(r) * (r/Rs)^kappa dr
#
# Response:
#   V_pred^2(r) = V_b^2(r) + Vinf(M)^2 * [1 - exp(-S/Sc)]^2
#
# Fixed MTS laws:
#   Rs(M)   = 1.402 * M9^0.279
#   Vinf(M) = 50.542 * M9^0.311
#
# Global fit:
#   lambda, m_inf, delta, sc0, kappa
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- stellar M/L from best global run -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_field(r, j_eff, Rs, m_inf, lam):
    """
    Solve:
      Rs^2 m'' - (m - m_inf) = -lam * j_eff(r)

    BCs:
      m'(0)=0  -> m0 = m1
      m(rmax)=m_inf
    """
    n = len(r)
    A = np.zeros((n, n))
    b = np.zeros(n)

    # inner Neumann
    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    # interior on nonuniform grid
    for i in range(1, n - 1):
        hL = r[i] - r[i - 1]
        hR = r[i + 1] - r[i]
        a = 2.0 / (hL * (hL + hR))
        c = 2.0 / (hR * (hL + hR))
        d = -2.0 / (hL * hR)

        A[i, i - 1] = (Rs**2) * a
        A[i, i]     = (Rs**2) * d - 1.0
        A[i, i + 1] = (Rs**2) * c
        b[i] = -m_inf - lam * j_eff[i]

    # outer Dirichlet
    A[n - 1, n - 1] = 1.0
    b[n - 1] = m_inf

    m = np.linalg.solve(A, b)
    return np.maximum(m, 0.0)

def cumulative_stiffness(r, m, Rs, kappa):
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), kappa)
    integrand = m * weight
    S = np.zeros_like(r)
    for i in range(1, len(r)):
        S[i] = S[i - 1] + 0.5 * (integrand[i] + integrand[i - 1]) * (r[i] - r[i - 1])
    return S

def mts_compact_source_strength_predict(r, Mb, jb, sigma_star, sigma_ref, vb2, params):
    """
    params = [lam, m_inf, delta, sc0, kappa]
    """
    lam, m_inf, delta, sc0, kappa = params

    M9 = Mb / 1e9
    Rs = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)

    C_sigma = np.exp(delta * (np.log(max(sigma_star, EPS)) - np.log(max(sigma_ref, EPS))))
    j_eff = jb * C_sigma

    m = solve_screened_field(r, j_eff, Rs, m_inf, lam)
    S = cumulative_stiffness(r, m, Rs, kappa)

    Sc = sc0 * (Rs ** (1.0 + kappa))
    Sc = max(Sc, EPS)

    y = 1.0 - np.exp(-S / Sc)
    vpred2 = vb2 + (Vinf ** 2) * (y ** 2)
    v_pred = np.sqrt(np.maximum(vpred2, 0.0))

    return v_pred, m, S, Rs, Vinf, y, C_sigma

# -----------------------------
# load galaxies
# -----------------------------
galaxies = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]

    dMdr = np.gradient(M_enc, r)
    dMdr = np.clip(dMdr, 0.0, None)
    norm = np.trapz(dMdr, r)
    jb = dMdr / max(norm, EPS)

    frac = M_enc / max(Mb, EPS)
    idx_h = min(np.searchsorted(frac, 0.5), len(r)-1)
    Rh = r[idx_h]
    Sigma_star = M_enc[idx_h] / max(Rh**2, EPS)

    i_vmax = int(np.argmax(v_obs))
    vg = abs(np.sqrt(vgas2[i_vmax]))
    vd = abs(np.sqrt(vdisk2[i_vmax]))
    vb = abs(np.sqrt(vbul2[i_vmax]))
    denom = vg + vd + vb + EPS

    galaxies.append({
        "name": os.path.basename(path),
        "r": r,
        "v_obs": v_obs,
        "jb": jb,
        "Mb": Mb,
        "vb2": vb2,
        "Sigma_star": Sigma_star,
        "bulge_frac": vb / denom,
        "Vmax_obs": np.max(v_obs),
    })

sigma_ref = np.median([g["Sigma_star"] for g in galaxies])

print("Galaxies loaded:", len(galaxies))
print("Sigma_ref =", sigma_ref)
print("log10 Sigma_ref =", np.log10(sigma_ref))
print()

# -----------------------------
# fit global parameters
# -----------------------------
def global_residuals(p):
    lam, m_inf, delta, sc0, kappa = p
    errs = []
    for g in galaxies:
        vp, _, _, _, _, _, _ = mts_compact_source_strength_predict(
            g["r"], g["Mb"], g["jb"], g["Sigma_star"], sigma_ref, g["vb2"],
            [lam, m_inf, delta, sc0, kappa]
        )
        errs.extend((vp - g["v_obs"]) / np.maximum(g["v_obs"], 10.0))
    return np.array(errs)

# Start near best single-scale screened bridge, no compactness modulation
p0 = [12.06, 1.81, 0.0, 5.77, -0.54]
bounds_lo = [0.01, 0.001, -3.0, 0.01, -1.5]
bounds_hi = [100.0, 10.0,  3.0, 100.0, 1.0]

res = least_squares(global_residuals, p0, bounds=(bounds_lo, bounds_hi), max_nfev=15000)
lam_fit, m_inf_fit, delta_fit, sc0_fit, kappa_fit = res.x

print("Fitted compactness-source-strength bridge parameters")
print("lambda =", lam_fit)
print("m_inf  =", m_inf_fit)
print("delta  =", delta_fit)
print("sc0    =", sc0_fit)
print("kappa  =", kappa_fit)
print()

# -----------------------------
# evaluate
# -----------------------------
rows = []

for g in galaxies:
    vp, msol, Ssol, Rs, Vinf, y, C_sigma = mts_compact_source_strength_predict(
        g["r"], g["Mb"], g["jb"], g["Sigma_star"], sigma_ref, g["vb2"],
        [lam_fit, m_inf_fit, delta_fit, sc0_fit, kappa_fit]
    )

    resid = vp - g["v_obs"]
    rmse = np.sqrt(np.mean(resid**2))
    mae = np.mean(np.abs(resid))
    frac_signed = resid / np.maximum(g["v_obs"], 1.0)

    x = g["r"] / max(Rs, EPS)
    s_in  = np.median(frac_signed[x < 0.7]) if np.any(x < 0.7) else np.nan
    s_mid = np.median(frac_signed[(x >= 0.7) & (x <= 2.0)]) if np.any((x >= 0.7) & (x <= 2.0)) else np.nan
    s_out = np.median(frac_signed[x > 2.0]) if np.any(x > 2.0) else np.nan

    rows.append({
        "name": g["name"],
        "rmse": rmse,
        "mae": mae,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "C_sigma": C_sigma,
        "logM": np.log10(g["Mb"]),
        "logSigma": np.log10(g["Sigma_star"]),
        "bulge_frac": g["bulge_frac"],
        "Vmax_obs": g["Vmax_obs"],
    })

rows = sorted(rows, key=lambda z: z["rmse"])

rmse_list  = [r["rmse"] for r in rows]
mae_list   = [r["mae"] for r in rows]
s_in_list  = [r["s_in"] for r in rows]
s_mid_list = [r["s_mid"] for r in rows]
s_out_list = [r["s_out"] for r in rows]
cs_list    = [r["C_sigma"] for r in rows]

v_split = np.median([r["Vmax_obs"] for r in rows])
low_rmse  = [r["rmse"] for r in rows if r["Vmax_obs"] < v_split]
high_rmse = [r["rmse"] for r in rows if r["Vmax_obs"] >= v_split]

print("MTS COMPACTNESS-SOURCE-STRENGTH FIRST-PRINCIPLES TEST")
print("=" * 120)
print("GLOBAL SUMMARY")
print("=" * 120)
print("Median RMSE:              ", med(rmse_list))
print("Mean RMSE:                ", np.mean(rmse_list))
print("Median MAE:               ", med(mae_list))
print("Median signed inner resid:", med(s_in_list))
print("Median signed mid resid:  ", med(s_mid_list))
print("Median signed outer resid:", med(s_out_list))
print("Median C_sigma:           ", med(cs_list))
print("Median low-Vmax RMSE:     ", med(low_rmse))
print("Median high-Vmax RMSE:    ", med(high_rmse))
print()

print("BEST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"Csig":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[:12]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["C_sigma"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("WORST 12 GALAXIES")
print("=" * 120)
print(f'{"name":<24} {"RMSE":>8} {"MAE":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"Csig":>8} {"logM":>8} {"logSig":>8} {"bulge":>8}')
for row in rows[-12:]:
    print(f'{row["name"]:<24} {row["rmse"]:8.2f} {row["mae"]:8.2f} {row["s_in"]:8.3f} {row["s_mid"]:8.3f} {row["s_out"]:8.3f} {row["C_sigma"]:8.3f} {row["logM"]:8.3f} {row["logSigma"]:8.3f} {row["bulge_frac"]:8.3f}')
print()

print("CORRELATION WITH RMSE")
print("=" * 120)
print("log10(Mbar):       ", corr([r["logM"] for r in rows], [r["rmse"] for r in rows]))
print("log10(Sigma_*):    ", corr([r["logSigma"] for r in rows], [r["rmse"] for r in rows]))
print("bulge fraction:    ", corr([r["bulge_frac"] for r in rows], [r["rmse"] for r in rows]))
print("C_sigma:           ", corr([r["C_sigma"] for r in rows], [r["rmse"] for r in rows]))

Galaxies loaded: 60
Sigma_ref = 319126537.4187316
log10 Sigma_ref = 8.503962920067265

Fitted compactness-source-strength bridge parameters
lambda = 15.362897785545464
m_inf  = 1.9226045443463347
delta  = 0.15524971565265555
sc0    = 5.619601274500557
kappa  = -0.4903382638349127

MTS COMPACTNESS-SOURCE-STRENGTH FIRST-PRINCIPLES TEST
GLOBAL SUMMARY
Median RMSE:               19.942203880184447
Mean RMSE:                 29.50810524768985
Median MAE:                17.164454059251035
Median signed inner resid: -0.16716530432648322
Median signed mid resid:   -0.002728813040260548
Median signed outer resid: -0.03799113915276019
Median C_sigma:            0.9998865425748111
Median low-Vmax RMSE:      9.451869978748288
Median high-Vmax RMSE:     44.1988791570942

BEST 12 GALAXIES
name                         RMSE      MAE     s_in    s_mid    s_out     Csig     logM   logSig    bulge
UGC07524_rotmod.dat          2.75     2.57   -0.128    0.032   -0.039    0.702    9.569    7.514    0.470
NG

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# DIAGNOSTIC: DOES RECONSTRUCTED m_data(r) FOLLOW
#             exp(-r/l), exp(-r/l)/sqrt(r), or exp(-r/l)/r ?
# ------------------------------------------------------------
# Uses current best bridge to reconstruct S_data and m_data:
#
#   V_obs^2 = V_b^2 + Vinf^2 * [1 - exp(-S/Sc)]^2
#
# Then:
#   S_data = -Sc * ln(1 - y_mem)
#   du = (r/Rs)^kappa dr
#   m_data = dS_data/du
#
# We then fit, galaxy by galaxy, in x = r/Rs coordinates:
#
#   model 1: m(x) = A * exp(-x/L)
#   model 2: m(x) = A * exp(-x/L) / sqrt(x)
#   model 3: m(x) = A * exp(-x/L) / x
#
# Since these are linear in log-space:
#
#   log m = c0 - x/L               (plain exp)
#   log m = c0 - x/L - 0.5 log x   (cyl envelope)
#   log m = c0 - x/L - 1.0 log x   (sph envelope)
#
# For each galaxy we fit c0 and slope in:
#   y = log m + p log x = c0 - b x
# where p = 0, 0.5, 1.0
#
# Then compare R^2 and RMSE in log-space.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- best current local+memory bridge parameters -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866
SC00     = 5.769116014395077
KAPPA0   = -0.5420338644899587

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def deriv1_nonuniform(x, y):
    return np.gradient(y, x)

def fit_log_template(x, m, p):
    """
    Fit:
      log m + p log x = c0 - b x
    where:
      p = 0.0  -> exp(-x/L)
      p = 0.5  -> exp(-x/L)/sqrt(x)
      p = 1.0  -> exp(-x/L)/x

    Returns:
      c0, b, L, r2, rmse_log
    """
    x = np.asarray(x, float)
    m = np.asarray(m, float)

    mask = np.isfinite(x) & np.isfinite(m) & (x > 0) & (m > 0)
    x = x[mask]
    m = m[mask]
    if len(x) < 8:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    y = np.log(m) + p * np.log(x)
    X = np.column_stack([np.ones_like(x), -x])

    coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    c0, b = coef
    yhat = X @ coef

    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse_log = np.sqrt(np.mean((y - yhat)**2))

    L = 1.0 / b if np.isfinite(b) and (b > 0) else np.nan
    return c0, b, L, r2, rmse_log

# -----------------------------
# reconstruct m_data and compare templates
# -----------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]
    M9 = Mb / 1e9

    Rs   = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)
    Sc   = SC00 * (Rs ** (1.0 + KAPPA0))

    # reconstruct memory contribution from data
    vmem2 = np.maximum(v_obs**2 - vb2, 0.0)
    ymem  = np.sqrt(vmem2) / max(Vinf, EPS)
    ymem  = np.clip(ymem, 0.0, 0.999999)

    S_data = -Sc * np.log(np.maximum(1.0 - ymem, EPS))

    # weighted coordinate u
    weight = np.power(np.clip(r / max(Rs, EPS), EPS, None), KAPPA0)
    u = np.zeros_like(r)
    for i in range(1, len(r)):
        u[i] = u[i-1] + 0.5 * (weight[i] + weight[i-1]) * (r[i] - r[i-1])

    m_data = deriv1_nonuniform(u, S_data)

    x = r / max(Rs, EPS)

    # use region where reconstruction is reasonably stable
    keep = (
        np.isfinite(x) &
        np.isfinite(m_data) &
        (x > 0.25) &
        (x < 8.0) &
        (m_data > 0)
    )

    xk = x[keep]
    mk = m_data[keep]

    if len(xk) < 10:
        continue

    c0_e,  b_e,  L_e,  r2_e,  rmse_e  = fit_log_template(xk, mk, p=0.0)
    c0_c,  b_c,  L_c,  r2_c,  rmse_c  = fit_log_template(xk, mk, p=0.5)
    c0_s,  b_s,  L_s,  r2_s,  rmse_s  = fit_log_template(xk, mk, p=1.0)

    # winner by lowest RMSE in log-space
    labels = ["exp", "exp_over_sqrtx", "exp_over_x"]
    rmses  = [rmse_e, rmse_c, rmse_s]
    best_i = int(np.nanargmin(rmses))
    best_label = labels[best_i]

    rows.append({
        "name": os.path.basename(path),
        "logM": np.log10(Mb),
        "nfit": len(xk),

        "r2_exp": r2_e,
        "L_exp": L_e,
        "rmse_exp": rmse_e,

        "r2_cyl": r2_c,
        "L_cyl": L_c,
        "rmse_cyl": rmse_c,

        "r2_sph": r2_s,
        "L_sph": L_s,
        "rmse_sph": rmse_s,

        "best_model": best_label,
    })

print("Galaxies compared:", len(rows))
print()

# -----------------------------
# summary counts
# -----------------------------
count_exp = sum(r["best_model"] == "exp" for r in rows)
count_cyl = sum(r["best_model"] == "exp_over_sqrtx" for r in rows)
count_sph = sum(r["best_model"] == "exp_over_x" for r in rows)

print("BEST TEMPLATE COUNTS")
print("=" * 120)
print("plain exp            =", count_exp)
print("exp / sqrt(x)        =", count_cyl)
print("exp / x              =", count_sph)
print()

print("MEDIAN FIT QUALITY")
print("=" * 120)
print("median R2 plain exp      =", med([r["r2_exp"] for r in rows]))
print("median R2 exp/sqrt(x)    =", med([r["r2_cyl"] for r in rows]))
print("median R2 exp/x          =", med([r["r2_sph"] for r in rows]))
print()
print("median RMSE plain exp    =", med([r["rmse_exp"] for r in rows]))
print("median RMSE exp/sqrt(x)  =", med([r["rmse_cyl"] for r in rows]))
print("median RMSE exp/x        =", med([r["rmse_sph"] for r in rows]))
print()
print("median L plain exp       =", med([r["L_exp"] for r in rows]))
print("median L exp/sqrt(x)     =", med([r["L_cyl"] for r in rows]))
print("median L exp/x           =", med([r["L_sph"] for r in rows]))
print()

# -----------------------------
# sort by cylindrical advantage
# -----------------------------
for r in rows:
    r["cyl_adv_rmse_vs_exp"] = r["rmse_exp"] - r["rmse_cyl"]
    r["cyl_adv_rmse_vs_sph"] = r["rmse_sph"] - r["rmse_cyl"]

rows_sorted = sorted(rows, key=lambda z: z["cyl_adv_rmse_vs_exp"], reverse=True)

print("TOP 12 GALAXIES WHERE exp/sqrt(x) BEATS plain exp MOST")
print("=" * 120)
print(f'{"name":<24} {"best":>16} {"dRMSE(cyl-exp)":>16} {"R2_exp":>10} {"R2_cyl":>10} {"R2_sph":>10} {"L_cyl":>10} {"logM":>8} {"n":>6}')
for r in rows_sorted[:12]:
    print(f'{r["name"]:<24} {r["best_model"]:>16} {r["cyl_adv_rmse_vs_exp"]:16.4f} {r["r2_exp"]:10.3f} {r["r2_cyl"]:10.3f} {r["r2_sph"]:10.3f} {r["L_cyl"]:10.3f} {r["logM"]:8.3f} {r["nfit"]:6d}')
print()

rows_sorted_worst = sorted(rows, key=lambda z: z["cyl_adv_rmse_vs_exp"])

print("BOTTOM 12 GALAXIES WHERE exp/sqrt(x) LOSES TO plain exp MOST")
print("=" * 120)
print(f'{"name":<24} {"best":>16} {"dRMSE(cyl-exp)":>16} {"R2_exp":>10} {"R2_cyl":>10} {"R2_sph":>10} {"L_cyl":>10} {"logM":>8} {"n":>6}')
for r in rows_sorted_worst[:12]:
    print(f'{r["name"]:<24} {r["best_model"]:>16} {r["cyl_adv_rmse_vs_exp"]:16.4f} {r["r2_exp"]:10.3f} {r["r2_cyl"]:10.3f} {r["r2_sph"]:10.3f} {r["L_cyl"]:10.3f} {r["logM"]:8.3f} {r["nfit"]:6d}')
print()

# -----------------------------
# correlation of cylindrical advantage with mass
# -----------------------------
print("CORRELATION DIAGNOSTICS")
print("=" * 120)
print("corr(logM, cyl_adv_rmse_vs_exp) =", corr([r["logM"] for r in rows], [r["cyl_adv_rmse_vs_exp"] for r in rows]))
print("corr(logM, R2_cyl - R2_exp)     =", corr([r["logM"] for r in rows], [(r["r2_cyl"] - r["r2_exp"]) for r in rows]))

Galaxies compared: 53

BEST TEMPLATE COUNTS
plain exp            = 31
exp / sqrt(x)        = 6
exp / x              = 16

MEDIAN FIT QUALITY
median R2 plain exp      = 0.26568477566093673
median R2 exp/sqrt(x)    = 0.24591583697523922
median R2 exp/x          = 0.30890613432860337

median RMSE plain exp    = 1.132389890899575
median RMSE exp/sqrt(x)  = 1.1196319184756724
median RMSE exp/x        = 1.092468362523826

median L plain exp       = 0.42195085119998005
median L exp/sqrt(x)     = 0.25003385031155334
median L exp/x           = 0.1809315919446691

TOP 12 GALAXIES WHERE exp/sqrt(x) BEATS plain exp MOST
name                                 best   dRMSE(cyl-exp)     R2_exp     R2_cyl     R2_sph      L_cyl     logM      n
NGC0024_rotmod.dat             exp_over_x           0.1220      0.257      0.246      0.234      0.146    9.450     14
NGC2915_rotmod.dat             exp_over_x           0.0936      0.608      0.599      0.588      0.175    9.000     11
UGC04278_rotmod.dat        

In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import least_squares

# ============================================================
# KERNEL TEST:
#   Does S_data(r) look like the cumulative of a Yukawa-like field
#   with kernel r^p ?
#
# We reconstruct from the observed curves:
#
#   V_obs^2 = V_b^2 + Vinf^2 * [1 - exp(-S/Sc)]^2
#
# so
#
#   S_data(r) = -Sc * ln(1 - sqrt(V_obs^2 - V_b^2)/Vinf)
#
# Then for each galaxy we build a simple Yukawa-like field template
#
#   m_Y(r) = A * exp(-r/l) / r
#
# and test three accumulation kernels:
#
#   K0   : dS = m_Y(r) * r^0      dr
#   K-1/2: dS = m_Y(r) * r^(-1/2) dr
#   K-1  : dS = m_Y(r) * r^(-1)   dr
#
# We fit A and l separately for each kernel by matching S_model(r) to S_data(r).
#
# This directly tests whether the successful kappa ~ -1/2 is the right
# field-to-memory map.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

# ----- fixed MTS laws -----
AR = 1.402
BETA_R = 0.279
AV = 50.542
ALPHA_V = 0.311

# ----- best current bridge params for reconstructing S_data -----
UPS_DISK = 0.7897399854812461
UPS_BUL  = 0.7084845386257866
SC00     = 5.769116014395077
KAPPA0   = -0.5420338644899587

# ----- constants -----
G = 4.30091e-6
EPS = 1e-12

# ----- quality cuts -----
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def corr(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if np.sum(m) < 3:
        return np.nan
    x = x[m]
    y = y[m]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) &
        np.isfinite(v_gas) & np.isfinite(v_disk) & np.isfinite(v_bul) &
        (r > 0) & (v_obs > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def cumulative_trapz(x, y):
    out = np.zeros_like(x, dtype=float)
    for i in range(1, len(x)):
        out[i] = out[i-1] + 0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1])
    return out

def yukawa_field(r, A, ell, Rs):
    # In dimensionless x=r/Rs form this is A * exp(-x/L) / x
    x = np.clip(r / max(Rs, EPS), EPS, None)
    L = max(ell / max(Rs, EPS), EPS)
    return A * np.exp(-x / L) / x

def build_S_model(r, Rs, A, ell, p):
    mY = yukawa_field(r, A, ell, Rs)
    x = np.clip(r / max(Rs, EPS), EPS, None)
    integrand = mY * np.power(x, p)
    return cumulative_trapz(r, integrand)

def fit_kernel_to_S(r, S_data, Rs, p):
    # Fit A and ell for a fixed kernel exponent p
    mask = np.isfinite(r) & np.isfinite(S_data) & (r > 0)
    r = r[mask]
    S_data = S_data[mask]
    if len(r) < 10:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # ignore exact origin and very outer edge if noisy
    x = r / max(Rs, EPS)
    keep = (x > 0.15) & (x < 8.0) & np.isfinite(S_data)
    r = r[keep]
    S_data = S_data[keep]
    if len(r) < 10:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # normalize target for shape fit + amplitude fit together
    # fit directly in S-space
    def resid(theta):
        logA, logL = theta
        A = np.exp(logA)
        ell = np.exp(logL) * Rs
        S_mod = build_S_model(r, Rs, A, ell, p)
        # allow a light weighting toward shape over huge absolute scales
        scale = np.maximum(np.max(S_data), 1.0)
        return (S_mod - S_data) / scale

    # initial guesses
    p0 = [np.log(max(np.max(S_data), 1e-3)), np.log(1.0)]
    bounds_lo = [np.log(1e-8), np.log(0.03)]
    bounds_hi = [np.log(1e8),  np.log(30.0)]

    res = least_squares(resid, p0, bounds=(bounds_lo, bounds_hi), max_nfev=5000)
    logA, logL = res.x
    A = np.exp(logA)
    Ldimless = np.exp(logL)
    ell = Ldimless * Rs
    S_mod = build_S_model(r, Rs, A, ell, p)

    rmse = np.sqrt(np.mean((S_mod - S_data)**2))
    ss_res = np.sum((S_mod - S_data)**2)
    ss_tot = np.sum((S_data - np.mean(S_data))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return A, ell, Ldimless, rmse, r2

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    vgas2  = np.maximum(v_gas**2, 0.0)
    vdisk2 = np.maximum((UPS_DISK * v_disk)**2, 0.0)
    vbul2  = np.maximum((UPS_BUL  * v_bul )**2, 0.0)
    vb2    = vgas2 + vdisk2 + vbul2

    M_enc = np.maximum(r * vb2 / G, 0.0)
    Mb = M_enc[-1]
    M9 = Mb / 1e9

    Rs   = AR * (M9 ** BETA_R)
    Vinf = AV * (M9 ** ALPHA_V)
    Sc   = SC00 * (Rs ** (1.0 + KAPPA0))

    # reconstruct S_data from observed memory contribution
    vmem2 = np.maximum(v_obs**2 - vb2, 0.0)
    ymem  = np.sqrt(vmem2) / max(Vinf, EPS)
    ymem  = np.clip(ymem, 0.0, 0.999999)

    S_data = -Sc * np.log(np.maximum(1.0 - ymem, EPS))

    A0, ell0, L0, rmse0, r20 = fit_kernel_to_S(r, S_data, Rs, p=0.0)
    A1, ell1, L1, rmse1, r21 = fit_kernel_to_S(r, S_data, Rs, p=-0.5)
    A2, ell2, L2, rmse2, r22 = fit_kernel_to_S(r, S_data, Rs, p=-1.0)

    rmses = np.array([rmse0, rmse1, rmse2], float)
    names = ["p=0", "p=-1/2", "p=-1"]

    if np.all(~np.isfinite(rmses)):
        continue

    best_i = int(np.nanargmin(rmses))
    best_name = names[best_i]

    rows.append({
        "name": os.path.basename(path),
        "logM": np.log10(Mb),

        "rmse_p0": rmse0,
        "rmse_pHalf": rmse1,
        "rmse_p1": rmse2,

        "r2_p0": r20,
        "r2_pHalf": r21,
        "r2_p1": r22,

        "L_p0": L0,
        "L_pHalf": L1,
        "L_p1": L2,

        "best_model": best_name,
        "adv_half_vs_0": rmse0 - rmse1,
        "adv_half_vs_1": rmse2 - rmse1,
    })

print("Galaxies compared:", len(rows))
print()

count_p0 = sum(r["best_model"] == "p=0" for r in rows)
count_pH = sum(r["best_model"] == "p=-1/2" for r in rows)
count_p1 = sum(r["best_model"] == "p=-1" for r in rows)

print("BEST KERNEL COUNTS")
print("=" * 120)
print("p = 0      :", count_p0)
print("p = -1/2   :", count_pH)
print("p = -1     :", count_p1)
print()

print("MEDIAN FIT QUALITY")
print("=" * 120)
print("median RMSE p=0      =", med([r["rmse_p0"] for r in rows]))
print("median RMSE p=-1/2   =", med([r["rmse_pHalf"] for r in rows]))
print("median RMSE p=-1     =", med([r["rmse_p1"] for r in rows]))
print()
print("median R2 p=0        =", med([r["r2_p0"] for r in rows]))
print("median R2 p=-1/2     =", med([r["r2_pHalf"] for r in rows]))
print("median R2 p=-1       =", med([r["r2_p1"] for r in rows]))
print()
print("median L p=0         =", med([r["L_p0"] for r in rows]))
print("median L p=-1/2      =", med([r["L_pHalf"] for r in rows]))
print("median L p=-1        =", med([r["L_p1"] for r in rows]))
print()

rows_top = sorted(rows, key=lambda z: z["adv_half_vs_0"], reverse=True)
print("TOP 12 GALAXIES WHERE p=-1/2 BEATS p=0 MOST")
print("=" * 120)
print(f'{"name":<24} {"best":>10} {"dRMSE(0-half)":>14} {"dRMSE(1-half)":>14} {"R2_0":>9} {"R2_h":>9} {"R2_1":>9} {"L_h":>9} {"logM":>8}')
for r in rows_top[:12]:
    print(f'{r["name"]:<24} {r["best_model"]:>10} {r["adv_half_vs_0"]:14.4f} {r["adv_half_vs_1"]:14.4f} {r["r2_p0"]:9.3f} {r["r2_pHalf"]:9.3f} {r["r2_p1"]:9.3f} {r["L_pHalf"]:9.3f} {r["logM"]:8.3f}')
print()

rows_bot = sorted(rows, key=lambda z: z["adv_half_vs_0"])
print("BOTTOM 12 GALAXIES WHERE p=-1/2 LOSES TO p=0 MOST")
print("=" * 120)
print(f'{"name":<24} {"best":>10} {"dRMSE(0-half)":>14} {"dRMSE(1-half)":>14} {"R2_0":>9} {"R2_h":>9} {"R2_1":>9} {"L_h":>9} {"logM":>8}')
for r in rows_bot[:12]:
    print(f'{r["name"]:<24} {r["best_model"]:>10} {r["adv_half_vs_0"]:14.4f} {r["adv_half_vs_1"]:14.4f} {r["r2_p0"]:9.3f} {r["r2_pHalf"]:9.3f} {r["r2_p1"]:9.3f} {r["L_pHalf"]:9.3f} {r["logM"]:8.3f}')
print()

print("CORRELATION DIAGNOSTICS")
print("=" * 120)
print("corr(logM, adv_half_vs_0) =", corr([r["logM"] for r in rows], [r["adv_half_vs_0"] for r in rows]))
print("corr(logM, R2_half-R2_0)  =", corr([r["logM"] for r in rows], [(r["r2_pHalf"] - r["r2_p0"]) for r in rows]))

Galaxies compared: 60

BEST KERNEL COUNTS
p = 0      : 49
p = -1/2   : 0
p = -1     : 11

MEDIAN FIT QUALITY
median RMSE p=0      = 21.22986245947233
median RMSE p=-1/2   = 22.163154543354786
median RMSE p=-1     = 22.556813747115605

median R2 p=0        = 0.5205246164985634
median R2 p=-1/2     = 0.3777300494751713
median R2 p=-1       = 0.25872600695809966

median L p=0         = 29.999999999997087
median L p=-1/2      = 29.99999999999745
median L p=-1        = 29.999999999998824

TOP 12 GALAXIES WHERE p=-1/2 BEATS p=0 MOST
name                           best  dRMSE(0-half)  dRMSE(1-half)      R2_0      R2_h      R2_1       L_h     logM
UGC11914_rotmod.dat            p=-1         0.2706        -0.2398 -157910506025425015455363366912.000 -154149833500958665637357420544.000 -150854885554583284072834924544.000     0.030   10.762
UGC05253_rotmod.dat            p=-1         0.1966        -0.1681 -145778722703101018656278052864.000 -143149165256401176040893841408.000 -14091964086943615331